# 06 — Extract then answer (E7): page-wise evidence extraction, an Evidence Memory, and text-only reasoning

**Objective.** Find out whether separating *visual evidence extraction* (one page at a time) from *cross-page reasoning* (text only, over an Evidence Memory) answers more questions correctly than E6's one-shot reader. The same pages, model snapshot and scorer are used.

**This is a development experiment.** E6's 118 held-out questions were inspected answer by answer to design E7 (section 1). They are now a **development set**: every E7 number on them is a development result, not a held-out one. A final evaluation needs documents E1–E7 never touched. It is proposed at the end and run only after the design is frozen.

**Status (2026-11-25): closed.** Both runs completed on 2026-11-24 (smoke 11 calls, full 1,309 calls; USD 5.911057 at list price). The verdict is **`regression`**: V0 38, X4 31, X8 29 of 102 answerable. The result is frozen in `experiments/frozen/e7/`, with an independent audit (38/38). Section 10 evaluates that frozen copy by default. The paid modes stay closed.

**Inputs** (this notebook fails with a clear message if one is missing):
- `data/processed/mmlongbench_doc/questions.parquet` and `render_manifest_e1.csv` (notebook 00);
- `data/processed/mmlongbench_doc/e5/heldout_pages.parquet` and `render_manifest_heldout.csv`, plus the cached 768-px renders (notebook 04);
- `experiments/frozen/e5/` (the frozen ColQwen2 rankings and evidence strata);
- `experiments/frozen/e6/` (the frozen E6 run: requests, replies, per-question table);
- `experiments/specs/e7_extract_then_answer_v1.json` (this experiment's specification);
- `experiments/frozen/e7/` (the frozen E7 runs and tables, evaluated in section 10).

**Outputs** (all git-ignored):
- `data/processed/mmlongbench_doc/e7/`: the E6 error audit, the request plan, the cost plan, the design, and evaluations;
- `experiments/runs/e7/<run_id>/`: live runs (paid modes only).

Nothing under `experiments/frozen/`, `src/visualdoc/` or `AGENTS.md` is written to.

**Modes** (`VISUALDOC_E7_MODE`):
- `offline` (the default): sends nothing;
- `smoke` and `full`: paid, and open only when the spec is frozen and committed and the owner has approved the stage.

`VISUALDOC_E7_RESUME_DIR` continues an interrupted run. `VISUALDOC_E7_RUN_DIR` evaluates a completed one.

**Next step.**
- If the verdict is `strong_support` or `promising_but_uncertain`: extract-then-answer becomes the candidate for one evaluation on an untouched test set (a separate, approved experiment).
- If it is `no_meaningful_gain` or `regression`: keep E6's one-shot reader.

## Method

### Why this experiment (from the E6 audit, section 1)
With ColQwen2's pages, the one-shot reader answers:
- 30 of 55 questions whose evidence is on one page (plus 1 of 1 without annotated evidence);
- 7 of 46 questions whose evidence spans several pages, and only 4 of 12 even when every evidence page was supplied.

27 of its 64 errors happen with sufficient evidence in front of it. Two thirds of those are of the kinds a page-at-a-time reading should help: table cells, lists, arithmetic, cross-page joins, distractor pages, and reads of figures, counts and fine print that one image would leave more room for.

### Hypothesis (frozen in the spec before any E7 call)
> On the 102 answerable development questions, reading each supplied page separately into a structured Evidence Memory and answering from that memory in a text-only step with optional deterministic arithmetic (**X4**) answers more questions correctly than E6's one-shot reader on the same pages (**V0**), especially on multi-page questions.

Whether the gain comes from the error types the architecture targets is analysed as a **diagnostic**, not a gate.

### Three arms, one session
| arm | pages | reader |
|---|---|---|
| **V0** | E6's ColQwen2 pages (top 5, then the context guard: 4 pages for 105 questions, 3 for 13) | E6's one-shot request, byte for byte (same fingerprint as the frozen E6 V request) |
| **X4** | exactly V0's pages | one extraction call per page, then the Evidence Memory, then one text-only reasoning call |
| **X8** | ColQwen2's top 8 (its first 3–4 pages are X4's) | the same extract-then-answer architecture |

**Identification.**
- **X4 − V0** = the extract-then-answer **architecture** effect: same pages, model, images and answer contract. This is the **primary** comparison.
  - It is not a pure "Evidence Memory effect". X4 changes four things at once: page-wise reading, structured evidence, text-only reasoning and optional tool use.
  - Only the tool's share is separable (section 9).
- **X8 − X4** = the effect of 4–5 additional pages under the new architecture.
- **X8 − V0** = the total practical change. It mixes two factors, so it is never read as the architecture effect.

V0 is re-run live, rather than taken from the frozen E6 run, because the same request can change its answer between sessions (5 of 37 image replies did in E3). Running all three arms in the same session, question by question, keeps backend drift out of the comparisons. The frozen E6 V result is still reported beside V0; the number of questions whose correctness flips between them measures session noise directly.

### The architecture
For question $q$ and pages $P = (p_1, \dots, p_k)$ in ColQwen2's rank order:

1. **Extraction (VLM, one image per call).** $x_{p} = \mathrm{Extract}(q, \mathrm{image}_p)$. It returns page-local facts $\{(\ell, v, u, t, s)\}$: label, value, unit, period, supporting text. It also returns a diagnostic `relevant` flag and notes. It must not answer, compute, or use other pages or outside knowledge.
2. **Evidence Memory (deterministic).** $E = \bigcup_{p \in P} \{\, (\mathrm{id}=p.i,\ \mathrm{page}=p,\ \ell, v, u, t, s) \,\}$, plus one status line per page: facts found, nothing relevant, or unreadable.
   - Pages and facts keep rank and reply order.
   - Nothing is merged, deduplicated or filtered.
   - `relevant` is never used to drop a page.
3. **Reasoning (LLM, text only).** $(a, C, \kappa) = \mathrm{Reason}(q, E)$: the shortest complete answer $a$, cited pages $C$, and optionally a calculation $\kappa$. The images are not sent again.
4. **Arithmetic (deterministic, no code execution).** $\kappa$ is an expression tree over fact ids, explicit constants, and a closed set of operations.
   - When it evaluates, its result replaces $a$, and the pages of its operand facts are added to $C$.
   - The model's own $a$ is kept. Scoring $a$ against the computed answer measures what the tool contributes inside each X arm.

**Same answer contract as V0.** The reasoning step's `answer` instruction is V0's sentence, word for word ("the shortest complete answer. Use a number … use exactly the string "Not answerable""). Only the input changes: the Evidence Memory instead of images.

### Assumptions and known confounds
- **Not one factor.** X4 changes the reading procedure (k extraction calls plus one reasoning call instead of one call), and adds the arithmetic tool. Their joint effect is the architecture effect. The tool's share is reported separately (model answer vs computed answer).
- **More calls.** X4 makes k + 1 calls instead of 1, at about the same image tokens. Any gain must be weighed against latency (reported).
- **Prompts written after looking.** The extraction and reasoning prompts were written after seeing E6's failure types: they are generic, but not blind. This is why E6 is a development set here and why a final test on untouched documents is required.
- **Citation validity is easier for X8**, which supplies 8 pages. Its guardrails are therefore read against X4, not V0.
- **Scoring.** The scorer is the frozen MMLongBench-Doc rule (notebook 03), unchanged; correct means score > 0.

### Decision rule (frozen in the spec; section 7)
X4 vs V0 is the primary comparison. **Engineering usefulness** and **statistical certainty** are judged separately.

**Engineering criteria** (all must hold):
- a gain ≥ +0.05;
- broken ≤ fixed / 2;
- correct abstentions ≥ V0 − 2 (of 16);
- citation validity ≥ V0 − 0.05.

**Statistical evidence** (reported separately):
- the document-cluster bootstrap interval and its one-sided lower bound;
- the exact document-level sign-flip p.

**Verdicts:**
- `strong_support`: the engineering criteria pass, the lower bound is > 0 and p ≤ 0.05;
- `promising_but_uncertain`: the engineering criteria pass, but the statistics are not conclusive. This is **not** evidence that the architecture failed;
- `no_meaningful_gain`: the gain is < 0.05;
- `regression`: a guardrail fails, accuracy falls by ≥ 0.05, or broken > fixed / 2.

**Mechanism.** The E6 audit categories of fixed and broken questions are a diagnostic only. The audit labels are used only for evaluation and never in a request.

In [1]:
# E7 setup: paths, the specification and the run mode. Nothing here depends on an E7 outcome.
from collections import Counter, deque
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_UP
from itertools import product
from math import ceil, comb, isclose, isfinite
from pathlib import Path
import ast
import base64
import hashlib
import inspect
import json
import os
import re
import subprocess
import tempfile
import time
import uuid

import httpx
import numpy as np
import openai
import pandas as pd
import tiktoken
from IPython.display import display
from openai.types.chat import ChatCompletion

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 170)


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repository root")


PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "mmlongbench_doc"
E5_DIR = PROCESSED_DIR / "e5"
E7_DIR = PROCESSED_DIR / "e7"
FROZEN_DIR = PROJECT_ROOT / "experiments" / "frozen"
FROZEN_E5_DIR = FROZEN_DIR / "e5"
FROZEN_E6_DIR = FROZEN_DIR / "e6"
FROZEN_E6_RUN = FROZEN_E6_DIR / "run_20261123T200722253880Z_d94e85f1_full"
SPEC_RELPATH = "experiments/specs/e7_extract_then_answer_v1.json"
E6_SPEC_RELPATH = "experiments/specs/e6_visual_retrieval_answering_v1.json"
E5_SPEC_RELPATH = "experiments/specs/e5_visual_retrieval_v1.json"
NOTEBOOK_RELPATH = "notebooks/06_extract_then_answer.ipynb"
RUNS_DIR = PROJECT_ROOT / "experiments" / "runs" / "e7"

REQUIRED_INPUTS = {
    PROCESSED_DIR / "questions.parquet": "run 00_document_processing",
    E5_DIR / "heldout_pages.parquet": "run 04_visual_retrieval",
    E5_DIR / "render_manifest_heldout.csv": "run 04_visual_retrieval",
    PROJECT_ROOT / SPEC_RELPATH: "the E7 specification is tracked in git: restore it",
    PROJECT_ROOT / E6_SPEC_RELPATH: "the E6 specification is tracked in git: restore it",
    PROJECT_ROOT / E5_SPEC_RELPATH: "the E5 specification is tracked in git: restore it",
    FROZEN_E5_DIR / "SHA256SUMS": "the frozen E5 result is tracked in git: restore it",
    FROZEN_E6_DIR / "SHA256SUMS": "the frozen E6 result is tracked in git: restore it",
    FROZEN_E6_RUN / "predictions_main_r1_V.jsonl": "the frozen E6 run is tracked in git: restore it",
}
missing = [f"{path.relative_to(PROJECT_ROOT)} ({fix})" for path, fix in REQUIRED_INPUTS.items() if not path.is_file()]
assert not missing, "E7 inputs missing:\n  " + "\n  ".join(missing)
E7_DIR.mkdir(parents=True, exist_ok=True)

SPEC_BYTES = (PROJECT_ROOT / SPEC_RELPATH).read_bytes()
SPEC_SHA256 = hashlib.sha256(SPEC_BYTES).hexdigest()
SPEC = json.loads(SPEC_BYTES)
assert SPEC["spec_id"] == "e7-extract-then-answer-v1"
SPEC_FROZEN = SPEC["status"] == "frozen-before-live-execution" and bool(SPEC.get("frozen_at"))
E7_MODE = os.getenv("VISUALDOC_E7_MODE", "offline")
assert E7_MODE in {"offline", "smoke", "full"}, "VISUALDOC_E7_MODE must be offline, smoke or full"
MODEL, V0_READER, EXTRACTION, REASONING, ARITHMETIC = (SPEC[k] for k in ("model", "v0_reader", "extraction", "reasoning", "arithmetic"))
EXECUTION, BUDGET, POP, RULE, METRICS = (SPEC[k] for k in ("execution", "budget", "population", "decision_rule", "metrics"))
ARMS = {name: arm["label"] for name, arm in SPEC["design"]["arms"].items()}
assert list(ARMS) == ["V0", "X4", "X8"]
PRICE = {"input": MODEL["input_usd_per_1m_tokens"] / 1e6, "output": MODEL["output_usd_per_1m_tokens"] / 1e6}

display(pd.Series({
    "specification": f"{SPEC['spec_id']} (sha256 {SPEC_SHA256[:12]})",
    "status": SPEC["status"] + ("" if SPEC_FROZEN else " (not frozen: the paid modes are closed)"),
    "population": "E6's 118 held-out questions, now a DEVELOPMENT set (102 answerable, 16 unanswerable)",
    "E7 mode": E7_MODE,
    "model": f"{MODEL['requested_snapshot']} (shutdown 2027-01-14), 768-px images, temperature 0",
    "arms": "; ".join(f"{k}: {v}" for k, v in ARMS.items()),
    "budget (USD)": f"approved {BUDGET['approved_ceiling_usd']}, proposed {BUDGET['proposed_ceiling_usd']}",
}, name="E7").to_frame())

,E7
specification,e7-extract-then-answer-v1 (sha256 7d124b81cbdf)
status,frozen-before-live-execution
population,"E6's 118 held-out questions, now a DEVELOPMENT set (102 answerable..."
E7 mode,offline
model,"gpt-4o-mini-2024-07-18 (shutdown 2027-01-14), 768-px images, tempe..."
arms,V0: one-shot control: E6's ColQwen2 pages and E6's exact request; ...
budget (USD),"approved 9.0, proposed 9.0"


In [2]:
# Frozen results E0-E7 are read-only inputs. Every frozen copy must equal its SHA256SUMS, and git must show no change to any protected
# path, at the start of this notebook and again at its end (Validation).
def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def verify_sha256sums(folder):
    entries = [line.split(maxsplit=1) for line in (folder / "SHA256SUMS").read_text().splitlines() if line.strip()]
    return {name: (folder / name).is_file() and sha256_bytes((folder / name).read_bytes()) == digest for digest, name in entries}


def git(*args):
    return subprocess.run(["git", *args], cwd=PROJECT_ROOT, capture_output=True, text=True).stdout.strip()


def git_show_bytes(commit, relpath):
    shown = subprocess.run(["git", "show", f"{commit}:{relpath}"], cwd=PROJECT_ROOT, capture_output=True)
    return shown.stdout if shown.returncode == 0 else None


def read_jsonl(path):
    with open(path) as handle:
        return [json.loads(line) for line in handle if line.strip()]


FROZEN_FOLDERS = {"E0 (mmlongbench_doc baseline)": "mmlongbench_doc", "E1": "e1", "E2": "e2", "E3": "e3", "E4": "e4", "E5": "e5", "E6": "e6", "E7": "e7"}
frozen_check = {}
for label, name in FROZEN_FOLDERS.items():
    checks = verify_sha256sums(FROZEN_DIR / name)
    assert checks and all(checks.values()), f"experiments/frozen/{name} differs from its SHA256SUMS: {[k for k, ok in checks.items() if not ok]}"
    frozen_check[label] = len(checks)

PROTECTED_PATHS = ["experiments/frozen", "src/visualdoc", "AGENTS.md", "tests", "archive", E5_SPEC_RELPATH, E6_SPEC_RELPATH]


def protected_changes():
    # Uncommitted changes (tracked or untracked) under any protected path; must stay empty.
    return git("status", "--porcelain", "--untracked-files=all", "--", *PROTECTED_PATHS).splitlines()


PROTECTED_AT_START = protected_changes()
assert PROTECTED_AT_START == [], f"a protected path has uncommitted changes: {PROTECTED_AT_START}"

# The frozen E6 run: its spec snapshot equals the tracked E6 spec, it completed, and it holds one V reply per held-out question.
E6_SPEC = json.loads((PROJECT_ROOT / E6_SPEC_RELPATH).read_bytes())
assert (FROZEN_E6_RUN / "spec_snapshot.json").read_bytes() == (PROJECT_ROOT / E6_SPEC_RELPATH).read_bytes()
e6_completion = json.loads((FROZEN_E6_RUN / "completion.json").read_text())
assert e6_completion["complete"] and e6_completion["cumulative_list_price_cost_usd"] == 3.721277
E6_RECORDS = {arm: {r["qid"]: r for r in read_jsonl(FROZEN_E6_RUN / f"predictions_main_r1_{arm}.jsonl")} for arm in ("A", "V")}
assert all(len(v) == 118 for v in E6_RECORDS.values())
e6_table = pd.read_csv(FROZEN_E6_DIR / "tables" / "per_question.csv").set_index("qid")
e6_summary = json.loads((FROZEN_E6_DIR / "tables" / "evaluation_summary.json").read_text())

display(pd.Series(frozen_check, name="files equal to SHA256SUMS").to_frame())
print(f"Protected paths without uncommitted changes: {', '.join(PROTECTED_PATHS)}. Frozen E6 run: 118 A and 118 V replies, "
      f"${e6_completion['cumulative_list_price_cost_usd']}, spec snapshot equal to the tracked E6 spec.")

,files equal to SHA256SUMS
E0 (mmlongbench_doc baseline),29
E1,30
E2,27
E3,30
E4,29
E5,11
E6,31


Protected paths without uncommitted changes: experiments/frozen, src/visualdoc, AGENTS.md, tests, archive, experiments/specs/e5_visual_retrieval_v1.json, experiments/specs/e6_visual_retrieval_answering_v1.json. Frozen E6 run: 118 A and 118 V replies, $3.721277, spec snapshot equal to the tracked E6 spec.


In [3]:
# The population (E6's 118 held-out questions), ColQwen2's frozen rankings and the cached 768-px renders.
E5_SPEC = json.loads((PROJECT_ROOT / E5_SPEC_RELPATH).read_bytes())
HELDOUT_DOCS = E5_SPEC["populations"]["held_out"]["selected_documents"]
assert HELDOUT_DOCS == E6_SPEC["population"]["documents"] == POP["documents"]
questions = pd.read_parquet(PROCESSED_DIR / "questions.parquet")
questions["evidence_pages"] = questions["evidence_pages"].map(lambda values: [int(v) for v in values])
questions["evidence_sources"] = questions["evidence_sources"].map(list)
held = questions[questions["doc_id"].isin(HELDOUT_DOCS)].sort_values("qid").reset_index(drop=True)
held_pages = pd.read_parquet(E5_DIR / "heldout_pages.parquet")
PAGE_COUNT = held_pages.groupby("doc_id").size()
held["evidence_population"] = [bool(r.answerable) and bool(r.evidence_pages) and min(r.evidence_pages) >= 1 and max(r.evidence_pages) <= PAGE_COUNT[r.doc_id]
                               for r in held.itertuples()]
QIDS = held["qid"].tolist()
ANSWERABLE = held.loc[held["answerable"], "qid"].tolist()
UNANSWERABLE = held.loc[~held["answerable"], "qid"].tolist()
EVIDENCE_QIDS = held.loc[held["evidence_population"], "qid"].tolist()
assert (len(QIDS), len(ANSWERABLE), len(UNANSWERABLE), len(EVIDENCE_QIDS)) == (POP["questions"], POP["answerable"], POP["unanswerable"], POP["evidence_population"]) \
    == (118, 102, 16, 101)
assert set(QIDS) == set(E6_RECORDS["V"]) and not held["in_subset"].any()
held_idx = held.set_index("qid")
doc_of = held_idx["doc_id"]
question_of = held_idx["question"]
assert not held.duplicated(["doc_id", "question"]).any(), "two questions share a text: the extraction cache key would merge them"

# ColQwen2's frozen full rankings (E5 arm V) and E5's evidence strata after the guard.
rankings = pd.read_parquet(FROZEN_E5_DIR / "tables" / "rankings.parquet")
rankings = rankings[(rankings["population"] == "held-out") & (rankings["arm"] == "V")]
RANKING = {r.qid: [int(p) for p in r.pages] for r in rankings.itertuples()}
assert set(RANKING) == set(QIDS) and all(sorted(RANKING[q]) == list(range(1, PAGE_COUNT[doc_of[q]] + 1)) for q in QIDS)
frozen_e5_held = pd.read_csv(FROZEN_E5_DIR / "tables" / "per_question.csv").query("population == 'held-out'").set_index("qid")
assert sorted(frozen_e5_held.index) == EVIDENCE_QIDS

# The renders: E5's manifest, checked against the frozen copy, and every PNG checked by hash when read.
held_manifest = pd.read_csv(E5_DIR / "render_manifest_heldout.csv")
frozen_manifest = pd.read_csv(FROZEN_E5_DIR / "tables" / "render_manifest_heldout.csv")
manifest_columns = ["doc_id", "page", "width_px", "height_px", "sha256"]
assert held_manifest[manifest_columns].equals(frozen_manifest[manifest_columns]), "held-out renders differ from the frozen E5 manifest: rerun 04"
held_manifest = held_manifest.set_index(["doc_id", "page"])
_png_cache = {}


def manifest_png(doc_id, page):
    # A cached 768-px render, checked against its manifest hash.
    if (doc_id, page) not in _png_cache:
        entry = held_manifest.loc[(doc_id, page)]
        path = PROJECT_ROOT / entry["image_path"]
        assert path.is_file(), f"missing render {path.relative_to(PROJECT_ROOT)}: rerun 04_visual_retrieval"
        png = path.read_bytes()
        assert sha256_bytes(png) == entry["sha256"], f"stale page image {doc_id} p.{page}: rerun the notebook that renders it"
        _png_cache[(doc_id, page)] = png
    return _png_cache[(doc_id, page)]


print(f"Population: {len(QIDS)} questions ({len(ANSWERABLE)} answerable, {len(UNANSWERABLE)} unanswerable, {len(EVIDENCE_QIDS)} with evidence pages) "
      f"in {len(HELDOUT_DOCS)} documents, {int(PAGE_COUNT.sum())} pages. Frozen ColQwen2 rankings for all {len(RANKING)} questions; the render "
      f"manifest equals the frozen E5 copy.")

Population: 118 questions (102 answerable, 16 unanswerable, 101 with evidence pages) in 14 documents, 647 pages. Frozen ColQwen2 rankings for all 118 questions; the render manifest equals the frozen E5 copy.


## 1. The E6 error audit, rebuilt reproducibly

Every one of the 64 answerable questions that E6's ColQwen2 arm (V) got wrong was inspected by hand:
- the question, the reference answer and the annotated evidence pages;
- the pages V supplied and the model's raw reply;
- the page text, and the 768-px page image where perception was in question.

The 9 unanswerable questions V answered were inspected the same way. Each received one **primary** failure label and one **group**:

- **A. Insufficient evidence before reasoning.** The required evidence never reached the reader: a retrieval miss, a context-guard drop, partial evidence, or an unsupported answer given on partial evidence.
- **B. Sufficient evidence, reader failure.** Perception, counting or spatial reading, table cells, lists, arithmetic, cross-page aggregation, semantic or distractor errors, or false abstention.
- **C. Evaluation or annotation issue.** A correct answer the frozen scorer rejects, or a debatable reference.

The labels below are written out as data, with their justification. They are judgements, so the cell checks them mechanically against everything that can be computed:
- the frozen E6 scores;
- E5's evidence strata, and the pages lost to the context guard;
- the replies themselves.

The two documented overrides of the annotation-based strata are listed explicitly.

**Use.** These labels are evaluation diagnostics only. They are never read by a cell that builds a request (checked in section 4).

In [4]:
# The frozen MMLongBench-Doc scorer (notebook 03 section 1, cell f653d232), written out unchanged; correct means score > 0.
NOT_ANSWERABLE = "Not answerable"

def levenshtein_distance(s1, s2):
    if len(s1) > len(s2):
        s1, s2 = s2, s1
    distances = range(len(s1) + 1)
    for i2, c2 in enumerate(s2):
        new_distances = [i2 + 1]
        for i1, c1 in enumerate(s1):
            new_distances.append(distances[i1] if c1 == c2 else 1 + min(distances[i1], distances[i1 + 1], new_distances[-1]))
        distances = new_distances
    return distances[-1]


def anls_compute(groundtruth, prediction, threshold=0.5):
    length = max(len(groundtruth.upper()), len(prediction.upper()))
    anls = 1.0 - (0.0 if length == 0 else levenshtein_distance(groundtruth, prediction) / length)
    return 0.0 if anls <= threshold else anls


def is_float_equal(reference, prediction, include_percentage=False, is_close=False):
    def get_precision(value):
        text = str(value)
        return len(text.split(".")[-1]) if "." in text else 3

    reference = float(str(reference).strip().rstrip("%").strip())
    try:
        prediction = float(str(prediction).strip().rstrip("%").strip())
    except (TypeError, ValueError):
        return False
    for item in ([reference / 100, reference, reference * 100] if include_percentage else [reference]):
        if is_close and isclose(item, prediction, rel_tol=0.01):
            return True
        precision = max(min(get_precision(prediction), get_precision(item)), 2)
        if round(prediction, precision) == round(item, precision):
            return True
    return False


def get_clean_string(s):
    s = str(s).lower().strip()
    s = re.sub(r"\s*\([^)]*\)", "", s).strip()  # official quirk kept: unit suffixes such as "miles" are not stripped
    s = re.sub(r"^['\"]|['\"]$", "", s).strip()
    s = s.strip().lstrip("$").strip()
    return s.strip().rstrip("%").strip()


EXACT_MATCH_PATTERNS = [r"\b\d+(-\d+|\s\d+)?\b", r"\b\d{4}[-\s]\d{2}[-\s]\d{2}\b", r"\b\d{4}[-\s]\d{2}\b", r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"]


def is_exact_match(s):
    return ("https://" in s or s.endswith(".py") or s.endswith("ipynb") or s.startswith("page") or "a.m." in s or "p.m." in s
            or any(re.fullmatch(pattern, s) for pattern in EXACT_MATCH_PATTERNS))


def isfloat(value):
    try:
        float(value)
        return True
    except (TypeError, ValueError):
        return False


def as_list(value):
    if isinstance(value, str) and value.startswith("["):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            pass
    return value if isinstance(value, list) else [value]


def eval_score(gt, pred, answer_type):
    if answer_type == "Int":
        try:
            gt, pred = int(gt), int(float(pred))
        except Exception:
            pred = ""
        score = gt == pred
    elif answer_type == "Float":
        try:
            gt, pred = float(get_clean_string(str(gt))), float(get_clean_string(str(pred)))
        except Exception:
            pred = ""
        score = is_float_equal(gt, pred, include_percentage=True, is_close=True)
    elif answer_type in ("Str", "None"):
        gt, pred = get_clean_string(gt), get_clean_string(pred)
        score = gt == pred if is_exact_match(gt) else anls_compute(gt, pred)
    else:  # List
        gt, pred = as_list(gt), as_list(pred)
        if len(gt) != len(pred):
            score = 0.0
        else:
            gt, pred = sorted(get_clean_string(i) for i in gt), sorted(get_clean_string(i) for i in pred)
            score = "-".join(gt) == "-".join(pred) if isfloat(gt[0]) or is_exact_match(gt[0]) else min(anls_compute(g, p) for g, p in zip(gt, pred))
    return float(score)


def accuracy_and_f1(scored):
    accuracy = scored["score"].mean()
    answerable, predicted_answer = scored["answer"] != NOT_ANSWERABLE, scored["pred"] != NOT_ANSWERABLE
    if answerable.sum() == 0 or predicted_answer.sum() == 0:
        return accuracy, 0.0
    recall = scored.loc[answerable, "score"].sum() / answerable.sum()
    precision = scored.loc[answerable, "score"].sum() / predicted_answer.sum()
    return accuracy, (2 * recall * precision / (recall + precision) if recall + precision > 0 else 0.0)


SCORING_CASES = [  # reference, prediction, format, expected (traced by hand from the official source, notebook 03)
    ("Less well-off", "less well off", "Str", 1 - 1 / 13), ("Pew Research Center", "Pew Research Centre", "Str", 1 - 2 / 19),
    ("Latinos", "Asian Americans", "Str", 0.0), ("Not answerable", "Not answerable", "None", 1.0), ("Not answerable", "The report does not say", "None", 0.0),
    ("12", "12.0", "Int", 1.0), ("12", "twelve", "Int", 0.0), ("18.29%", "18.3", "Float", 1.0), ("0.1", "10%", "Float", 1.0), ("51%", "49", "Float", 0.0),
    ("['Circle', 'Rectangle']", "['rectangle', 'circle']", "List", 1.0), ("['White', '10%']", "['White']", "List", 0.0),
    ("['2014', '2016']", "['2016', '2014']", "List", 1.0), ("https://www.example.com", "www.example.com", "Str", 0.0), ("2016-06", "2016-06", "Str", 1.0),
    ("$1,200 (approx.)", "1,200", "Str", 1.0), ("5 miles", "5", "Str", 0.0)]
assert all(isclose(eval_score(r, p, f), e, abs_tol=1e-9) for r, p, f, e in SCORING_CASES) and len(SCORING_CASES) == 17
toy = pd.DataFrame({"answer": ["A", "B", NOT_ANSWERABLE], "pred": ["A", NOT_ANSWERABLE, "C"], "score": [1.0, 0.0, 0.0]})
assert all(isclose(x, y) for x, y in zip(accuracy_and_f1(toy), (1 / 3, 0.5)))

# The scorer reproduces every frozen E6 score (both arms, 118 questions) from the frozen replies.
for arm in ("A", "V"):
    for qid in QIDS:
        record = E6_RECORDS[arm][qid]
        ref = held_idx.loc[qid]
        assert isclose(eval_score(ref["answer"], record["pred"], ref["answer_format"]), e6_table.loc[qid, f"score_{arm}"], abs_tol=1e-12), (arm, qid)
E6_CORRECT = {arm: {q: eval_score(held_idx.loc[q, "answer"], E6_RECORDS[arm][q]["pred"], held_idx.loc[q, "answer_format"]) > 0 for q in QIDS} for arm in ("A", "V")}
e6_counts = {arm: sum(E6_CORRECT[arm][q] for q in ANSWERABLE) for arm in ("A", "V")}
e6_abstain = {arm: sum(E6_CORRECT[arm][q] for q in UNANSWERABLE) for arm in ("A", "V")}
assert e6_counts == {"A": 25, "V": 38} and e6_abstain == {"A": 9, "V": 7}
assert sum(E6_CORRECT["V"][q] and not E6_CORRECT["A"][q] for q in ANSWERABLE) == 13 and sum(E6_CORRECT["A"][q] and not E6_CORRECT["V"][q] for q in ANSWERABLE) == 0
print(f"Scorer: {len(SCORING_CASES)} hand-traced cases pass; every frozen E6 score is reproduced (A 25/102, V 38/102, 13 fixed / 0 broken, "
      f"correct abstentions A 9/16, V 7/16).")

Scorer: 17 hand-traced cases pass; every frozen E6 score is reproduced (A 25/102, V 38/102, 13 fixed / 0 broken, correct abstentions A 9/16, V 7/16).


In [5]:
# The audit labels (manual, 2026-11-24), written out as data: qid, group, primary failure, justification.
# A = insufficient evidence before reasoning; B = sufficient evidence, reader failure; C = evaluation or annotation issue.
AUDIT_CATEGORIES = ["retrieval miss", "context guard drop", "partial evidence", "visual reading/perception", "counting/spatial", "table-cell alignment",
                    "exhaustive list extraction", "numerical calculation", "multi-page aggregation", "semantic/distractor error", "false abstention",
                    "unsupported answer", "scoring mismatch", "annotation issue", "other"]
AUDIT_ROWS = [
    # --- C: evaluation or annotation issue (4) ---
    ("mmlb-0684", "C", "scoring mismatch", "says men 81% vs women 78% (correct) in a sentence; ANLS against 'men' is 0"),
    ("mmlb-0593", "C", "scoring mismatch", "'Marc Silski' vs reference 'Special Agent Marc Silski': ANLS 0.44 <= 0.5"),
    ("mmlb-0689", "C", "annotation issue", "page 9 gives counts ('not percentages'); reference 1.4 for 'excited' is not on the page; reply 51 (= hopeful) also wrong"),
    ("mmlb-1013", "C", "annotation issue", "reply 17,564 = Costco stockholders' equity (a defensible 'common equity'); reference 18,078 = total equity incl. NCI"),
    # --- B: sufficient evidence, reader failure (27) ---
    ("mmlb-1006", "B", "visual reading/perception", "says feature 5 is not illustrated although label 5 is drawn; Int question answered with a list"),
    ("mmlb-0062", "B", "visual reading/perception", "generic sentence instead of the arrow labels in figure 1"),
    ("mmlb-0063", "B", "visual reading/perception", "line colour gold instead of red; cites page 5, which was not supplied"),
    ("mmlb-0042", "B", "visual reading/perception", "lists the Business Intelligence bracket instead of the Business Analytics bracket"),
    ("mmlb-0580", "B", "visual reading/perception", "two small URLs on the cover page; counts one at 768 px"),
    ("mmlb-0065", "B", "table-cell alignment", "Table 3 Align. column: ChatGPT (0.1) = 85.9 is highest; answers 0.5"),
    ("mmlb-0080", "B", "table-cell alignment", "datasets with exactly three method citations in Table II; picks rows with two or four"),
    ("mmlb-0077", "B", "counting/spatial", "distinct icons in figure 3: 6 instead of 9"),
    ("mmlb-0200", "B", "counting/spatial", "counts the two pinching figures with the one two-finger-scrolling figure"),
    ("mmlb-0204", "B", "counting/spatial", "left of the camera: microphone instead of the ambient light sensor"),
    ("mmlb-1001", "B", "counting/spatial", "counts the website with the three social media accounts (4 vs 3)"),
    ("mmlb-1002", "B", "counting/spatial", "people wearing glasses in a photo: 2 instead of 3"),
    ("mmlb-0048", "B", "counting/spatial", "hand-drawn cartoons over the 4 supplied evidence pages: 3 instead of 4"),
    ("mmlb-0081", "B", "exhaustive list extraction", "subsections A-C of section V on pages 10-11 (supplied); answers 'Augmentation'"),
    ("mmlb-0594", "B", "exhaustive list extraction", "adds checks to other payees to the two Mont Blanc checks"),
    ("mmlb-0141", "B", "exhaustive list extraction", "Columbia chapter authors across pages 14-15: finds 2 of 3"),
    ("mmlb-1003", "B", "multi-page aggregation", "milestone worth 8 units (p.11) -> 16-week internship (p.16); answers 8"),
    ("mmlb-0143", "B", "multi-page aggregation", "research-highlight chapter names (p.12) -> chapter numbers (contents p.7); ignores p.12, lists all 9"),
    ("mmlb-1059", "B", "numerical calculation", "allowance 156 -> 62 is a 60.3% decrease; answers 2.6"),
    ("mmlb-1014", "B", "numerical calculation", "long-term liabilities = 6,692 + 2,642 + 2,415; returns one line, in billions"),
    ("mmlb-1057", "B", "numerical calculation", "OVERRIDE sufficient: page 40 alone gives 20,522 / (131,310 - 27,709) = 0.198; answers 0.154"),
    ("mmlb-1060", "B", "semantic/distractor error", "lists operating-expense lines; the cost-of-sales sentence on supplied p.44 is ignored"),
    ("mmlb-0409", "B", "semantic/distractor error", "answers from the meiosis slide (p.27) although the named slides 12 and 14 are supplied and tagged"),
    ("mmlb-0050", "B", "semantic/distractor error", "inverts the cartoon: the hippo is the highest paid person's opinion, not data-driven decisions"),
    ("mmlb-0140", "B", "semantic/distractor error", "OVERRIDE sufficient: the contents page is p.7 (supplied), annotated p.6; lists appendices as chapters"),
    ("mmlb-0076", "B", "false abstention", "tree figure supplied with legible branch names; abstains"),
    ("mmlb-0598", "B", "false abstention", "handwritten numbers on supplied pages 15-16; abstains"),
    # --- A: insufficient evidence before reasoning (33) ---
    ("mmlb-0078", "A", "context guard drop", "evidence p.6 ranked 5th, dropped by the guard; p.17 also needed"),
    ("mmlb-1056", "A", "context guard drop", "evidence p.38 ranked 5th, dropped by the guard; p.40 not retrieved"),
    ("mmlb-0082", "A", "context guard drop", "evidence p.9 ranked 5th, dropped by the guard; complete without the guard"),
    ("mmlb-1051", "A", "retrieval miss", "derived ratio (DPO); the statement pages are not in the top 5"),
    ("mmlb-1052", "A", "retrieval miss", "derived ratio; the statement pages are not in the top 5"),
    ("mmlb-1053", "A", "retrieval miss", "derived ratio (ROA); the statement pages are not in the top 5"),
    ("mmlb-1054", "A", "retrieval miss", "derived ratio (ROE); the statement pages are not in the top 5"),
    ("mmlb-1055", "A", "retrieval miss", "derived ratio (effective tax rate); the statement pages are not in the top 5"),
    ("mmlb-1058", "A", "retrieval miss", "derived ratio (debt / EBITDA); the statement pages are not in the top 5"),
    ("mmlb-1015", "A", "retrieval miss", "EBITDA; the statement pages are not in the top 5"),
    ("mmlb-1027", "A", "retrieval miss", "plain-text page 9 not in the top 5"),
    ("mmlb-1061", "A", "retrieval miss", "question typo 'least cost' for 'lease cost'; page 47 not in the top 5"),
    ("mmlb-0990", "A", "retrieval miss", "table/figure page 25 not in the top 5"),
    ("mmlb-0686", "A", "partial evidence", "count line plots in the whole report (5 evidence pages)"),
    ("mmlb-1007", "A", "partial evidence", "p.11 missing"),
    ("mmlb-0066", "A", "partial evidence", "compare section lengths; p.13 missing"),
    ("mmlb-0067", "A", "partial evidence", "the definition on p.3 is missing; answers from p.4-5"),
    ("mmlb-1016", "A", "partial evidence", "debt note p.53 missing; 6,692 (balance-sheet line) is a defensible alternative"),
    ("mmlb-1017", "A", "partial evidence", "debt note p.53 missing"),
    ("mmlb-1018", "A", "partial evidence", "p.40 and p.53 missing; also a calculation"),
    ("mmlb-1020", "A", "partial evidence", "p.53 missing; reply consistent with 6,692 / 41,190"),
    ("mmlb-1023", "A", "partial evidence", "p.53 missing; also uses total liabilities as debt"),
    ("mmlb-0199", "A", "partial evidence", "p.49 dropped by the guard and p.50 not retrieved"),
    ("mmlb-0202", "A", "partial evidence", "p.68 dropped by the guard and p.66 not retrieved"),
    ("mmlb-0203", "A", "partial evidence", "phone-number table p.55 missing; abstains (reasonable given the pages)"),
    ("mmlb-1000", "A", "partial evidence", "two-hop: course list p.8 missing"),
    ("mmlb-0407", "A", "partial evidence", "count faces across the whole deck"),
    ("mmlb-0045", "A", "partial evidence", "pp. 9 and 16 missing"),
    ("mmlb-0046", "A", "partial evidence", "p.50 missing; abstains"),
    ("mmlb-0049", "A", "partial evidence", "8 evidence pages, more than the 4-page budget"),
    ("mmlb-0582", "A", "partial evidence", "p.2 missing"),
    ("mmlb-0142", "A", "partial evidence", "6 evidence pages; whole-document count"),
    ("mmlb-0597", "A", "unsupported answer", "claims redactions on pp. 4, 6, 8 (none visible on p.4); p.14 missing"),
]
STRATUM_OVERRIDES = {  # annotation-based strata that misstate sufficiency, with the reason (both checked by hand)
    "mmlb-1057": "page 40 alone is sufficient: cash 20,522 and total liabilities 131,310 - 27,709 = 103,601",
    "mmlb-0140": "the annotated page 6 is 'About this report'; the table of contents is page 7, which was supplied",
}
# The 9 unanswerable questions V answered (not part of the 64; reported with abstention).
UNANSWERABLE_ROWS = [
    ("mmlb-0043", "unsupported answer", "false premise ('business cooperation' instead of 'business analytics'); answers the original question"),
    ("mmlb-0068", "unsupported answer", "false premise ('either' instead of 'both'); answers the original question"),
    ("mmlb-0079", "unsupported answer", "no method meets both constraints; names one"),
    ("mmlb-0138", "unsupported answer", "the person is not in the document; names an agency"),
    ("mmlb-0205", "unsupported answer", "asks about a Thinkpad in a MacBook guide; answers for the MacBook"),
    ("mmlb-0406", "unsupported answer", "no such case in chapter 9; answers from another case"),
    ("mmlb-0408", "annotation issue", "'How many cats appear?' -> 0 is a defensible answer; the reference requires 'Not answerable'"),
    ("mmlb-0577", "unsupported answer", "no yellow italic text on page 9; quotes a heading"),
    ("mmlb-0596", "unsupported answer", "no such interview in the complaint; answers with another interview"),
]
audit = pd.DataFrame(AUDIT_ROWS, columns=["qid", "group", "category", "justification"]).set_index("qid")
audit_unanswerable = pd.DataFrame(UNANSWERABLE_ROWS, columns=["qid", "category", "justification"]).set_index("qid")
assert audit.index.is_unique and set(audit["category"]) <= set(AUDIT_CATEGORIES) and set(audit["group"]) == {"A", "B", "C"}
print(f"{len(audit)} answerable and {len(audit_unanswerable)} unanswerable audit labels loaded.")

64 answerable and 9 unanswerable audit labels loaded.


In [6]:
# The labels checked against everything computable, then the decomposition tables.
def evidence_stratum(qid, supplied):
    # all / some / none of the annotated evidence pages among the supplied pages; 'no evidence pages' for mmlb-0599.
    evidence = held_idx.loc[qid, "evidence_pages"]
    if qid not in EVIDENCE_QIDS:
        return "no annotated evidence"
    found = set(evidence) & set(supplied)
    return "complete" if found == set(evidence) else ("partial" if found else "none")


e6v = pd.DataFrame({
    "doc_id": doc_of[ANSWERABLE], "doc_type": held_idx.loc[ANSWERABLE, "doc_type"], "answer_format": held_idx.loc[ANSWERABLE, "answer_format"],
    "n_evidence": [len(held_idx.loc[q, "evidence_pages"]) for q in ANSWERABLE],
    "stratum": [evidence_stratum(q, E6_RECORDS["V"][q]["supplied_pages"]) for q in ANSWERABLE],
    "correct": [E6_CORRECT["V"][q] for q in ANSWERABLE], "pred": [E6_RECORDS["V"][q]["pred"] for q in ANSWERABLE]}, index=pd.Index(ANSWERABLE, name="qid"))
e6v["evidence_span"] = np.where(e6v["n_evidence"] == 0, "no annotated evidence", np.where(e6v["n_evidence"] >= 2, "multi-page", "single-page"))
# The stratum of E5's frozen table is the same computation (supplied_V after the guard).
for q in EVIDENCE_QIDS:
    row = frozen_e5_held.loc[q]
    assert e6v.loc[q, "stratum"] == ("complete" if row["complete_after_V"] else ("partial" if row["recall_after_V"] > 0 else "none")), q

# 1. The labelled set is exactly the set of answerable questions V got wrong.
wrong = e6v.index[~e6v["correct"]]
assert sorted(audit.index) == sorted(wrong) and len(wrong) == 64
assert sorted(audit_unanswerable.index) == sorted(q for q in UNANSWERABLE if not E6_CORRECT["V"][q]) and len(audit_unanswerable) == 9
audit = audit.join(e6v)
audit["lost_to_guard"] = [int(frozen_e5_held.loc[q, "evidence_lost_to_guard_V"]) if q in EVIDENCE_QIDS else 0 for q in audit.index]
audit["top5_complete"] = [set(held_idx.loc[q, "evidence_pages"]) <= set(RANKING[q][:5]) for q in audit.index]
audit["evidence_in_top5"] = [bool(set(held_idx.loc[q, "evidence_pages"]) & set(RANKING[q][:5])) for q in audit.index]
abstained = audit["pred"].map(lambda p: p == NOT_ANSWERABLE)

# 2. Group against the evidence strata: A only where evidence was incomplete, B only where it was complete, except the documented overrides.
assert set(STRATUM_OVERRIDES) <= set(audit.index[audit["group"] == "B"])
for q, row in audit.iterrows():
    if row["group"] == "A":
        assert row["stratum"] in ("partial", "none"), q
    elif row["group"] == "B":
        assert row["stratum"] == "complete" or q in STRATUM_OVERRIDES, q
# 3. Category against what can be computed.
guard_rule = (audit["lost_to_guard"] > 0) & ((audit["stratum"] == "none") | audit["top5_complete"])
assert set(audit.index[audit["category"] == "context guard drop"]) == set(audit.index[(audit["group"] == "A") & guard_rule])
assert (~audit.loc[audit["category"] == "retrieval miss", "evidence_in_top5"]).all()
assert (audit.loc[audit["category"] == "partial evidence", "stratum"] == "partial").all()
assert abstained[audit["category"] == "false abstention"].all() and not abstained[audit["category"].isin(["unsupported answer", "scoring mismatch"])].any()

# 4. The diagnostic numbers quoted for E6 (verified, not assumed).
span = e6v.groupby("evidence_span")["correct"].agg(["sum", "count"])
strata = e6v.groupby("stratum")["correct"].agg(["sum", "count"])
# Corrects an earlier quotation of 31/56 "single-page", which counted mmlb-0599 (no annotated evidence pages) as single-page.
assert span.loc["single-page"].tolist() == [30, 55] and span.loc["multi-page"].tolist() == [7, 46] and span.loc["no annotated evidence"].tolist() == [1, 1]
assert strata.loc["complete"].tolist() == [32, 61] and strata.loc["partial"].tolist() == [3, 25] and strata.loc["none"].tolist() == [2, 15]
assert e6v.loc[(e6v["evidence_span"] == "multi-page") & (e6v["stratum"] == "complete"), "correct"].agg(["sum", "count"]).tolist() == [4, 12]

# 5. How much evidence more pages of the same ranking would complete, for group A.
group_a = audit.index[audit["group"] == "A"]
topk_complete = {k: int(sum(set(held_idx.loc[q, "evidence_pages"]) <= set(RANKING[q][:k]) for q in group_a)) for k in (4, 5, 8, 10, 15, 20)}
assert topk_complete[8] == 6 and topk_complete[10] == 9 and topk_complete[15] == 16

TARGETED = METRICS["mechanism"]["targeted_categories"]
assert set(TARGETED) <= set(AUDIT_CATEGORIES)
audit["targeted"] = audit["category"].isin(TARGETED)
audit.reset_index().to_csv(E7_DIR / "e6_error_audit.csv", index=False)
audit_unanswerable.reset_index().to_csv(E7_DIR / "e6_unanswerable_audit.csv", index=False)

by_category = (audit.groupby(["group", "category"]).size().rename("errors").reset_index()
               .assign(targeted=lambda d: d["category"].isin(TARGETED)).sort_values(["group", "errors"], ascending=[True, False]))
display(by_category.set_index(["group", "category"]))
display(pd.DataFrame({"A insufficient evidence": [int((audit["group"] == "A").sum())], "B reader failure": [int((audit["group"] == "B").sum())],
                      "C evaluation / annotation": [int((audit["group"] == "C").sum())], "total": [len(audit)]}, index=["E6 V errors (of 102)"]))


def breakdown(column):
    table = pd.DataFrame({"questions": e6v.groupby(column).size(), "V correct": e6v.groupby(column)["correct"].sum(),
                          "errors": audit.groupby(column).size()}).fillna(0).astype(int)
    for g in "ABC":
        table[f"errors {g}"] = audit[audit["group"] == g].groupby(column).size().reindex(table.index).fillna(0).astype(int)
    return table.sort_values("questions", ascending=False)


for column in ("answer_format", "doc_type", "evidence_span", "stratum"):
    display(breakdown(column))
print(f"Group A errors whose evidence would be complete within ColQwen2's top k: {topk_complete} (of {len(group_a)}).")
print(f"Targeted by extract-then-answer (spec, fixed before any E7 call): {TARGETED} -> {int(audit['targeted'].sum())} of 64 E6 errors.")
examples = audit.groupby("category").head(1).reset_index()[["qid", "group", "category", "justification"]]
examples["question"] = examples["qid"].map(lambda q: question_of[q][:80])
examples["reference"] = examples["qid"].map(lambda q: str(held_idx.loc[q, "answer"])[:40])
examples["E6 V reply"] = examples["qid"].map(lambda q: str(E6_RECORDS["V"][q]["pred"])[:50])
display(examples.set_index("qid")[["group", "category", "question", "reference", "E6 V reply"]])
display(audit_unanswerable.assign(reply=[str(E6_RECORDS["V"][q]["pred"])[:60] for q in audit_unanswerable.index]))

errors  targeted
group category                                    
A     partial evidence                19     False
      retrieval miss                  10     False
      context guard drop               3     False
      unsupported answer               1     False
B     counting/spatial                 6     False
      visual reading/perception        5     False
      semantic/distractor error        4      True
      exhaustive list extraction       3      True
      numerical calculation            3      True
      false abstention                 2     False
      multi-page aggregation           2      True
      table-cell alignment             2      True
C     annotation issue                 2     False
      scoring mismatch                 2     False

,A insufficient evidence,B reader failure,C evaluation / annotation,total
E6 V errors (of 102),33,27,4,64


,questions,V correct,errors,errors A,errors B,errors C
answer_format,,,,,,
Str,29,13,16,10,4,2
Float,28,10,18,13,4,1
Int,24,10,14,4,10,0
List,21,5,16,6,9,1


,questions,V correct,errors,errors A,errors B,errors C
doc_type,,,,,,
Financial report,31,11,20,15,4,1
Tutorial/Workshop,14,6,8,4,4,0
Administration/Industry file,13,7,6,2,3,1
Research report / Introduction,13,6,7,2,3,2
Academic paper,12,1,11,4,7,0
Guidebook,11,5,6,4,2,0
Brochure,8,2,6,2,4,0


,questions,V correct,errors,errors A,errors B,errors C
evidence_span,,,,,,
single-page,55,30,25,3,19,3
multi-page,46,7,39,30,8,1
no annotated evidence,1,1,0,0,0,0


,questions,V correct,errors,errors A,errors B,errors C
stratum,,,,,,
complete,61,32,29,0,25,4
partial,25,3,22,21,1,0
none,15,2,13,12,1,0
no annotated evidence,1,1,0,0,0,0


Group A errors whose evidence would be complete within ColQwen2's top k: {4: 0, 5: 1, 8: 6, 10: 9, 15: 16, 20: 18} (of 33).
Targeted by extract-then-answer (spec, fixed before any E7 call): ['table-cell alignment', 'exhaustive list extraction', 'numerical calculation', 'multi-page aggregation', 'semantic/distractor error'] -> 14 of 64 E6 errors.


,group,category,question,reference,E6 V reply
qid,,,,,
mmlb-0684,C,scoring mismatch,Do more men or women (in %) think a female president will be elect...,men,More men (81%) think a female president will be el
mmlb-0689,C,annotation issue,"How many % of voters reactions are ""uneasy"" and ""excited"" about Tr...","[53, 1.4]","[53, 51]"
mmlb-1006,B,visual reading/perception,"Regarding macbook high security solution, which numbers of feature...",4,"[4, 5]"
mmlb-0065,B,table-cell alignment,"From the paper, which temperature gives ChatGPT the highest alignm...",0.1,0.5
mmlb-0077,B,counting/spatial,"In figure 3, how many distinct icons are used?",9,6
mmlb-0081,B,exhaustive list extraction,"Which subsection does the section ""AUGMENTATION PROCESS IN RAG"" in...","['Iterative Retrieval ', 'Recursive Retr",Augmentation
mmlb-1003,B,multi-page aggregation,How many weeks do students need to reach work experience milestone...,16,8
mmlb-1059,B,numerical calculation,what is the percentage change of return for allowance from 2016 t...,60.3%,2.6
mmlb-1060,B,semantic/distractor error,what are the components of cost of sales for Amazon's FY2017?,['the purchase price of consumer product,"['Cost of sales', 'Fulfillment', 'Marketing', 'Tec"


,category,justification,reply
qid,,,
mmlb-0043,unsupported answer,false premise ('business cooperation' instead of 'business analyti...,"['Alerts', 'Query Drilldown (OLAP)', 'Ad-Hoc Reports', 'Stan"
mmlb-0068,unsupported answer,false premise ('either' instead of 'both'); answers the original q...,A sentence can map to either [NA] or a list of sub-graph kno
mmlb-0079,unsupported answer,no method meets both constraints; names one,CoG
mmlb-0138,unsupported answer,the person is not in the document; names an agency,U.S. Environmental Protection Agency
mmlb-0205,unsupported answer,asks about a Thinkpad in a MacBook guide; answers for the MacBook,Microphone
mmlb-0406,unsupported answer,no such case in chapter 9; answers from another case,"['purple', 'white']"
mmlb-0408,annotation issue,'How many cats appear?' -> 0 is a defensible answer; the reference...,0
mmlb-0577,unsupported answer,no yellow italic text on page 9; quotes a heading,Recognizing and Responding to a Challenge
mmlb-0596,unsupported answer,no such interview in the complaint; answers with another interview,"Durden was interviewed by agents on October 24, 2016."


## 2. The pages of each arm, and V0 = E6's request, byte for byte

V0's reader is notebook 05's rebuild of the application's reader, written out again here with no change:
- the prompt `mmlb-image-v1`;
- the image-token rule, the context guard and the request format;
- the pre-call check, the parser and the citation checks.

Every V0 request is rebuilt from the question string, ColQwen2's frozen ranking and the cached renders. Its fingerprint must equal the one the frozen E6 run logged for arm V, for all 118 questions. The page lists of all three arms are then fixed:
- **X4** uses V0's supplied pages exactly;
- **X8** uses the first 8 pages of the same frozen ranking.

In [7]:
# V0's reader: notebook 05's rebuild of the application's reader (src/visualdoc/answering.py and citations.py at E4's pinned hashes), unchanged.
assert V0_READER == E6_SPEC["reader"], "V0's reader settings must equal E6's"
READER = V0_READER
IMAGE_PROMPT_VERSION = READER["system_prompt_version"]
IMAGE_SYSTEM_PROMPT = """You answer questions about a single PDF document using only the page images provided.
Each page image is preceded by a tag [Page N], where N is the 1-based page number in the PDF.

Return a JSON object with exactly these keys:
  "answer": the shortest complete answer. Use a number (integer or decimal) when the answer is a number,
            and a JSON list when several items are asked for. If the page images do not contain the answer,
            use exactly the string "Not answerable".
  "cited_pages": a list of the page numbers N whose images support the answer ([] if "Not answerable").

Do not use knowledge outside the page images. Do not guess."""
assert sha256_bytes((IMAGE_PROMPT_VERSION + IMAGE_SYSTEM_PROMPT).encode())[:12] == READER["system_prompt_hash"] == "35ff2e8b91f9"
SNAPSHOT = MODEL["requested_snapshot"]
CONTEXT_TOKENS = MODEL["context_window_tokens"]
MAX_OUTPUT_TOKENS = READER["max_completion_tokens"]
FRAMING_TOKENS = READER["framing_token_allowance"]
TOP_K = READER["top_k"]
IMAGE_BASE_TOKENS, IMAGE_TILE_TOKENS = 2_833, 5_667
PNG_SIGNATURE = b"\x89PNG\r\n\x1a\n"
ENCODING = tiktoken.get_encoding("o200k_base")
MESSAGE_OVERHEAD_TOKENS = 9  # chat framing beyond the text tokens (notebook 02 section 5; +1 billed on every frozen image call)


def png_size(png):
    if not isinstance(png, bytes) or not png.startswith(PNG_SIGNATURE) or len(png) < 24:
        raise ValueError("page image is not a PNG")
    return int.from_bytes(png[16:20], "big"), int.from_bytes(png[20:24], "big")


def image_tokens(width, height):
    # GPT-4o-mini, detail high: fit in 2048 x 2048, shrink (never enlarge) the short side to 768, then 2,833 + 5,667 per 512-px tile.
    scale = min(1.0, 2048 / max(width, height))
    width, height = width * scale, height * scale
    short_scale = 768 / min(width, height)
    if short_scale < 1:
        width, height = width * short_scale, height * short_scale
    return IMAGE_BASE_TOKENS + IMAGE_TILE_TOKENS * ceil(round(width, 6) / 512) * ceil(round(height, 6) / 512)


def question_header(question):
    return f"Question: {question}\n\nPage images:"


def page_tag(number):
    return f"[Page {number}]"


def guard(question, ranked_pages, limit=CONTEXT_TOKENS - MAX_OUTPUT_TOKENS):
    # ranked_pages: [(page, png), ...] best first. Keep the longest prefix whose prompt-token bound fits; never reorder.
    total = FRAMING_TOKENS + len(IMAGE_SYSTEM_PROMPT.encode()) + len(question_header(question).encode())
    kept = []
    for number, png in ranked_pages:
        cost = len(page_tag(number).encode()) + image_tokens(*png_size(png))
        if total + cost > limit:
            break
        kept.append((number, png))
        total += cost
    return kept, [number for number, _ in ranked_pages[len(kept):]]


def build_request(question, pages):
    if not str(question).strip():
        raise ValueError("the question is empty")
    if not pages or len({number for number, _ in pages}) != len(pages):
        raise ValueError("at least one page is needed, and no page may be supplied twice")
    content = [{"type": "text", "text": question_header(question)}]
    for number, png in pages:
        png_size(png)
        content += [{"type": "text", "text": page_tag(number)},
                    {"type": "image_url", "image_url": {"url": "data:image/png;base64," + base64.b64encode(png).decode("ascii"), "detail": READER["image_detail"]}}]
    return {"model": SNAPSHOT, "messages": [{"role": "system", "content": IMAGE_SYSTEM_PROMPT}, {"role": "user", "content": content}],
            "response_format": {"type": READER["response_format"]}, "max_completion_tokens": MAX_OUTPUT_TOKENS, "temperature": READER["temperature"]}


def decoded_image(part):
    return base64.b64decode(part["image_url"]["url"].split(",", 1)[1])


def loggable(request):
    # The request with every image replaced by its SHA-256 and size: safe to save, no Base64.
    def strip(part):
        if part.get("type") == "image_url":
            raw = decoded_image(part)
            return {"type": "image", "sha256": sha256_bytes(raw), "bytes": len(raw), "detail": part["image_url"].get("detail")}
        return part

    messages = [{"role": m["role"], "content": m["content"] if isinstance(m["content"], str) else [strip(p) for p in m["content"]]} for m in request["messages"]]
    return {**{key: value for key, value in request.items() if key != "messages"}, "messages": messages}


def request_fingerprint(request):
    return sha256_bytes(json.dumps(loggable(request), sort_keys=True).encode())


def request_parts(request):
    texts, images = [], []
    for message in request["messages"]:
        for part in [{"type": "text", "text": message["content"]}] if isinstance(message["content"], str) else message["content"]:
            if part["type"] == "text":
                texts.append(part["text"])
            elif part["type"] == "image_url":
                png = decoded_image(part)
                width, height = png_size(png)
                images.append({"sha256": sha256_bytes(png), "width_px": width, "height_px": height})
            else:
                raise ValueError(f"unexpected {part['type']!r} part")
    return texts, images


def expected_prompt_tokens(request):
    # The exact expectation: o200k_base tokens of every text + 9 + billed image tokens.
    texts, images = request_parts(request)
    return sum(len(ENCODING.encode(t)) for t in texts) + MESSAGE_OVERHEAD_TOKENS + sum(image_tokens(i["width_px"], i["height_px"]) for i in images)


def prompt_token_bound(request):
    # The application's upper bound (no tokenizer): byte-level BPE never needs more tokens than UTF-8 bytes; + 100 framing.
    texts, images = request_parts(request)
    return FRAMING_TOKENS + sum(len(t.encode("utf-8")) for t in texts) + sum(image_tokens(i["width_px"], i["height_px"]) for i in images)


def call_cost_upper(bound, max_output_tokens):
    return bound * PRICE["input"] + max_output_tokens * PRICE["output"]


def check_request(request, max_cost_usd=MODEL["max_request_cost_usd"]):
    # The application's pre-call check: raises (nothing is sent) on an unknown model, unbounded output, context overflow or cost.
    if request.get("model") != SNAPSHOT:
        raise ValueError(f"no context size or price is known for model {request.get('model')!r}")
    output = request.get("max_completion_tokens")
    if not isinstance(output, int) or isinstance(output, bool) or output < 1:
        raise ValueError("the request must bound its output with a positive max_completion_tokens")
    bound = prompt_token_bound(request)
    if bound + output > CONTEXT_TOKENS:
        raise ValueError(f"the request could need {bound + output:,} tokens ({bound:,} prompt + {output:,} output), "
                         f"more than the {CONTEXT_TOKENS:,}-token context")
    cost = bound * PRICE["input"] + output * PRICE["output"]
    if cost > max_cost_usd:
        raise ValueError(f"the request could cost ${cost:.4f}, above the ${max_cost_usd:.4f} per-request ceiling")
    return bound, cost


def cited_page_number(value):
    # A JSON integer, a whole float such as 2.0, or ASCII digits such as "4"; never rounded; pages start at 1.
    if isinstance(value, int) and not isinstance(value, bool):
        number = value
    elif isinstance(value, float) and value.is_integer():
        number = int(value)
    elif isinstance(value, str) and value.isascii() and value.isdigit():
        number = int(value)
    else:
        raise ValueError(f"cited page {value!r} is not a page number")
    if number < 1:
        raise ValueError(f"cited page {value!r} is not a page number (pages start at 1)")
    return number


def parse_reply(raw_text, finish_reason="stop"):
    # (answer, cited pages, error or None), as the application's parse_reply plus ask's finish-reason rule.
    try:
        payload = json.loads(raw_text)
        answer = payload["answer"]
        cited_pages = payload.get("cited_pages", [])
        if not isinstance(cited_pages, list):
            raise TypeError(f"cited_pages is a {type(cited_pages).__name__}, not a list")
        cited = [cited_page_number(page) for page in cited_pages]
    except (json.JSONDecodeError, KeyError, TypeError, ValueError) as error:
        answer, cited, problem = "", [], f"unparseable model output: {error}"
    else:
        problem = None
        if isinstance(answer, str) and answer.strip().lower() == "not answerable":
            answer = NOT_ANSWERABLE
    if finish_reason != "stop" and problem is None:
        problem = f"reply did not finish normally (finish_reason={finish_reason!r})"
    return answer, cited, problem


def citation_check(answer, cited, supplied, evidence=None):
    # valid: every cited page was supplied; off_by_one: every invalid page = a supplied page - 1 (the printed page number);
    # missing: an answer other than "Not answerable" without a citation; cites_evidence: needs the annotation (evaluation only).
    cited = list(dict.fromkeys(cited))
    supplied = set(supplied)
    invalid = [page for page in cited if page not in supplied]
    abstained = answer == NOT_ANSWERABLE
    return {"valid": not invalid, "off_by_one": bool(invalid) and all(page + 1 in supplied for page in invalid),
            "missing": not abstained and not cited, "abstained": abstained,
            "cites_evidence": None if evidence is None else bool(set(cited) & set(evidence))}


# Hand checks: billed image sizes of the frozen runs, and each parser branch.
assert image_tokens(768, 994) == 25_501 and image_tokens(1024, 768) == 25_501 and image_tokens(1366, 768) == 36_835
assert parse_reply('{"answer": "not answerable", "cited_pages": []}') == (NOT_ANSWERABLE, [], None)
assert parse_reply('{"answer": 12, "cited_pages": ["4", 2.0]}') == (12, [4, 2], None)
assert parse_reply('{"answer": 12, "cited_pages": [2.7]}')[2].startswith("unparseable")
assert parse_reply('{"answer": 12, "cited_pages": [true]}')[2].startswith("unparseable")
assert parse_reply('{"answer": "Europe IPO", "cited_pa', "length")[2].startswith("unparseable")
assert parse_reply('{"answer": 5, "cited_pages": [3]}', "length")[2] == "reply did not finish normally (finish_reason='length')"
assert citation_check(5, [3, 7], [3, 8]) == {"valid": False, "off_by_one": True, "missing": False, "abstained": False, "cites_evidence": None}
assert citation_check("Not answerable", [], [3], evidence=[3])["missing"] is False and citation_check(5, [], [3])["missing"] is True
print("V0 reader written out; hand checks of the image-token rule, the parser and the citation checks passed.")

V0 reader written out; hand checks of the image-token rule, the parser and the citation checks passed.


In [8]:
# Every V0 request, rebuilt from the question string, the frozen ranking and the cached renders; it must equal E6's V request.
V0_REQUESTS = {}
for qid in QIDS:
    question, doc_id = question_of[qid], doc_of[qid]
    top5 = RANKING[qid][:TOP_K]
    kept, dropped = guard(question, [(page, manifest_png(doc_id, page)) for page in top5])
    request = build_request(question, kept)
    bound, _ = check_request(request)
    images = request_parts(request)[1]
    V0_REQUESTS[qid] = {"qid": qid, "kind": "v0", "doc_id": doc_id, "page": None, "pages": [p for p, _ in kept], "request": request,
                        "fingerprint": request_fingerprint(request), "image_sha256": [i["sha256"] for i in images],
                        "expected_prompt_tokens": expected_prompt_tokens(request), "prompt_token_bound": bound,
                        "max_output_tokens": request["max_completion_tokens"], "cost_upper_usd": call_cost_upper(bound, request["max_completion_tokens"])}

# V0 equals the frozen E6 V request for every question: fingerprint, pages, image hashes, expected and bound tokens.
for qid, entry in V0_REQUESTS.items():
    frozen = E6_RECORDS["V"][qid]
    assert entry["fingerprint"] == frozen["fingerprint"] == frozen["sent_fingerprint"], qid
    assert entry["pages"] == frozen["supplied_pages"] and entry["image_sha256"] == frozen["image_sha256"], qid
    assert entry["expected_prompt_tokens"] == frozen["expected_prompt_tokens"] and entry["prompt_token_bound"] == frozen["prompt_token_bound"], qid

# The page lists of the three arms.
X8_K = SPEC["design"]["arms"]["X8"]["pages_k"]
ARM_PAGES = {"V0": {q: V0_REQUESTS[q]["pages"] for q in QIDS}, "X4": {q: list(V0_REQUESTS[q]["pages"]) for q in QIDS}, "X8": {q: RANKING[q][:X8_K] for q in QIDS}}
for q in QIDS:
    x4, x8 = ARM_PAGES["X4"][q], ARM_PAGES["X8"][q]
    assert x4 == ARM_PAGES["V0"][q] and x8[: len(x4)] == x4 and len(x8) == X8_K == len(set(x8)), q
EXTRACTION_UNITS = [(q, p) for q in QIDS for p in ARM_PAGES["X8"][q]]  # every (question, page) read once: X4's pages are X8's first pages
assert len(EXTRACTION_UNITS) == len(set(EXTRACTION_UNITS)) == len(QIDS) * X8_K == SPEC["execution"]["plan_composition"]["per_replicate"]["extract"]
pages_table = pd.DataFrame({arm: Counter(len(ARM_PAGES[arm][q]) for q in QIDS) for arm in ARMS}).fillna(0).astype(int).sort_index()
pages_table.index.name = "pages per question"
assert pages_table.to_dict()["V0"] == {3: 13, 4: 105, 8: 0} and pages_table.to_dict()["X8"] == {3: 0, 4: 0, 8: 118}
display(pages_table.T)
reuse = sum(len(ARM_PAGES["X4"][q]) for q in QIDS)
print(f"V0 = the frozen E6 V request for {len(V0_REQUESTS)}/{len(QIDS)} questions (fingerprint, pages, image hashes, tokens). X4 reads V0's pages "
      f"({reuse} page readings), X8 the top {X8_K} ({len(EXTRACTION_UNITS)} page readings); the {reuse} X4 readings are X8's first pages, so "
      f"{len(EXTRACTION_UNITS)} extraction calls serve {reuse + len(EXTRACTION_UNITS)} page readings.")

pages per question,3,4,8
V0,13,105,0
X4,13,105,0
X8,0,0,118


V0 = the frozen E6 V request for 118/118 questions (fingerprint, pages, image hashes, tokens). X4 reads V0's pages (459 page readings), X8 the top 8 (944 page readings); the 459 X4 readings are X8's first pages, so 944 extraction calls serve 1403 page readings.


## 3. Stage 1: page-wise evidence extraction

One VLM call per (question, page). The model sees the question and **one** page image, tagged `[Page N]` exactly as in V0, and returns page-local evidence only:

```json
{"page": 40, "relevant": true,
 "facts": [{"label": "...", "value": 20522, "unit": "USD millions", "period": "2017", "supporting_text": "..."}],
 "notes": null}
```

- **Provenance comes from the request, not the reply.** The page number the model reports is checked; a mismatch is recorded, never trusted.
- **`relevant` is diagnostic only.** It never removes a page or a fact from the Evidence Memory.
- **The parser is defensive, not forgiving.**
  - A reply that is not a JSON object, has no `facts` list, or did not finish normally makes the page *unreadable*. It then enters the memory as such.
  - A single malformed fact is dropped and counted.
  - A list of scalar values is split into one fact per value, so nothing is merged.
- **What the prompt contains.** Nothing beyond the question, the page tag and the image: no answer format, no evidence annotation, no question identifier.

In [9]:
# Stage 1: the extraction prompt, request and parser.
EXTRACTION_PROMPT_VERSION = EXTRACTION["prompt_version"]
EXTRACTION_SYSTEM_PROMPT = """You read one page image from a PDF document. You are also given a question about the whole document.
Do not answer the question. Your only task is to record what this page shows that could help someone answer it.
The page image is preceded by a tag [Page N], where N is the 1-based page number in the PDF.

Return a JSON object with exactly these keys:
  "page": N,
  "relevant": true if anything on this page bears on the question, otherwise false,
  "facts": a list of facts read from this page ([] if none). Each fact is an object with the keys
      "label": what the value is: the entity, list, table row and column, chart series or visual element it belongs to,
      "value": the value exactly as shown on this page (a number, a short text, a name, one list item,
               or a count or description of something visible on this page),
      "unit": the unit or scale if one is shown (for example "%" or "USD millions"), otherwise null,
      "period": the year, date or period the value refers to if one is shown, otherwise null,
      "supporting_text": a short verbatim snippet from the page, or a short description of where the value appears,
  "notes": anything that makes these facts uncertain or incomplete (for example a value that is hard to read), otherwise null.

Rules:
- Use only this page. Do not use outside knowledge, and do not infer facts that are not shown on this page.
- Do not calculate, combine or compare values, and do not state an answer to the question. Record each value as a separate fact.
- When the question concerns several items, record every one of them that appears on this page.
- For a value in a table, include both its row and its column in the label.
- If nothing on this page bears on the question, return "relevant": false and "facts": []."""
assert sha256_bytes((EXTRACTION_PROMPT_VERSION + EXTRACTION_SYSTEM_PROMPT).encode())[:12] == EXTRACTION["system_prompt_hash"]
EXTRACTION_MAX_OUTPUT = EXTRACTION["max_completion_tokens"]
MAX_FACTS_PER_PAGE = EXTRACTION["max_facts_per_page"]
FACT_FIELDS = ("label", "value", "unit", "period", "supporting_text")


def extraction_header(question):
    return f"Question: {question}\n\nPage image:"


def build_extraction_request(question, page, png):
    # Built from the question text, the page number and the page image only.
    if not str(question).strip():
        raise ValueError("the question is empty")
    png_size(png)
    content = [{"type": "text", "text": extraction_header(question)}, {"type": "text", "text": page_tag(page)},
               {"type": "image_url", "image_url": {"url": "data:image/png;base64," + base64.b64encode(png).decode("ascii"), "detail": EXTRACTION["image_detail"]}}]
    return {"model": SNAPSHOT, "messages": [{"role": "system", "content": EXTRACTION_SYSTEM_PROMPT}, {"role": "user", "content": content}],
            "response_format": {"type": EXTRACTION["response_format"]}, "max_completion_tokens": EXTRACTION_MAX_OUTPUT,
            "temperature": EXTRACTION["temperature"]}


def _optional_text(value):
    # null, a string, or a number (for example a year written as 2017), kept as text; anything else is malformed.
    if value is None:
        return None
    if isinstance(value, str):
        return value.strip() or None
    if isinstance(value, (int, float)) and not isinstance(value, bool) and isfinite(value):
        return str(value)
    raise ValueError(f"{type(value).__name__} is not text")


def _scalar(value):
    if isinstance(value, str) and value.strip():
        return value.strip()
    if isinstance(value, bool) or (isinstance(value, (int, float)) and isfinite(value)):
        return value
    raise ValueError(f"value {value!r} is not a scalar")


def normalize_facts(raw_fact):
    # One reported fact -> a list of clean facts (a list of scalar values becomes one fact per value; nothing is merged).
    if not isinstance(raw_fact, dict):
        raise ValueError("a fact is not an object")
    label = raw_fact.get("label")
    if not isinstance(label, str) or not label.strip():
        raise ValueError("a fact has no label")
    optional = {key: _optional_text(raw_fact.get(key)) for key in ("unit", "period", "supporting_text")}
    values = raw_fact.get("value")
    values = values if isinstance(values, list) and values else [values]
    return [{"label": label.strip(), "value": _scalar(v), **optional} for v in values]


def parse_extraction(raw_text, finish_reason, page):
    # -> {status 'ok' | 'unreadable', error, relevant (diagnostic), facts, notes, dropped_facts, split_facts, reported_page, page_mismatch}
    result = {"status": "unreadable", "error": None, "relevant": None, "facts": [], "notes": None, "dropped_facts": 0, "split_facts": 0,
              "reported_page": None, "page_mismatch": False}
    if finish_reason != "stop":
        result["error"] = f"reply did not finish normally (finish_reason={finish_reason!r})"
        return result
    try:
        payload = json.loads(raw_text)
        if not isinstance(payload, dict):
            raise TypeError("the reply is not a JSON object")
        if not isinstance(payload.get("facts"), list):
            raise TypeError("'facts' is missing or not a list")
    except (json.JSONDecodeError, TypeError) as error:
        result["error"] = f"unparseable extraction: {error}"
        return result
    facts = []
    for raw_fact in payload["facts"]:
        try:
            clean = normalize_facts(raw_fact)
        except ValueError:
            result["dropped_facts"] += 1
            continue
        result["split_facts"] += len(clean) - 1
        facts.extend(clean)
    result["dropped_facts"] += max(0, len(facts) - MAX_FACTS_PER_PAGE)
    relevant = payload.get("relevant")
    try:
        notes = _optional_text(payload.get("notes"))
    except ValueError:
        notes = None
    reported = payload.get("page")
    try:
        mismatch = reported is not None and cited_page_number(reported) != page
    except ValueError:
        mismatch = True
    result.update(status="ok", facts=facts[:MAX_FACTS_PER_PAGE], relevant=relevant if isinstance(relevant, bool) else None, notes=notes,
                  reported_page=reported, page_mismatch=mismatch)
    return result


# Hand checks (invented replies; interface tests only).
ok_reply = json.dumps({"page": 40, "relevant": True, "notes": None, "facts": [
    {"label": "Revenue", "value": "1,234", "unit": "USD millions", "period": 2020, "supporting_text": "Revenue ... 1,234"},
    {"label": "Colours listed", "value": ["Blue", "Green"], "unit": None, "period": None, "supporting_text": None},
    {"label": "", "value": 3}, {"value": 4}, "not an object", {"label": "nested", "value": {"a": 1}}]})
parsed = parse_extraction(ok_reply, "stop", 40)
assert parsed["status"] == "ok" and parsed["relevant"] is True and parsed["dropped_facts"] == 4 and parsed["split_facts"] == 1
assert [f["value"] for f in parsed["facts"]] == ["1,234", "Blue", "Green"] and parsed["facts"][0]["period"] == "2020"
assert parsed["facts"][1]["label"] == parsed["facts"][2]["label"] == "Colours listed" and not parsed["page_mismatch"]
assert parse_extraction('{"page": 41, "relevant": false, "facts": []}', "stop", 40)["page_mismatch"] is True
assert parse_extraction('{"relevant": "yes", "facts": []}', "stop", 40)["relevant"] is None  # diagnostic only: never fails the page
for bad, finish in (('{"page": 40, "facts": [{"label": "x", "val', "length"), ("not json", "stop"), ('{"facts": "none"}', "stop"),
                    ('["a list"]', "stop"), ('{"page": 40, "relevant": true}', "stop")):
    assert parse_extraction(bad, finish, 40)["status"] == "unreadable", bad
assert list(inspect.signature(build_extraction_request).parameters) == ["question", "page", "png"]
print(f"Extraction prompt {EXTRACTION_PROMPT_VERSION} (hash {EXTRACTION['system_prompt_hash']}); parser hand checks pass "
      f"(valid, split list, dropped facts, page mismatch, truncated, non-JSON, missing facts).")

Extraction prompt e7-extract-v1 (hash b30616381e85); parser hand checks pass (valid, split list, dropped facts, page mismatch, truncated, non-JSON, missing facts).


## 4. Stage 2: the Evidence Memory, deterministic arithmetic, and text-only reasoning

**Evidence Memory.** Built deterministically from the extraction replies of the arm's pages, in ColQwen2's rank order:

$$E = \Big\{\, (\mathrm{id} = p.i,\ \mathrm{page} = p,\ \mathrm{label},\ \mathrm{value},\ \mathrm{unit},\ \mathrm{period},\ \mathrm{supporting\_text}) \ : \ p \in P,\ i = 1..n_p \Big\}$$

It also holds one line per page read: status (*facts recorded*, *nothing relevant recorded* or *unreadable*) and the model's notes. Nothing is merged, deduplicated or filtered, and the diagnostic `relevant` flag is left out. Along with the question, the memory is **the only input** of the reasoning step.

**Arithmetic.** The reasoning step may return a calculation: an expression tree whose every leaf is a fact id from the memory. There are no free constants, so every operand carries its page.
- **Operations.** A closed set: `add`, `subtract`, `multiply`, `divide`, `sum`, `mean`, `min`, `max`, `percent_change` ((new − old) / |old| × 100), `greater`, `less`, `equal` (Yes / No), `count`, `argmax`, `argmin` (the label of the extreme fact).
- **Parsing fact values.** Strict: thousands separators, currency signs, a trailing %, and parentheses for negatives are accepted. Anything else ("20 million", "about 5") is refused.
- **Rounding.** Half-up, to the number of decimals the question requested.
- **Limits.** Depth ≤ 6 and ≤ 64 nodes.
- **No code execution.** Nothing is executed: the tree is walked by a small interpreter.

**Final answer.**
- When the calculation evaluates, its result is the answer, and its operand pages are added to the cited pages.
- When it fails, the model's own `answer` is used and the failure is recorded.
- A calculation attached to "Not answerable" is ignored.

In [10]:
# The Evidence Memory: deterministic, provenance-preserving, nothing collapsed.
MEMORY_PAGE_KEYS = ("page", "status", "notes")
MEMORY_FACT_KEYS = ("id", "page") + FACT_FIELDS
PAGE_STATUS = {"ok_with_facts": "facts recorded", "ok_without_facts": "nothing relevant recorded", "unreadable": "unreadable"}


def build_memory(pages, extractions):
    # pages: the arm's pages in rank order; extractions: page -> parse_extraction() result for this question.
    memory = {"pages": [], "facts": []}
    for page in pages:
        extraction = extractions[page]
        if extraction["status"] != "ok":
            status = PAGE_STATUS["unreadable"]
        else:
            status = PAGE_STATUS["ok_with_facts"] if extraction["facts"] else PAGE_STATUS["ok_without_facts"]
        memory["pages"].append({"page": page, "status": status, "notes": extraction["notes"]})
        for index, fact in enumerate(extraction["facts"], start=1):
            memory["facts"].append({"id": f"p{page}.f{index}", "page": page, **{key: fact[key] for key in FACT_FIELDS}})
    return memory


def validate_memory(memory, pages):
    # Only the schema's keys, the planned pages in order, unique ids, every fact on a planned page.
    assert set(memory) == {"pages", "facts"}
    assert [entry["page"] for entry in memory["pages"]] == list(pages)
    assert all(tuple(entry) == MEMORY_PAGE_KEYS and entry["status"] in PAGE_STATUS.values() for entry in memory["pages"])
    assert all(tuple(fact) == MEMORY_FACT_KEYS and fact["page"] in pages for fact in memory["facts"])
    ids = [fact["id"] for fact in memory["facts"]]
    assert len(ids) == len(set(ids))
    return True


def memory_text(memory):
    # Valid JSON, one page or fact per line (readable and deterministic).
    pages = ",\n  ".join(json.dumps(entry, ensure_ascii=False) for entry in memory["pages"])
    facts = ",\n  ".join(json.dumps(fact, ensure_ascii=False) for fact in memory["facts"])
    return '{"pages": [\n  ' + pages + '\n ],\n "facts": [\n  ' + facts + "\n ]}"


# Checks on invented extractions (interface tests only).
_toy = {40: parse_extraction(json.dumps({"page": 40, "relevant": True, "facts": [
            {"label": "Revenue", "value": "1,234", "unit": "USD millions", "period": "2020", "supporting_text": "Revenue $ 1,234"},
            {"label": "Cost", "value": "2,000", "unit": "USD millions", "period": "2020", "supporting_text": None},
            {"label": "Total", "value": "5,000", "unit": "USD millions", "period": "2020", "supporting_text": None}],
            "notes": None}), "stop", 40),
        36: parse_extraction('{"page": 36, "relevant": false, "facts": [], "notes": "a contents page; no figures"}', "stop", 36),
        74: parse_extraction('{"page": 74, "relevant": true, "facts": [{"label": "x", "val', "length", 74)}
_toy_memory = build_memory([40, 36, 74], _toy)
assert validate_memory(_toy_memory, [40, 36, 74]) and json.loads(memory_text(_toy_memory)) == _toy_memory
assert [f["id"] for f in _toy_memory["facts"]] == ["p40.f1", "p40.f2", "p40.f3"] and {f["page"] for f in _toy_memory["facts"]} == {40}
assert [p["status"] for p in _toy_memory["pages"]] == ["facts recorded", "nothing relevant recorded", "unreadable"]
assert _toy_memory["pages"][1]["notes"] == "a contents page; no figures"
assert '"relevant":' not in memory_text(_toy_memory) and all("relevant" not in f for f in _toy_memory["facts"])  # the diagnostic flag never reaches the reasoning step
# Order follows the rank order given, not page numbers; nothing is deduplicated.
_dup = {5: parse_extraction('{"facts": [{"label": "a", "value": 1}, {"label": "a", "value": 1}]}', "stop", 5),
        2: parse_extraction('{"facts": [{"label": "a", "value": 1}]}', "stop", 2)}
assert [f["id"] for f in build_memory([5, 2], _dup)["facts"]] == ["p5.f1", "p5.f2", "p2.f1"]
print("Evidence Memory: builder, schema validation and serialization checks pass (provenance kept, order kept, duplicates kept, "
      "unreadable pages marked, 'relevant' withheld).")

Evidence Memory: builder, schema validation and serialization checks pass (provenance kept, order kept, duplicates kept, unreadable pages marked, 'relevant' withheld).


In [11]:
# The deterministic arithmetic: a small interpreter over an expression tree whose leaves are fact ids. No code is executed.
ARITH_OPS = ARITHMETIC["operations"]  # name -> [minimum arguments, maximum arguments or null]
MAX_DEPTH, MAX_NODES, MAX_ROUND = ARITHMETIC["max_depth"], ARITHMETIC["max_nodes"], ARITHMETIC["max_round_decimals"]
FACT_ONLY_OPS = {"count", "argmax", "argmin"}
_NUMBER = re.compile(r"[+-]?(\d+(\.\d*)?|\.\d+)")


class CalculationError(ValueError):
    pass


def parse_number(value):
    # A fact value as a number: thousands separators, a currency sign, a trailing % and (negative) parentheses are accepted; nothing else.
    if isinstance(value, bool):
        raise CalculationError("a yes/no value is not a number")
    if isinstance(value, (int, float)):
        if not isfinite(value):
            raise CalculationError("not a finite number")
        return float(value)
    if not isinstance(value, str):
        raise CalculationError(f"{value!r} is not a number")
    text = value.strip().replace("−", "-").replace(" ", " ")
    negative = text.startswith("(") and text.endswith(")")
    if negative:
        text = text[1:-1].strip()
    text = re.sub(r"^([+-]?)\s*(US\$|USD|\$|€|£|¥)\s*", r"\1", text)
    text = text.replace(",", "").replace(" ", "")
    if text.endswith("%"):
        text = text[:-1]
    if not _NUMBER.fullmatch(text):
        raise CalculationError(f"{value!r} is not a plain number")
    number = float(text)
    return -number if negative else number


def _round_half_up(value, decimals):
    quantum = Decimal(1).scaleb(-decimals)
    # Normalized to 12 significant digits first, so binary noise (69.94999999999999 for 69.95) does not decide the rounding.
    rounded = Decimal(f"{value:.12g}").quantize(quantum, rounding=ROUND_HALF_UP)
    return int(rounded) if decimals == 0 else float(rounded)


def evaluate_calculation(calculation, memory):
    # -> {"status": "ok", "result", "operand_facts", "operand_pages"} or {"status": "error", "error"}
    facts = {fact["id"]: fact for fact in memory["facts"]}
    used, nodes = [], [0]

    def fact_of(node):
        if not isinstance(node, dict) or set(node) != {"fact"}:
            raise CalculationError("this operation takes fact references only")
        if node["fact"] not in facts:
            raise CalculationError(f"unknown fact id {node['fact']!r}")
        used.append(node["fact"])
        return facts[node["fact"]]

    def number(value):
        if isinstance(value, bool) or not isinstance(value, float):
            raise CalculationError("an operand is not a number")
        return value

    def walk(node, depth):
        nodes[0] += 1
        if nodes[0] > MAX_NODES or depth > MAX_DEPTH:
            raise CalculationError("the expression is too large")
        if isinstance(node, dict) and set(node) == {"fact"}:
            return parse_number(fact_of(node)["value"])
        if not isinstance(node, dict) or set(node) != {"op", "args"} or not isinstance(node["args"], list):
            raise CalculationError(f"malformed node {str(node)[:60]}")
        op, args = node["op"], node["args"]
        if op not in ARITH_OPS:
            raise CalculationError(f"unknown operation {op!r}")
        low, high = ARITH_OPS[op]
        if len(args) < low or (high is not None and len(args) > high):
            raise CalculationError(f"{op} takes {low}..{high} arguments, got {len(args)}")
        if op == "count":
            for arg in args:
                fact_of(arg)
            return float(len(args))
        if op in ("argmax", "argmin"):
            pairs = [(parse_number(fact_of(arg)["value"]), facts[arg["fact"]]["label"]) for arg in args]
            best = (max if op == "argmax" else min)(value for value, _ in pairs)
            labels = {label for value, label in pairs if value == best}
            if len(labels) != 1:
                raise CalculationError(f"{op}: tie between {sorted(labels)}")
            return labels.pop()
        values = [number(walk(arg, depth + 1)) for arg in args]
        if op in ("add", "sum"):
            return float(sum(values))
        if op == "subtract":
            return values[0] - values[1]
        if op == "multiply":
            return float(np.prod(values))
        if op == "divide":
            if values[1] == 0:
                raise CalculationError("division by zero")
            return values[0] / values[1]
        if op == "mean":
            return float(sum(values) / len(values))
        if op in ("min", "max"):
            return float((min if op == "min" else max)(values))
        if op == "percent_change":
            if values[0] == 0:
                raise CalculationError("percent change from zero")
            return (values[1] - values[0]) / abs(values[0]) * 100
        return {"greater": values[0] > values[1], "less": values[0] < values[1], "equal": values[0] == values[1]}[op]

    try:
        if not isinstance(calculation, dict) or "expression" not in calculation or not set(calculation) <= {"expression", "round"}:
            raise CalculationError("a calculation is an object with 'expression' and optional 'round'")
        decimals = calculation.get("round")
        if decimals is not None and (isinstance(decimals, bool) or not isinstance(decimals, int) or not 0 <= decimals <= MAX_ROUND):
            raise CalculationError(f"round must be null or an integer 0..{MAX_ROUND}")
        result = walk(calculation["expression"], 1)
        if isinstance(result, bool):
            result = "Yes" if result else "No"
        elif isinstance(result, float):
            if not isfinite(result) or abs(result) > 1e15:
                raise CalculationError("the result is not a finite number")
            if decimals is not None:
                result = _round_half_up(result, decimals)
            elif result.is_integer():
                result = int(result)
            else:
                result = float(f"{result:.10g}")
    except CalculationError as error:
        return {"status": "error", "error": str(error)}
    operand_facts = list(dict.fromkeys(used))
    return {"status": "ok", "result": result, "operand_facts": operand_facts,
            "operand_pages": list(dict.fromkeys(facts[f]["page"] for f in operand_facts))}


# Checks (invented memories; interface tests only).
F = lambda fid: {"fact": fid}
OP = lambda name, *args: {"op": name, "args": list(args)}
_m = {"pages": [], "facts": [
    {"id": "p40.f1", "page": 40, "label": "Item A", "value": "1,234", "unit": None, "period": None, "supporting_text": None},
    {"id": "p40.f2", "page": 40, "label": "Item B", "value": "2,000", "unit": None, "period": None, "supporting_text": None},
    {"id": "p40.f3", "page": 40, "label": "Item C", "value": "$5,000", "unit": None, "period": None, "supporting_text": None},
    {"id": "p43.f1", "page": 43, "label": "Old", "value": 200, "unit": None, "period": None, "supporting_text": None},
    {"id": "p43.f2", "page": 43, "label": "New", "value": "80", "unit": None, "period": None, "supporting_text": None},
    {"id": "p7.f1", "page": 7, "label": "Model A", "value": "71.5", "unit": None, "period": None, "supporting_text": None},
    {"id": "p7.f2", "page": 7, "label": "Model B", "value": "70.1", "unit": None, "period": None, "supporting_text": None},
    {"id": "p7.f3", "page": 7, "label": "Model C", "value": "69.8", "unit": None, "period": None, "supporting_text": None},
    {"id": "p7.f4", "page": 7, "label": "Model D (tie)", "value": "71.5", "unit": None, "period": None, "supporting_text": None},
    {"id": "p9.f1", "page": 9, "label": "Loss", "value": "(1,137)", "unit": None, "period": None, "supporting_text": None},
    {"id": "p9.f2", "page": 9, "label": "Share", "value": "53%", "unit": None, "period": None, "supporting_text": None},
    {"id": "p9.f3", "page": 9, "label": "Size", "value": "20 million", "unit": None, "period": None, "supporting_text": None},
    {"id": "p9.f4", "page": 9, "label": "Flag", "value": True, "unit": None, "period": None, "supporting_text": None},
    {"id": "p9.f5", "page": 9, "label": "Zero", "value": "0", "unit": None, "period": None, "supporting_text": None}]}
ratio = evaluate_calculation({"expression": OP("divide", F("p40.f1"), OP("subtract", F("p40.f3"), F("p40.f2"))), "round": 3}, _m)
assert ratio == {"status": "ok", "result": 0.411, "operand_facts": ["p40.f1", "p40.f3", "p40.f2"], "operand_pages": [40]}
assert evaluate_calculation({"expression": OP("percent_change", F("p43.f1"), F("p43.f2")), "round": 1}, _m)["result"] == -60.0
assert evaluate_calculation({"expression": OP("argmax", F("p7.f1"), F("p7.f2"), F("p7.f3"))}, _m)["result"] == "Model A"
assert evaluate_calculation({"expression": OP("count", F("p7.f1"), F("p7.f2"), F("p9.f4"))}, _m)["result"] == 3
assert evaluate_calculation({"expression": OP("sum", F("p40.f1"), F("p9.f1"))}, _m)["result"] == 97
assert evaluate_calculation({"expression": OP("mean", F("p7.f2"), F("p7.f3")), "round": 1}, _m)["result"] == 70.0  # 69.95 -> 70.0 half-up
assert evaluate_calculation({"expression": OP("greater", F("p9.f2"), F("p7.f3"))}, _m)["result"] == "No"
assert evaluate_calculation({"expression": OP("divide", F("p43.f2"), F("p7.f2"))}, _m)["result"] == 1.141226819
assert _round_half_up(0.0125, 3) == 0.013 and _round_half_up(2.5, 0) == 3  # half-up, not banker's rounding
for bad in ({"expression": F("p99.f1")}, {"expression": OP("power", F("p40.f1"), F("p40.f2"))}, {"expression": OP("subtract", F("p40.f1"))},
            {"expression": OP("divide", F("p40.f1"), F("p9.f5"))}, {"expression": OP("add", F("p40.f1"), F("p9.f3"))},
            {"expression": OP("add", F("p40.f1"), F("p9.f4"))}, {"expression": OP("add", F("p40.f1"), {"constant": 365})},
            {"expression": OP("count", OP("add", F("p40.f1"), F("p40.f2")))}, {"expression": OP("argmax", F("p7.f1"), F("p7.f4"))},
            {"expression": F("p40.f1"), "round": -1}, {"expression": F("p40.f1"), "round": True}, {"expression": F("p40.f1"), "code": "import os"},
            {"expression": "p40.f1 / p40.f2"}, "1 + 1", None, {"expression": OP("percent_change", F("p9.f5"), F("p40.f1"))}):
    assert evaluate_calculation(bad, _m)["status"] == "error", bad
deep = F("p40.f1")
for _ in range(MAX_DEPTH):
    deep = OP("add", deep, F("p40.f2"))
assert evaluate_calculation({"expression": deep}, _m)["status"] == "error"
assert parse_number("-$1,234.5") == -1234.5 and parse_number("(1,137)") == -1137 and parse_number("12%") == 12 and parse_number("−3") == -3
print(f"Arithmetic: {len(ARITH_OPS)} operations; hand checks pass, including 1,234 / (5,000 - 2,000) = 0.411, a -60% "
      f"change, argmax, count, and 17 refusals (unknown fact or operation, arity, division by zero, non-numeric value, constants, code, depth).")

Arithmetic: 15 operations; hand checks pass, including 1,234 / (5,000 - 2,000) = 0.411, a -60% change, argmax, count, and 17 refusals (unknown fact or operation, arity, division by zero, non-numeric value, constants, code, depth).


In [12]:
# Stage 3: text-only reasoning over the Evidence Memory. Its answer contract is V0's, word for word; only the input differs.
REASONING_PROMPT_VERSION = REASONING["prompt_version"]
REASONING_SYSTEM_PROMPT = """You answer questions about a single PDF document using only an evidence memory: facts that were read from some of its pages, one page at a time.
You do not see the pages. Each fact has an id, the page number N it was read from, a label, a value, and, when available, a unit, a period and a supporting snippet.
The memory also lists every page that was read, with its status.

Return a JSON object with exactly these keys:
  "answer": the shortest complete answer. Use a number (integer or decimal) when the answer is a number,
            and a JSON list when several items are asked for. If the evidence memory does not contain the answer,
            use exactly the string "Not answerable".
  "cited_pages": a list of the page numbers N whose facts support the answer ([] if "Not answerable").
  "calculation": null, unless the answer requires arithmetic, counting or comparing numbers. Then an object
            {"expression": E, "round": R}, where R is the number of decimal places the question asks for (null if none) and E is
              {"fact": "<id>"}                          the numeric value of a fact in the memory, or
              {"op": "<operation>", "args": [E, ...]}   with <operation> one of
                  add, subtract, multiply, divide, sum, mean, min, max   (subtract and divide take exactly two arguments)
                  percent_change    [old, new] -> (new - old) / |old| x 100
                  greater, less, equal    two arguments -> Yes or No
                  count             facts -> the number of facts listed
                  argmax, argmin    facts -> the label of the fact with the largest or smallest value
            Every number in a calculation must be a fact from the memory. When you give a calculation, also put your own
            result in "answer"; the calculation is evaluated exactly and its result is used.

Facts from different pages can describe different things: check labels, units and periods before combining them.
Do not use knowledge outside the evidence memory. Do not guess."""
assert sha256_bytes((REASONING_PROMPT_VERSION + REASONING_SYSTEM_PROMPT).encode())[:12] == REASONING["system_prompt_hash"]
REASONING_MAX_OUTPUT = REASONING["max_completion_tokens"]
# The shared answer contract: V0's sentences, verbatim (only 'page images' -> 'evidence memory' differs).
for sentence in ('"answer": the shortest complete answer. Use a number (integer or decimal) when the answer is a number,',
                 "and a JSON list when several items are asked for.", 'use exactly the string "Not answerable".', "Do not guess."):
    assert sentence in IMAGE_SYSTEM_PROMPT and sentence in REASONING_SYSTEM_PROMPT, sentence


def reasoning_user_text(question, memory):
    return f"Question: {question}\n\nEvidence memory:\n{memory_text(memory)}"


def build_reasoning_request(question, memory):
    # Built from the question text and the Evidence Memory only; no image is sent again.
    if not str(question).strip():
        raise ValueError("the question is empty")
    validate_memory(memory, [entry["page"] for entry in memory["pages"]])
    return {"model": SNAPSHOT, "messages": [{"role": "system", "content": REASONING_SYSTEM_PROMPT},
                                            {"role": "user", "content": reasoning_user_text(question, memory)}],
            "response_format": {"type": REASONING["response_format"]}, "max_completion_tokens": REASONING_MAX_OUTPUT, "temperature": REASONING["temperature"]}


def parse_reasoning(raw_text, finish_reason="stop"):
    # (answer, cited pages, calculation or None, error or None): V0's parser rules, plus the optional calculation.
    answer, cited, problem = parse_reply(raw_text, finish_reason)
    calculation = None
    if problem is None:
        calculation = json.loads(raw_text).get("calculation")
    return answer, cited, calculation, problem


def finalize_answer(answer, cited, calculation, memory):
    # The final answer and citations, and a record of what the arithmetic did.
    record = {"model_answer": answer, "model_cited_pages": list(cited), "calculation": calculation, "calc_status": "none", "calc_error": None,
              "operand_facts": [], "operand_pages": []}
    final, final_cited = answer, list(cited)
    if calculation is not None:
        if answer == NOT_ANSWERABLE:
            record["calc_status"] = "ignored: the reply abstained"
        else:
            outcome = evaluate_calculation(calculation, memory)
            if outcome["status"] == "ok":
                final = outcome["result"]
                final_cited = final_cited + [page for page in outcome["operand_pages"] if page not in final_cited]
                record.update(calc_status="ok", operand_facts=outcome["operand_facts"], operand_pages=outcome["operand_pages"])
            else:
                record.update(calc_status="error", calc_error=outcome["error"])
    return final, final_cited, record


# Checks (invented replies; interface tests only).
_req = build_reasoning_request("What is the ratio?", _toy_memory)
assert [m["role"] for m in _req["messages"]] == ["system", "user"] and isinstance(_req["messages"][1]["content"], str)
assert "image_url" not in json.dumps(_req) and check_request(_req)
_calc = {"expression": OP("divide", F("p40.f1"), OP("subtract", F("p40.f3"), F("p40.f2"))), "round": 3}
reply = json.dumps({"answer": 0.2, "cited_pages": [40], "calculation": _calc})
a, c, k, e = parse_reasoning(reply)
assert (a, c, e) == (0.2, [40], None) and k == _calc
assert finalize_answer(a, c, k, _toy_memory)[:2] == (0.411, [40])
assert finalize_answer(a, [], k, _toy_memory)[:2] == (0.411, [40])  # operand pages become citations
assert finalize_answer(NOT_ANSWERABLE, [], k, _toy_memory)[2]["calc_status"] == "ignored: the reply abstained"
bad = finalize_answer(0.2, [40], {"expression": F("p99.f1")}, _toy_memory)
assert bad[0] == 0.2 and bad[2]["calc_status"] == "error"  # a failed calculation falls back to the model's own answer
assert parse_reasoning('{"answer": "not answerable", "cited_pages": []}')[:2] == (NOT_ANSWERABLE, [])
assert parse_reasoning('{"answer": 3, "cited_pages": [2], "calcu', "length")[3] is not None
assert parse_reasoning('{"cited_pages": [2]}')[3].startswith("unparseable")
assert list(inspect.signature(build_reasoning_request).parameters) == ["question", "memory"]
print(f"Reasoning prompt {REASONING_PROMPT_VERSION} (hash {REASONING['system_prompt_hash']}): V0's answer contract verbatim; parser and "
      f"final-answer checks pass (computed result replaces the model's, operand pages cited, abstention kept, failed calculation falls back).")

Reasoning prompt e7-reason-v1 (hash 54cd7a60a931): V0's answer contract verbatim; parser and final-answer checks pass (computed result replaces the model's, operand pages cited, abstention kept, failed calculation falls back).


## 5. Every planned request, and the leakage controls

All 118 V0 requests and all 944 extraction requests are built offline, before any call, and pass the pre-call check: known model, bounded output, context, and a per-request cost ceiling.

The 236 reasoning requests depend on extraction replies, so they are built during the run. Each is built by the same function and checked by the same rules before it is sent.

**Leakage controls** (asserted below; a failure blocks the notebook):
1. **Structural.** Every text part of every planned request is one of:
   - the arm's system prompt;
   - the question header built from the question text;
   - a `[Page N]` tag.

   The reasoning request is the reasoning prompt plus the question and the Evidence Memory, and nothing else. Its memory holds only schema keys, and only values read by the model from the arm's pages.
2. **Fields.** No request carries a question id, a reference answer, an answer format, an evidence-page list, answerability or scoring metadata.
3. **Prompts.**
   - The three system prompts and the templates contain no question id.
   - They contain no reference answer of the 118 questions (4 characters or longer; "Not answerable" is the abstention token every arm shares).
   - They contain no answer-format name.
4. **Builders.** `build_request`, `build_extraction_request` and `build_reasoning_request` take only the question text, page numbers, page images or the memory.
5. **Code.** The notebook file on disk is scanned. No cell that builds, plans or sends a request mentions a question id, the audit labels, the evidence annotations, the answer formats or the reference answers.

In [13]:
# Every extraction request (X8's 8 pages per question; X4 reads the first 3-4 of them), built and checked before any call.
EXTRACT_REQUESTS = {}
for qid, page in EXTRACTION_UNITS:
    request = build_extraction_request(question_of[qid], page, manifest_png(doc_of[qid], page))
    bound, _ = check_request(request)
    images = request_parts(request)[1]
    EXTRACT_REQUESTS[(qid, page)] = {"qid": qid, "kind": "extract", "doc_id": doc_of[qid], "page": page, "pages": [page], "request": request,
                                     "fingerprint": request_fingerprint(request), "image_sha256": [i["sha256"] for i in images],
                                     "expected_prompt_tokens": expected_prompt_tokens(request), "prompt_token_bound": bound,
                                     "max_output_tokens": EXTRACTION_MAX_OUTPUT, "cost_upper_usd": call_cost_upper(bound, EXTRACTION_MAX_OUTPUT)}
assert len({e["fingerprint"] for e in EXTRACT_REQUESTS.values()}) == len(EXTRACT_REQUESTS), "two extraction requests are identical"
assert all(e["image_sha256"] == [held_manifest.loc[(e["doc_id"], e["page"]), "sha256"]] for e in EXTRACT_REQUESTS.values())


def extraction_cache_unit(qid, page):
    # The extraction cache key's content: (question, page, extraction prompt version, model), plus the page image hash and document for safety.
    entry = EXTRACT_REQUESTS[(qid, page)]
    return {"question": question_of[qid], "doc_id": doc_of[qid], "page": page, "image_sha256": entry["image_sha256"][0],
            "extraction_prompt_version": EXTRACTION_PROMPT_VERSION, "extraction_prompt_hash": EXTRACTION["system_prompt_hash"], "model": SNAPSHOT}


EXTRACTION_UNIT_KEY = {unit: sha256_bytes(json.dumps(extraction_cache_unit(*unit), sort_keys=True).encode()) for unit in EXTRACTION_UNITS}
assert len(set(EXTRACTION_UNIT_KEY.values())) == len(EXTRACTION_UNITS)

# Reasoning requests: a planning bound on their prompt tokens (the exact count is computed when each is built during the run).
REASON_PLANNING_PROMPT_TOKENS = REASONING["planning_prompt_token_bound"]
reason_fixed = len(ENCODING.encode(REASONING_SYSTEM_PROMPT)) + max(len(ENCODING.encode(reasoning_user_text(question_of[q], {"pages": [], "facts": []}))) for q in QIDS)
assert reason_fixed + 8 * EXTRACTION_MAX_OUTPUT * 1.4 < REASON_PLANNING_PROMPT_TOKENS  # 8 full extraction replies, +40 % for ids and re-serialization
REASON_COST_UPPER = call_cost_upper(REASON_PLANNING_PROMPT_TOKENS + FRAMING_TOKENS, REASONING_MAX_OUTPUT)
print(f"{len(EXTRACT_REQUESTS)} extraction requests built and checked (unique fingerprints and cache keys; every image is E5's render). "
      f"Reasoning: fixed part <= {reason_fixed:,} tokens; planning bound {REASON_PLANNING_PROMPT_TOKENS:,} prompt tokens per request "
      f"(upper cost ${REASON_COST_UPPER:.5f}).")

944 extraction requests built and checked (unique fingerprints and cache keys; every image is E5's render). Reasoning: fixed part <= 502 tokens; planning bound 14,000 prompt tokens per request (upper cost $0.00247).


In [14]:
# Leakage controls, asserted on every planned request and on the notebook's own request-building code.
FORBIDDEN_FIELDS = ("evidence_pages", "evidence_sources", "answer_format", "answerable", "reference", "score", "qid")
qid_pattern = re.compile(r"mmlb-\d{4}")

# 1-2. Structure and fields of every planned image request.
for qid, entry in V0_REQUESTS.items():
    texts = request_parts(entry["request"])[0]
    assert texts == [IMAGE_SYSTEM_PROMPT, question_header(question_of[qid])] + [page_tag(p) for p in entry["pages"]], qid
for (qid, page), entry in EXTRACT_REQUESTS.items():
    assert request_parts(entry["request"])[0] == [EXTRACTION_SYSTEM_PROMPT, extraction_header(question_of[qid]), page_tag(page)], (qid, page)
for entry in [*V0_REQUESTS.values(), *EXTRACT_REQUESTS.values()]:
    assert set(entry["request"]) == {"model", "messages", "response_format", "max_completion_tokens", "temperature"}
    logged = json.dumps(loggable(entry["request"]))
    assert not qid_pattern.search(logged) and "base64" not in logged
    assert not any(f'"{field}"' in logged for field in FORBIDDEN_FIELDS)


def check_reasoning_request(request, question, memory):
    # The run-time version of controls 1-2 for a reasoning request (also called by the executor before every reasoning call).
    assert set(request) == {"model", "messages", "response_format", "max_completion_tokens", "temperature"}
    assert request["messages"] == [{"role": "system", "content": REASONING_SYSTEM_PROMPT}, {"role": "user", "content": reasoning_user_text(question, memory)}]
    logged = json.dumps(request)
    assert not qid_pattern.search(logged) and "image_url" not in logged
    assert not any(f'"{field}"' in memory_text(memory) for field in FORBIDDEN_FIELDS)
    return True


assert check_reasoning_request(build_reasoning_request(question_of[QIDS[0]], _toy_memory), question_of[QIDS[0]], _toy_memory)

# 3. Prompts and templates: no question id, no reference answer (>= 4 characters), no answer-format name.
templates = [IMAGE_SYSTEM_PROMPT, EXTRACTION_SYSTEM_PROMPT, REASONING_SYSTEM_PROMPT, question_header(""), extraction_header(""),
             reasoning_user_text("", {"pages": [], "facts": []}), page_tag(0)]
references = {str(a).strip() for a in held["answer"]} - {NOT_ANSWERABLE}
hits = [(ref, i) for ref in references if len(ref) >= 4 for i, t in enumerate(templates) if ref.lower() in t.lower()]
assert not hits, hits
assert not any(qid_pattern.search(t) for t in templates)
assert not any(re.search(r"\b(Str|Int|Float|List|None)\b", t) for t in templates)

# 4. Builders take only the question text, pages, images or the memory.
assert list(inspect.signature(build_request).parameters) == ["question", "pages"]
assert list(inspect.signature(build_extraction_request).parameters) == ["question", "page", "png"]
assert list(inspect.signature(build_reasoning_request).parameters) == ["question", "memory"]

# 5. The notebook's request-building, planning and sending cells (read from disk) never touch annotations, labels or question ids.
REQUEST_CELL_IDS = ["e7-v0-reader", "e7-v0-requests", "e7-extract", "e7-memory", "e7-arithmetic", "e7-reason", "e7-requests", "e7-plan", "e7-executor", "e7-run"]
FORBIDDEN_IN_CODE = {  # regular expression -> what it would reveal (the reply key payload["answer"] and NOT_ANSWERABLE are allowed)
    r"mmlb-": "a question id",
    r"\baudit\b|\bAUDIT|STRATUM_OVERRIDES|TARGETED": "the E6 audit labels",
    r"E6_CORRECT|e6_table|\be6v\b": "E6 correctness",
    r"evidence_pages|evidence_sources|frozen_e5_held": "evidence annotations",
    r"answer_format": "answer formats",
    r"held_idx|held\[|questions\[": "the annotated question table",
    r"(?<!NOT_)\bANSWERABLE\b|\bUNANSWERABLE\b|['\"]answerable['\"]|\.answerable\b": "answerability",
    r"\.loc\[[^\]]*['\"]answer['\"]|\b(ref|reference|row)\[['\"]answer['\"]\]": "reference answers",
}
notebook_on_disk = json.loads((PROJECT_ROOT / NOTEBOOK_RELPATH).read_text())
cell_source = {c.get("id"): "".join(c["source"]) if isinstance(c["source"], list) else c["source"] for c in notebook_on_disk["cells"]}
assert set(REQUEST_CELL_IDS) <= set(cell_source), sorted(set(REQUEST_CELL_IDS) - set(cell_source))
violations = [(cell, what) for cell in REQUEST_CELL_IDS for pattern, what in FORBIDDEN_IN_CODE.items() if re.search(pattern, cell_source[cell])]
assert not violations, violations
# The scanner itself: it flags annotation access and lets the model-reply key and the abstention token through.
flags = lambda code: [what for pattern, what in FORBIDDEN_IN_CODE.items() if re.search(pattern, code)]
assert flags('ref = held_idx.loc[q, "answer"]') == ["the annotated question table", "reference answers"]
assert flags("for q in ANSWERABLE:") == ["answerability"] and flags('x = row["answer"]') == ["reference answers"]
assert flags('answer = payload["answer"]') == [] and flags("if answer == NOT_ANSWERABLE:") == [] and flags("auditor = 1") == []
print(f"Leakage controls pass: {len(V0_REQUESTS) + len(EXTRACT_REQUESTS)} planned image requests contain only their prompt, the question "
      f"and page tags; no question id or annotation field; no reference answer (>= 4 characters, {len(references)} checked) or format name "
      f"in any prompt; builders take only question, pages, images or memory; {len(REQUEST_CELL_IDS)} request cells free of "
      f"{len(FORBIDDEN_IN_CODE)} forbidden identifiers.")

Leakage controls pass: 1062 planned image requests contain only their prompt, the question and page tags; no question id or annotation field; no reference answer (>= 4 characters, 89 checked) or format name in any prompt; builders take only question, pages, images or memory; 10 request cells free of 8 forbidden identifiers.


## 6. Plan, calls, tokens, cost and runtime, before any call

**One question at a time, all three arms together.** Each question is one block: its V0 call, its 8 extraction calls (X8's pages, rank order), then its X4 and X8 reasoning calls.
- The order of the two reasoning calls is counterbalanced too: X4's comes first in a seeded 59 of the 118 blocks, chosen independently of V0's position.
- The 118 blocks run in one seeded order.
- In 59 blocks (a seeded half) V0 goes first; in the other 59, V0 follows the extractions.
- The arms of a question are therefore always minutes apart, on the same backend, and session drift cannot favour one arm.

**Each page extraction is paid for once.** The extraction cache is keyed by (question, page, extraction prompt version, model), together with the image hash and the run. X4's reasoning reads the same cached replies as X8's for its 3–4 pages.

**Smoke stage.** One block for the question with the largest V0 request (`mmlb-0068`), never scored: 11 calls.

**Full stage.** The smoke block again, then the 118 blocks.

**Cost and limits.** Upper bounds use the application's bound on prompt tokens and the full output allowance of every call. The cumulative ceiling covers retries, possibly billed failures and resumed runs. The paid gate refuses to start on or after the snapshot's shutdown (2027-01-14).

In [15]:
# The seeded plans, their calls, tokens, list-price cost and duration.
SEEDS = EXECUTION["seeds"]
SMOKE_QID = EXECUTION["technical_smoke_qid"]
assert SMOKE_QID == max(QIDS, key=lambda q: (V0_REQUESTS[q]["expected_prompt_tokens"], q)), "the smoke question is the largest V0 request"


def question_block(replicate_id, qid, v0_first, x4_first):
    v0 = [{"replicate_id": replicate_id, "qid": qid, "kind": "v0", "arm": "V0", "page": None}]
    extracts = [{"replicate_id": replicate_id, "qid": qid, "kind": "extract", "arm": "X", "page": page} for page in ARM_PAGES["X8"][qid]]
    reasons = [{"replicate_id": replicate_id, "qid": qid, "kind": "reason", "arm": arm, "page": None}
               for arm in (("X4", "X8") if x4_first else ("X8", "X4"))]
    return (v0 + extracts if v0_first else extracts + v0) + reasons


def execution_plan(mode):
    plan = [dict(item, block_index=0) for item in question_block("technical_smoke", SMOKE_QID, v0_first=True, x4_first=True)]
    if mode == "full":
        order = list(QIDS)
        np.random.default_rng(SEEDS["question_order"]).shuffle(order)
        v0_first = {order[i] for i in np.random.default_rng(SEEDS["v0_position"]).permutation(len(order))[: len(order) // 2]}
        x4_first = {order[i] for i in np.random.default_rng(SEEDS["reasoning_order"]).permutation(len(order))[: len(order) // 2]}
        for block_index, qid in enumerate(order, start=1):
            plan += [dict(item, block_index=block_index) for item in question_block("main_r1", qid, qid in v0_first, qid in x4_first)]
    for index, item in enumerate(plan, start=1):
        item["execution_index"] = index
    return plan


def planned_entry(item):
    return V0_REQUESTS[item["qid"]] if item["kind"] == "v0" else EXTRACT_REQUESTS[(item["qid"], item["page"])] if item["kind"] == "extract" else None


def item_cost_upper(item):
    entry = planned_entry(item)
    return REASON_COST_UPPER if entry is None else entry["cost_upper_usd"]


PLANS = {mode: execution_plan(mode) for mode in ("smoke", "full")}
composition = {mode: dict(Counter(f"{i['replicate_id']}:{i['kind']}" for i in plan)) for mode, plan in PLANS.items()}
assert len(PLANS["smoke"]) == EXECUTION["plan_composition"]["smoke_calls"] and len(PLANS["full"]) == EXECUTION["plan_composition"]["full_calls"]
assert PLANS["full"][:11] == PLANS["smoke"] and execution_plan("full") == PLANS["full"]
main = [i for i in PLANS["full"] if i["replicate_id"] == "main_r1"]
assert Counter(i["kind"] for i in main) == Counter({"extract": 944, "v0": 118, "reason": 236})
assert len({(i["qid"], i["page"]) for i in main if i["kind"] == "extract"}) == 944  # no extraction is planned twice
first_kind = Counter(next(i["kind"] for i in main if i["qid"] == q) for q in QIDS)
assert first_kind == Counter({"v0": 59, "extract": 59})
first_reason = Counter(next(i["arm"] for i in main if i["qid"] == q and i["kind"] == "reason") for q in QIDS)
assert first_reason == Counter({"X4": 59, "X8": 59})  # reasoning order counterbalanced with a fixed seed
assert pd.crosstab(pd.Series({q: next(i["kind"] for i in main if i["qid"] == q) for q in QIDS}),
                   pd.Series({q: next(i["arm"] for i in main if i["qid"] == q and i["kind"] == "reason") for q in QIDS})).values.min() > 0
for qid in QIDS:  # within a block, both reasoning calls come after all of the question's extractions
    block = [i for i in main if i["qid"] == qid]
    assert max(k for k, i in enumerate(block) if i["kind"] == "extract") < min(k for k, i in enumerate(block) if i["kind"] == "reason")

# Tokens and cost. Typical = exact image-call prompts + assumed reply lengths and memory sizes (planning only, from the spec).
TYPICAL = EXECUTION["typical_assumptions"]


def typical_prompt_tokens(item):
    entry = planned_entry(item)
    if entry is not None:
        return entry["expected_prompt_tokens"]
    pages = len(ARM_PAGES[item["arm"]][item["qid"]])
    return reason_fixed + pages * TYPICAL["memory_tokens_per_page"]


def typical_output_tokens(item):
    return {"v0": TYPICAL["v0_output_tokens"], "extract": TYPICAL["extraction_output_tokens"], "reason": TYPICAL["reasoning_output_tokens"]}[item["kind"]]


rows = []
for mode, plan in PLANS.items():
    for kind in ("v0", "extract", "reason"):
        items = [i for i in plan if i["kind"] == kind]
        rows.append({"mode": mode, "kind": kind, "calls": len(items), "prompt tokens (typical)": sum(typical_prompt_tokens(i) for i in items),
                     "upper bound USD": sum(item_cost_upper(i) for i in items),
                     "typical USD": sum(typical_prompt_tokens(i) * PRICE["input"] + typical_output_tokens(i) * PRICE["output"] for i in items)})
cost_plan = pd.DataFrame(rows)
totals = cost_plan.groupby("mode")[["calls", "prompt tokens (typical)", "upper bound USD", "typical USD"]].sum()
totals.loc["all stages"] = totals.sum()
UPPER, TYPICAL_USD = totals["upper bound USD"], totals["typical USD"]
hours = (totals.loc["all stages", "prompt tokens (typical)"] / EXECUTION["pacer_tokens_per_minute"]
         + totals.loc["all stages", "calls"] * EXECUTION["pause_between_calls_s"] / 60) / 60
assert {k: round(float(UPPER[m]), 4) for k, m in (("smoke", "smoke"), ("full", "full"), ("all_stages", "all stages"))} == BUDGET["estimate_upper_bound_usd"], UPPER.round(4)
assert round(float(TYPICAL_USD["all stages"]), 4) == BUDGET["estimate_typical_usd"]["all_stages"], TYPICAL_USD.round(4)
assert UPPER["all stages"] < BUDGET["approved_ceiling_usd"] == BUDGET["proposed_ceiling_usd"] == 9.0
assert round(hours, 1) == EXECUTION["expected_duration_hours"], round(hours, 2)
cost_plan.to_csv(E7_DIR / "cost_plan.csv", index=False)
display(cost_plan.set_index(["mode", "kind"]).round(4))
display(totals.round(4))
per_arm = pd.DataFrame({
    "V0": [len(QIDS), sum(V0_REQUESTS[q]["expected_prompt_tokens"] for q in QIDS)],
    "X4 (if run alone)": [sum(len(ARM_PAGES['X4'][q]) + 1 for q in QIDS),
                          sum(EXTRACT_REQUESTS[(q, p)]["expected_prompt_tokens"] for q in QIDS for p in ARM_PAGES["X4"][q])],
    "X8 (if run alone)": [sum(len(ARM_PAGES['X8'][q]) + 1 for q in QIDS),
                          sum(EXTRACT_REQUESTS[(q, p)]["expected_prompt_tokens"] for q in QIDS for p in ARM_PAGES["X8"][q])]},
    index=["calls", "image-call prompt tokens"]).T
per_arm["vs V0"] = (per_arm["image-call prompt tokens"] / per_arm.loc["V0", "image-call prompt tokens"]).round(3)
display(per_arm)
print(f"Order: V0 first in {first_kind['v0']} blocks, X4's reasoning before X8's in {first_reason['X4']} blocks (seeded, independent).")
print(f"Plans: smoke {len(PLANS['smoke'])} calls, full {len(PLANS['full'])} (smoke block again + 118 blocks of 11). Upper bound of every stage "
      f"${UPPER['all stages']:.4f}, typical ${TYPICAL_USD['all stages']:.4f}, cumulative hard ceiling ${BUDGET['approved_ceiling_usd']:.2f} (approved). "
      f"Expected duration at {EXECUTION['pacer_tokens_per_minute']:,} tokens per minute: {hours:.1f} h.")

calls  prompt tokens (typical)  upper bound USD  typical USD
mode  kind                                                                 
smoke v0           1                   110705           0.0170       0.0166
      extract      8                   298320           0.0505       0.0459
      reason       2                     4524           0.0049       0.0008
full  v0         119                 12280633           1.8832       1.8440
      extract    952                 25970064           4.5754       4.0383
      reason     238                   571956           0.5890       0.1029

,calls,prompt tokens (typical),upper bound USD,typical USD
mode,,,,
full,1309,38822653,7.0477,5.9853
smoke,11,413549,0.0724,0.0634
all stages,1320,39236202,7.1201,6.0487


,calls,image-call prompt tokens,vs V0
V0,118,12169928,1.000
X4 (if run alone),577,12351221,1.015
X8 (if run alone),1062,25671744,2.109


Order: V0 first in 59 blocks, X4's reasoning before X8's in 59 blocks (seeded, independent).
Plans: smoke 11 calls, full 1309 (smoke block again + 118 blocks of 11). Upper bound of every stage $7.1201, typical $6.0487, cumulative hard ceiling $9.00 (approved). Expected duration at 150,000 tokens per minute: 4.7 h.


## 7. The pre-registered analysis and decision rule

**Unit and statistics.**
- The unit is the question, paired across arms. Questions cluster in 14 documents, so the interval resamples whole documents: a document-cluster percentile bootstrap, 10,000 draws, seeded.
- The test is E6's exact document-level sign-flip test (one-sided, 2¹⁴ sign assignments).
- McNemar is reported, descriptively only, because it ignores clustering.
- These functions are notebook 05's. They are checked below by reproducing E6's frozen primary result exactly.

**Primary: X4 vs V0** on the 102 answerable questions. Engineering usefulness and statistical certainty are judged **separately**.

| | criterion |
|---|---|
| E1 | gain Δ = acc(X4) − acc(V0) ≥ +0.05 |
| E2 | broken ≤ fixed / 2 |
| G1 | correct abstentions on the 16 unanswerable: X4 ≥ V0 − 2 |
| G2 | citation validity on 118 replies: X4 − V0 ≥ −0.05 |
| S1 | one-sided 95 % bootstrap lower bound (5th percentile) > 0 (the two-sided 95 % interval is reported beside it) |
| S2 | one-sided document sign-flip p ≤ 0.05 |

**Verdicts, in this order.**
1. G1 or G2 fails → `regression`.
2. Δ ≤ −0.05 → `regression`.
3. Δ < +0.05 → `no_meaningful_gain`.
4. E2 fails (broken > fixed / 2 despite the gain) → `regression`.
5. S1 and S2 hold → `strong_support`.
6. Otherwise → `promising_but_uncertain`.

Statistical inconclusiveness alone is never read as failure of the architecture.

**Secondary.**
- **X8 vs X4** (additional page access) is judged by the same function, with its guardrails read against X4.
- **X8 vs V0** (total practical change, two factors) is reported with the same function, descriptively.

**Mechanism (diagnostic only, never a gate).** It reports:
- the E6 audit categories of the fixed and of the broken questions, and the share of fixes in the targeted categories;
- single-page vs multi-page change, and the change within the complete / partial / no-evidence strata;
- the arithmetic tool's contribution (computed answer vs the model's own answer).

**What a verdict is.** Development evidence on questions used to design E7. Even `strong_support` only licenses freezing the architecture for a final test on untouched documents.

In [16]:
# The frozen analysis functions (notebook 05's), the E7 decision rule (engineering usefulness and statistical certainty reported
# separately), the mechanism diagnostic (never a gate), their boundary tests, and E6's primary reproduced exactly.
BOOTSTRAP_SEED, BOOTSTRAP_DRAWS = SEEDS["bootstrap"], 10_000
MIN_DIFF, ALPHA, MAX_BROKEN_RATIO = RULE["minimum_meaningful_difference"], RULE["alpha_one_sided"], RULE["maximum_broken_to_fixed_ratio"]
_SIGN_VECTORS = {}


def document_totals(values, docs):
    values, docs = np.asarray(values, dtype=float), np.asarray(docs)
    order = sorted(set(docs))
    return np.array([values[docs == d].sum() for d in order]), np.array([(docs == d).sum() for d in order])


def cluster_bootstrap(values, docs, *, seed=BOOTSTRAP_SEED, draws=BOOTSTRAP_DRAWS):
    sums, counts = document_totals(values, docs)
    picks = np.random.default_rng(seed).integers(0, len(sums), size=(draws, len(sums)))
    return sums[picks].sum(axis=1) / counts[picks].sum(axis=1)


def sign_vectors(n):
    if n not in _SIGN_VECTORS:
        _SIGN_VECTORS[n] = np.array(list(product([1, -1], repeat=n)), dtype=float)
    return _SIGN_VECTORS[n]


def sign_flip_p(values, docs, *, weighting="question", alternative="greater"):
    sums, counts = document_totals(values, docs)
    statistic = sums if weighting == "question" else sums / counts
    null = sign_vectors(len(statistic)) @ statistic
    observed = statistic.sum()
    if alternative == "greater":
        return float(np.mean(null >= observed - 1e-9))
    return float(np.mean(np.abs(null) >= abs(observed) - 1e-9))


def mcnemar_exact_p(fixed, broken):
    n = fixed + broken
    return 1.0 if n == 0 else min(1.0, 2 * sum(comb(n, i) for i in range(min(fixed, broken) + 1)) / 2**n)


def paired_comparison(before, after, docs, *, seed=BOOTSTRAP_SEED, draws=BOOTSTRAP_DRAWS):
    before, after, docs = np.asarray(before, float), np.asarray(after, float), np.asarray(docs)
    diff = after - before
    boot = cluster_bootstrap(diff, docs, seed=seed, draws=draws)
    fixed, broken = int(((before == 0) & (after > 0)).sum()), int(((before > 0) & (after == 0)).sum())
    order = sorted(set(docs))
    return {"questions": len(diff), "documents": len(order), "before": float(before.mean()), "after": float(after.mean()), "delta": float(diff.mean()),
            "fixed": fixed, "broken": broken, "both_correct": int(((before > 0) & (after > 0)).sum()), "both_wrong": int(((before == 0) & (after == 0)).sum()),
            "bootstrap_p05": float(np.percentile(boot, 5)), "ci95": [float(x) for x in np.percentile(boot, [2.5, 97.5])],
            "sign_flip_p_one_sided": sign_flip_p(diff, docs), "sign_flip_p_two_sided": sign_flip_p(diff, docs, alternative="two-sided"),
            "mcnemar_p_ignores_clustering": mcnemar_exact_p(fixed, broken),
            "document_weighted_delta": float(np.mean([diff[docs == d].mean() for d in order])),
            "leave_one_document_out": {d: float(diff[docs != d].mean()) for d in order}}


def decide_e7(comparison, guardrails):
    # Engineering usefulness and statistical certainty, judged separately (spec decision_rule; order fixed before any reply).
    engineering = {"E1 gain >= +0.05": comparison["delta"] >= MIN_DIFF,
                   "E2 broken <= fixed / 2": comparison["broken"] <= MAX_BROKEN_RATIO * comparison["fixed"], **guardrails}
    statistics = {"S1 one-sided 95 % bootstrap lower bound (5th percentile) > 0": comparison["bootstrap_p05"] > 0,
                  "S2 one-sided document sign-flip p <= 0.05": comparison["sign_flip_p_one_sided"] <= ALPHA}
    if not all(guardrails.values()):
        verdict, reason = "regression", "a guardrail (abstention or citation validity) failed"
    elif comparison["delta"] <= -MIN_DIFF:
        verdict, reason = "regression", "accuracy fell by 0.05 or more"
    elif comparison["delta"] < MIN_DIFF:
        verdict, reason = "no_meaningful_gain", "gain below +0.05"
    elif not engineering["E2 broken <= fixed / 2"]:
        verdict, reason = "regression", "excessive broken questions (broken > fixed / 2) despite the gain"
    elif all(statistics.values()):
        verdict, reason = "strong_support", "engineering criteria and both statistical criteria hold"
    else:
        verdict, reason = "promising_but_uncertain", "engineering criteria hold; the statistical evidence is not conclusive"
    return {"verdict": verdict, "reason": reason, "engineering": engineering, "statistics": statistics,
            "ci95_two_sided": comparison["ci95"], "reading": RULE["readings"][verdict], "next_step": RULE["next_steps"][verdict]}


def mechanism_diagnostic(fixed_qids, broken_qids):
    # Diagnostic only (never a gate): which frozen E6 categories the fixed and broken questions carry.
    label = lambda q: audit.loc[q, "category"] if q in audit.index else "E6 V was correct (no label)"
    targeted = [q for q in fixed_qids if q in audit.index and audit.loc[q, "targeted"]]
    return {"fixed": len(fixed_qids), "fixed_in_targeted_categories": len(targeted),
            "share_of_fixes_targeted": len(targeted) / len(fixed_qids) if fixed_qids else None,
            "fixed_by_category": dict(Counter(label(q) for q in fixed_qids)), "broken": len(broken_qids),
            "broken_by_category": dict(Counter(label(q) for q in broken_qids))}


# A. Every boundary of the rule, on synthetic summaries (interface tests, not E7 outcomes).
def synthetic(delta, p05=0.01, ci=(0.01, 0.2), p=0.01, fixed=10, broken=2):
    return {"delta": delta, "bootstrap_p05": p05, "ci95": list(ci), "sign_flip_p_one_sided": p, "fixed": fixed, "broken": broken}


ok, bad = {"G1 abstention": True, "G2 citation validity": True}, {"G1 abstention": False, "G2 citation validity": True}
cases = {"all criteria": (synthetic(0.08), ok, "strong_support"),
         "lower bound at 0": (synthetic(0.08, p05=0.0), ok, "promising_but_uncertain"),
         "p just above 0.05": (synthetic(0.08, p=0.0501), ok, "promising_but_uncertain"),
         "gain exactly 0.05, no statistics": (synthetic(0.05, p05=-0.01, p=0.2, fixed=6, broken=1), ok, "promising_but_uncertain"),
         "gain just below 0.05": (synthetic(4 / 102, fixed=6, broken=2), ok, "no_meaningful_gain"),
         "no change, many flips": (synthetic(0.0, p05=-0.05, p=0.6, fixed=7, broken=7), ok, "no_meaningful_gain"),
         "small loss": (synthetic(-0.03, p05=-0.08, p=0.8, fixed=3, broken=6), ok, "no_meaningful_gain"),
         "loss of 0.05": (synthetic(-0.05, p05=-0.1, p=0.9, fixed=2, broken=7), ok, "regression"),
         "guardrail fails despite gain": (synthetic(0.10), bad, "regression"),
         "guardrail fails without gain": (synthetic(0.0, fixed=3, broken=3), bad, "regression"),
         "gain but broken > fixed / 2": (synthetic(0.06, fixed=14, broken=8), ok, "regression"),
         "gain and broken = fixed / 2": (synthetic(0.06, fixed=12, broken=6), ok, "strong_support")}
assert {n: decide_e7(s, g)["verdict"] for n, (s, g, _) in cases.items()} == {n: v for n, (*_, v) in cases.items()}
assert set(RULE["readings"]) == set(RULE["next_steps"]) == {v for *_, v in cases.values()}
assert "not evidence that the architecture failed" in RULE["readings"]["promising_but_uncertain"]
d = mechanism_diagnostic(["mmlb-0065", "mmlb-0042", "mmlb-0044"], ["mmlb-0044"])  # table cell (targeted), perception, E6-correct
assert (d["fixed_in_targeted_categories"], round(d["share_of_fixes_targeted"], 3), d["broken_by_category"]) == (1, 0.333, {"E6 V was correct (no label)": 1})
assert mechanism_diagnostic([], [])["share_of_fixes_targeted"] is None
assert mcnemar_exact_p(8, 2) == 0.109375 and sign_flip_p([1, 1, 1], ["a", "b", "c"]) == 1 / 8

# B. E6's frozen primary (V - A), reproduced exactly from the frozen per-question table with E6's seed.
e6_frame = e6_table.loc[ANSWERABLE]
e6_repro = paired_comparison(e6_frame["correct_A"], e6_frame["correct_V"], e6_frame["doc_id"], seed=E6_SPEC["execution"]["seeds"]["bootstrap"])
frozen_primary = e6_summary["primary"]
for key in ("delta", "fixed", "broken", "bootstrap_p05", "ci95", "sign_flip_p_one_sided", "sign_flip_p_two_sided", "mcnemar_p_ignores_clustering",
            "document_weighted_delta"):
    assert e6_repro[key] == frozen_primary[key], key
print(f"Decision rule: {len(cases)} synthetic boundary cases give the specified verdicts; the mechanism diagnostic behaves as specified. "
      f"E6's frozen primary is reproduced exactly (delta {e6_repro['delta']:.4f}, 13 fixed / 0 broken, CI "
      f"[{e6_repro['ci95'][0]:.4f}, {e6_repro['ci95'][1]:.4f}], one-sided p {e6_repro['sign_flip_p_one_sided']}).")

Decision rule: 12 synthetic boundary cases give the specified verdicts; the mechanism diagnostic behaves as specified. E6's frozen primary is reproduced exactly (delta 0.1275, 13 fixed / 0 broken, CI [0.0745, 0.1932], one-sided p 0.0009765625).


In [17]:
# Planning simulation (assumptions, not results): how often the engineering criteria (E1, E2) and strong support (E1, E2, S1, S2) hold
# for X4 vs V0 on these 102 questions and 14 documents, guardrails assumed to pass,
# when each arm's observed correctness also flips at random between sessions (E3: 5 of 37 image replies changed, about 7 % per arm).
PLAN_SIM = SPEC["planning_simulation"]
sim_rng = np.random.default_rng(SEEDS["planning_simulation"])
sim_docs = held_idx.loc[ANSWERABLE, "doc_id"].to_numpy()
sim_order = sorted(set(sim_docs))
membership = np.array([[d == o for o in sim_order] for d in sim_docs], dtype=float)
sim_counts = membership.sum(axis=0)
sim_signs = sign_vectors(len(sim_order))
v_correct = np.array([E6_CORRECT["V"][q] for q in ANSWERABLE])  # E6 V's correctness stands in for V0's truth


def simulated_rates(true_gain, flip, sims, boot_draws):
    engineering, strong = 0, 0
    wrong = np.flatnonzero(~v_correct)
    for _ in range(sims):
        x4_truth = v_correct.copy()
        x4_truth[sim_rng.choice(wrong, int(round(true_gain * len(ANSWERABLE))), replace=False)] = True
        v0_seen = v_correct ^ (sim_rng.random(len(ANSWERABLE)) < flip)
        x4_seen = x4_truth ^ (sim_rng.random(len(ANSWERABLE)) < flip)
        diff = x4_seen.astype(float) - v0_seen.astype(float)
        sums = membership.T @ diff
        picks = sim_rng.integers(0, len(sim_order), size=(boot_draws, len(sim_order)))
        boot = sums[picks].sum(axis=1) / sim_counts[picks].sum(axis=1)
        p_value = np.mean(sim_signs @ sums >= sums.sum() - 1e-9)
        fixed, broken = int((~v0_seen & x4_seen).sum()), int((v0_seen & ~x4_seen).sum())
        passes = diff.mean() >= MIN_DIFF and broken <= MAX_BROKEN_RATIO * fixed
        engineering += passes
        strong += passes and np.percentile(boot, 5) > 0 and p_value <= ALPHA
    return engineering / sims, strong / sims


rates = {(gain, flip): simulated_rates(gain, flip, PLAN_SIM["simulations"], PLAN_SIM["bootstrap_draws"])
         for flip in PLAN_SIM["flip_rates"] for gain in PLAN_SIM["true_gains"]}
power = pd.DataFrame({f"{kind} (flip {flip:.3f} per arm)": {f"true gain +{gain:.2f}": rates[(gain, flip)][k] for gain in PLAN_SIM["true_gains"]}
                      for flip in PLAN_SIM["flip_rates"] for k, kind in enumerate(("engineering criteria", "strong_support"))})
assert power.round(3).to_dict() == PLAN_SIM["rates"], power.round(3).to_dict()
display(power.round(2))
print("Planning only (never a result): a true gain of +0.08 meets the engineering criteria in 61-78 % of simulated runs and +0.12 in "
      "82-97 %; strong_support needs about +0.12 or more. With no true effect the engineering criteria still pass in 2-5 % of runs "
      "(session noise), so 'promising_but_uncertain' is a candidate for the final test, not a confirmation.")

,engineering criteria (flip 0.035 per arm),strong_support (flip 0.035 per arm),engineering criteria (flip 0.070 per arm),strong_support (flip 0.070 per arm)
true gain +0.00,0.02,0.01,0.05,0.02
true gain +0.05,0.38,0.22,0.34,0.14
true gain +0.08,0.78,0.48,0.61,0.31
true gain +0.12,0.97,0.82,0.82,0.58
true gain +0.16,1.00,0.96,0.95,0.83


Planning only (never a result): a true gain of +0.08 meets the engineering criteria in 61-78 % of simulated runs and +0.12 in 82-97 %; strong_support needs about +0.12 or more. With no true effect the engineering criteria still pass in 2-5 % of runs (session noise), so 'promising_but_uncertain' is a candidate for the final test, not a confirmation.


## 8. Execution: budget guard, append-only log, extraction cache, resume, smoke gate

This is notebook 05's executor, extended to E7's three kinds of call.

**What carries over from E6.**
- **Before every call:**
  - the pre-call check: model, output bound, context, and a $0.02 per-request ceiling;
  - an integrity check: the bytes about to be sent equal the plan;
  - the cumulative budget guard: replies received, plus possibly billed failures, plus the upper bound of every remaining call, must fit the approved ceiling.
- **Every reply** is appended (fsynced) to the run's cache before its log record.
- **A resumed run** re-sends nothing that already has a reply.
- **Stops:**
  - a 429 with a quota or billing code stops the run, and a waitable 429 is retried with backoff;
  - an integrity failure stops the run and cannot be resumed.

**What is new in E7.**
- **Extraction cache.** An extraction reply is cached under its request fingerprint. The fingerprint encodes the question, the page tag, the page image, the extraction prompt and the model. X4's and X8's reasoning calls read the same cached reply for a shared page. The plan holds each (question, page) exactly once, so no extraction is paid for twice.
- **Reasoning requests are built during the run.** They come from the cached extraction replies of the arm's pages, with the same deterministic functions tested in section 4, and pass the leakage check before they are sent. The Evidence Memory is logged with the reasoning call.
- **Smoke gate (11 calls, technical).** It checks:
  - call composition and request fingerprints, with V0 equal to the frozen E6 V request;
  - parsing, the extraction schema, and Evidence Memory schema and provenance;
  - cache reuse: 8 extraction calls serve 11 page readings;
  - the reasoning requests;
  - that the arithmetic request/result path follows the frozen rule, and that operand pages are cited;
  - token diagnostics, and billed cost ≤ the block's planned bound.

  Abstention, calculation status and citation behaviour are recorded as **observations**: they describe the model and never gate. Resume and failure handling are covered by the interface tests in section 9.
- **Unreadable extractions.** A reply that cannot be parsed is recorded as a contract failure: in the smoke stage the gate fails; in the full stage the page enters the memory as unreadable. The run's population is never truncated by outcome.
- **Logs.** No annotation (reference answer, answer format, evidence pages, answerability) is written to a run's logs. Evaluation joins them afterwards.

In [18]:
# E7 executor: engine, list-price budget guard (cumulative, failed attempts included), run-scoped cache, durable log, resume, smoke gate.
MAX_ATTEMPTS = EXECUTION["max_attempts_per_request"]
RATE_LIMIT = EXECUTION["rate_limit"]
TOLERANCE = EXECUTION["token_tolerance"]
RESPONDED = {"ok", "parse_error"}  # a reply exists; parse_error is the model's own failure (scored 0, or an unreadable page)
NOT_AN_ATTEMPT = {"budget_stop", "not_attempted", "rate_limited", "blocked"}
POSSIBLY_BILLED_ERRORS = {"network", "server", "unexpected"}
RESPONSE_FIELDS = ("raw_response", "finish_reason", "served_model", "system_fingerprint", "prompt_tokens", "completion_tokens", "cached_prompt_tokens")
CALL_FIELDS = RESPONSE_FIELDS + ("api_latency_s", "sent_fingerprint")
QUOTA_CODES = {"insufficient_quota", "credit_balance_exhausted", "billing_hard_limit_reached", "billing_not_active"}
SECRET_PATTERN = re.compile(r"sk-[A-Za-z0-9_\-\*]{4,}")
SHUTDOWN_DATE = datetime(2027, 1, 14).date()


def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="milliseconds")


def append_durable(path, payload):
    with open(path, "a") as handle:
        handle.write(json.dumps(payload, default=str) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def write_new_text(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "x") as handle:  # never overwrites
        handle.write(text)


def classify_api_error(error):
    if isinstance(error, openai.RateLimitError):
        if getattr(error, "code", None) in QUOTA_CODES or getattr(error, "type", None) in QUOTA_CODES:
            return "quota_or_billing", False
        return "rate_limit", True
    if isinstance(error, openai.AuthenticationError):
        return "authentication", False
    if isinstance(error, openai.PermissionDeniedError):
        return "permission", False
    if isinstance(error, openai.NotFoundError):
        return "model_access", False
    if isinstance(error, (openai.BadRequestError, openai.UnprocessableEntityError)):
        return "invalid_request", False
    if isinstance(error, openai.APIConnectionError):
        return "network", True
    if isinstance(error, openai.InternalServerError) or (isinstance(error, openai.APIStatusError) and error.status_code in (408, 409)):
        return "server", True
    return "unexpected", False


def describe_api_error(error):
    category, retryable = classify_api_error(error)
    return {"category": category, "retryable": retryable, "exception": type(error).__name__, "status_code": getattr(error, "status_code", None),
            "code": getattr(error, "code", None), "type": getattr(error, "type", None),
            "message": SECRET_PATTERN.sub("sk-<redacted>", str(getattr(error, "message", error)))[:300], "request_id": getattr(error, "request_id", None)}


def response_fields(response):
    usage = response.usage
    details = getattr(usage, "prompt_tokens_details", None) if usage else None
    return {"raw_response": response.choices[0].message.content or "", "finish_reason": response.choices[0].finish_reason, "served_model": response.model,
            "system_fingerprint": getattr(response, "system_fingerprint", None), "prompt_tokens": usage.prompt_tokens if usage else None,
            "completion_tokens": usage.completion_tokens if usage else None, "cached_prompt_tokens": getattr(details, "cached_tokens", None)}


class TokenPacer:
    # Keeps the expected prompt tokens sent in any 60-second window under tokens_per_minute.
    def __init__(self, tokens_per_minute, clock=time.monotonic, sleep=time.sleep):
        self.tokens_per_minute, self.clock, self.sleep, self.window = tokens_per_minute, clock, sleep, deque()

    def wait(self, tokens):
        while True:
            now = self.clock()
            while self.window and now - self.window[0][0] >= 60:
                self.window.popleft()
            if not self.window or sum(t for _, t in self.window) + tokens <= self.tokens_per_minute:
                break
            self.sleep(60 - (now - self.window[0][0]))
        self.window.append((self.clock(), tokens))


def make_engine(create):
    # One call: the pre-call check (raises before sending), then the SDK call; records latency and the fingerprint of what the SDK received.
    def engine(request):
        check_request(request)
        seen = {}

        def recording_create(**kwargs):
            seen["sent_fingerprint"] = request_fingerprint(kwargs)
            started = time.perf_counter()
            seen["response"] = create(**kwargs)
            seen["api_latency_s"] = round(time.perf_counter() - started, 3)
            return seen["response"]

        fields = response_fields(recording_create(**request))
        return {**fields, "api_latency_s": seen["api_latency_s"], "sent_fingerprint": seen["sent_fingerprint"]}

    return engine


def cache_key(run_id, replicate_id, fingerprint):
    return sha256_bytes(f"{run_id}:{replicate_id}:{fingerprint}".encode())


def load_cache(path):
    return {entry["cache_key"]: entry for entry in read_jsonl(path)} if path.is_file() else {}


def reply_cost(fields):
    return (fields.get("prompt_tokens") or 0) * PRICE["input"] + (fields.get("completion_tokens") or 0) * PRICE["output"]


def failed_attempt_charge(log_path):
    if not log_path.is_file():
        return 0.0
    return sum(r["cost_upper_usd"] for r in read_jsonl(log_path)
               if r["status"] == "api_error" and (r.get("error") or {}).get("category") in POSSIBLY_BILLED_ERRORS)


def run_spent_usd(cache, log_path):
    return sum(reply_cost(entry) for entry in cache.values()) + failed_attempt_charge(log_path)


def prior_spend_usd(runs_dir, spec_sha256, exclude_run_id):
    total = 0.0
    for settings_path in sorted(runs_dir.glob("*/settings.json")) if runs_dir.is_dir() else []:
        settings = json.loads(settings_path.read_text())
        if settings.get("spec_sha256") == spec_sha256 and settings.get("run_id") != exclude_run_id:
            total += run_spent_usd(load_cache(settings_path.parent / "response_cache.jsonl"), settings_path.parent / "execution_log.jsonl")
    return total


def latest_records(log_records):
    latest = {}
    for record in log_records:
        latest[record["execution_index"]] = record
    return [latest[index] for index in sorted(latest)]


def remaining_items(plan, log_records):
    assert not any(r.get("integrity_valid") is False or r["status"] == "integrity_error" for r in log_records), \
        "an integrity failure is logged in this run; it cannot be resumed"
    responded = {r["execution_index"] for r in log_records if r["status"] in RESPONDED}
    attempts = Counter(r["execution_index"] for r in log_records if r["status"] not in NOT_AN_ATTEMPT)
    return [item for item in plan if item["execution_index"] not in responded], attempts


def build_reason_entry(item, run_id, cache):
    # The reasoning request of an arm, from the cached extraction replies of its pages (same run and replicate only).
    qid, arm = item["qid"], item["arm"]
    pages = ARM_PAGES[arm][qid]
    extractions, keys = {}, []
    for page in pages:
        key = cache_key(run_id, item["replicate_id"], EXTRACT_REQUESTS[(qid, page)]["fingerprint"])
        if key not in cache:
            return None, f"the extraction of page {page} has no reply in this run"
        extractions[page] = parse_extraction(cache[key]["raw_response"], cache[key]["finish_reason"], page)
        keys.append(key)
    memory = build_memory(pages, extractions)
    validate_memory(memory, pages)
    request = build_reasoning_request(question_of[qid], memory)
    check_reasoning_request(request, question_of[qid], memory)
    bound, _ = check_request(request)
    return {"qid": qid, "kind": "reason", "doc_id": doc_of[qid], "page": None, "pages": pages, "request": request,
            "fingerprint": request_fingerprint(request), "image_sha256": [], "expected_prompt_tokens": expected_prompt_tokens(request),
            "prompt_token_bound": bound, "max_output_tokens": REASONING_MAX_OUTPUT, "cost_upper_usd": call_cost_upper(bound, REASONING_MAX_OUTPUT),
            "memory": memory, "memory_sha256": sha256_bytes(memory_text(memory).encode()), "extraction_cache_keys": keys,
            "page_status": {str(p): extractions[p]["status"] for p in pages}}, None


def verify_entry(item, entry, run_id, cache):
    # Re-derived from the bytes about to be sent: image requests equal the plan; a reasoning request is rebuilt identically from the cache.
    if item["kind"] == "reason":
        again, _ = build_reason_entry(item, run_id, cache)
        return again is not None and again["fingerprint"] == entry["fingerprint"] == request_fingerprint(entry["request"])
    return [i["sha256"] for i in request_parts(entry["request"])[1]] == entry["image_sha256"] and request_fingerprint(entry["request"]) == entry["fingerprint"]


def interpret(item, entry, raw_text, finish_reason):
    # The kind-specific parse of a reply.
    if item["kind"] == "v0":
        answer, cited, problem = parse_reply(raw_text, finish_reason)
        return {"pred": answer if problem is None else "", "cited_pages": cited if problem is None else [], "parse_error": problem}
    if item["kind"] == "extract":
        extraction = parse_extraction(raw_text, finish_reason, item["page"])
        return {"extraction": extraction, "parse_error": extraction["error"]}
    answer, cited, calculation, problem = parse_reasoning(raw_text, finish_reason)
    if problem is not None:
        return {"pred": "", "cited_pages": [], "parse_error": problem, "arithmetic": None}
    final, final_cited, arithmetic = finalize_answer(answer, cited, calculation, entry["memory"])
    return {"pred": final, "cited_pages": final_cited, "parse_error": None, "arithmetic": arithmetic}


def response_checks(record):
    integrity = {"request_content_verified": bool(record.get("request_content_verified")),
                 "sent_request_equals_plan": record.get("sent_fingerprint") == record.get("fingerprint"),
                 "served_model_is_the_snapshot": record.get("served_model") == SNAPSHOT}
    contract = {"reply_parsed": record.get("parse_error") is None, "finished_with_stop": record.get("finish_reason") == "stop"}
    diagnostics = {"prompt_tokens_within_tolerance": record.get("prompt_tokens") is not None
                   and abs(record["prompt_tokens"] - record["expected_prompt_tokens"]) <= TOLERANCE,
                   "cached_token_field_recorded": record.get("cached_prompt_tokens") is not None}
    return integrity, contract, diagnostics


def execute_plan(plan, engine, run_id, cache, cache_path, log_path, *, stop_on_gate_failure, budget_usd, attempts=None, pause_s=0.0, pacer=None, sleep=time.sleep):
    attempts = Counter() if attempts is None else attempts
    for position, item in enumerate(plan):
        attempts[item["execution_index"]] += 1
        if attempts[item["execution_index"]] > MAX_ATTEMPTS:
            return f"attempt limit reached at execution index {item['execution_index']}"
        if item["kind"] == "reason":
            entry, blocked = build_reason_entry(item, run_id, cache)
        else:
            entry, blocked = planned_entry(item), None
        record = {**item, "attempt": attempts[item["execution_index"]], "arm_label": ARMS.get(item["arm"], "extraction (shared by X4 and X8)"),
                  "doc_id": doc_of[item["qid"]], "requested_model": SNAPSHOT, "status": "not_attempted", "source": None, "raw_response": None,
                  "request_started_at": None, "request_completed_at": None}
        if blocked is not None:
            record["status"] = "blocked"
            append_durable(log_path, {**record, "error": {"category": "blocked", "message": blocked}})
            return f"blocked at {item['qid']} {item['arm']}: {blocked}"
        key = cache_key(run_id, item["replicate_id"], entry["fingerprint"])
        record.update({"supplied_pages": entry["pages"], "image_sha256": entry["image_sha256"], "fingerprint": entry["fingerprint"],
                       "request_content_verified": verify_entry(item, entry, run_id, cache), "expected_prompt_tokens": entry["expected_prompt_tokens"],
                       "prompt_token_bound": entry["prompt_token_bound"], "cost_upper_usd": entry["cost_upper_usd"],
                       "cache_provenance": {"namespace": f"{run_id}:{item['replicate_id']}", "cache_key": key, "hit": False, "saved_at": None}})
        if item["kind"] == "reason":
            record.update({k: entry[k] for k in ("memory", "memory_sha256", "extraction_cache_keys", "page_status")})
        stop_reason, cached = None, cache.get(key)
        projected = run_spent_usd(cache, log_path) + max(entry["cost_upper_usd"], item_cost_upper(item)) + sum(item_cost_upper(later) for later in plan[position + 1:])
        if not record["request_content_verified"]:
            record["status"] = "integrity_error"
            stop_reason = f"request content differs from the plan at {item['qid']} {item['kind']} {item['arm']}"
        elif cached is not None:  # only this run and replicate can supply it: an interrupted earlier attempt, or a shared page
            record.update({field: cached.get(field) for field in CALL_FIELDS}, source="cache")
            record["cache_provenance"].update(hit=True, saved_at=cached["saved_at"])
        elif projected > budget_usd:
            record["status"] = "budget_stop"
            stop_reason = f"budget: projected ${projected:.4f} exceeds the remaining ${budget_usd:.4f} before {item['qid']} {item['kind']}"
        else:
            waits = list(RATE_LIMIT["backoff_s"])
            if pacer is not None:
                pacer.wait(entry["expected_prompt_tokens"])
            result = None
            while True:
                sleep(pause_s)
                record["request_started_at"] = utc_now()
                try:
                    result = engine(entry["request"])
                except openai.RateLimitError as error:
                    category, _ = classify_api_error(error)
                    record.update(status="rate_limited" if category == "rate_limit" else "api_error", request_completed_at=utc_now(), error=describe_api_error(error))
                    if category == "rate_limit" and waits:
                        append_durable(log_path, {**record, "rate_limit_wait_s": waits[0]})
                        sleep(waits.pop(0))
                        record = {k: v for k, v in record.items() if k != "error"} | {"status": "not_attempted", "request_completed_at": None}
                        continue
                    stop_reason = (f"rate_limit persisted after {len(RATE_LIMIT['backoff_s'])} waits at {item['qid']}" if category == "rate_limit"
                                   else f"{category} at {item['qid']}")
                except openai.OpenAIError as error:
                    record.update(status="api_error", request_completed_at=utc_now(), error=describe_api_error(error))
                    stop_reason = f"{record['error']['category']} at {item['qid']}"
                except ValueError as error:  # the pre-call check refused: nothing was sent
                    record.update(status="integrity_error", request_completed_at=utc_now(), error={"category": "pre_call_refusal", "message": str(error)})
                    stop_reason = f"pre-call check refused the request at {item['qid']}: {error}"
                break
            if result is not None:
                record["request_completed_at"] = utc_now()
                fields = {field: result.get(field) for field in CALL_FIELDS}
                payload = {"cache_key": key, "cache_namespace": record["cache_provenance"]["namespace"], "replicate_id": item["replicate_id"],
                           "execution_index": item["execution_index"], "kind": item["kind"], "fingerprint": entry["fingerprint"], "saved_at": utc_now(), **fields}
                append_durable(cache_path, payload)  # the reply is durable before its log record
                cache[key] = payload
                record.update(fields, source="live")
                record["cache_provenance"]["saved_at"] = payload["saved_at"]
        if record["raw_response"] is not None:
            record.update(interpret(item, entry, record["raw_response"], record["finish_reason"]))
            record["status"] = "ok" if record["parse_error"] is None else "parse_error"
            integrity, contract, diagnostics = response_checks(record)
            record.update(prompt_token_difference=None if record.get("prompt_tokens") is None else record["prompt_tokens"] - record["expected_prompt_tokens"],
                          integrity_checks=integrity, contract_checks=contract, diagnostic_checks=diagnostics,
                          integrity_valid=all(integrity.values()), contract_valid=all(contract.values()), diagnostic_valid=all(diagnostics.values()))
            if not record["integrity_valid"]:
                stop_reason = f"integrity check failed at {item['qid']}: " + ", ".join(k for k, v in integrity.items() if not v)
            elif stop_on_gate_failure and not (record["contract_valid"] and record["diagnostic_valid"]):
                stop_reason = f"technical gate failed at {item['qid']} {item['kind']}: " + ", ".join(k for k, v in {**contract, **diagnostics}.items() if not v)
        append_durable(log_path, record)
        if stop_reason is not None:
            return stop_reason
    return None


def arithmetic_consistent(record):
    # The final answer and citations follow the frozen arithmetic rule exactly, whatever the model asked for.
    arithmetic = record.get("arithmetic")
    if record["status"] != "ok":
        return arithmetic is None
    status = arithmetic["calc_status"]
    if status == "none":
        return arithmetic["calculation"] is None and record["pred"] == arithmetic["model_answer"]
    if status.startswith("ignored"):
        return record["pred"] == NOT_ANSWERABLE
    if status == "error":
        return record["pred"] == arithmetic["model_answer"] and bool(arithmetic["calc_error"])
    again = evaluate_calculation(arithmetic["calculation"], record["memory"])
    return again["status"] == "ok" and again["result"] == record["pred"] and set(again["operand_pages"]) <= set(record["cited_pages"])


def smoke_gate(records):
    # Technical checks (they gate the full stage) and observations of model behaviour (recorded; never a gate).
    kinds = Counter(r["kind"] for r in records)
    v0 = [r for r in records if r["kind"] == "v0"]
    extracts = [r for r in records if r["kind"] == "extract"]
    reasons = {r["arm"]: r for r in records if r["kind"] == "reason"}
    complete = (kinds == Counter({"v0": 1, "extract": X8_K, "reason": 2}) and {r["qid"] for r in records} == {SMOKE_QID}
                and set(reasons) == {"X4", "X8"} and all(r["status"] in RESPONDED for r in records))
    checks = {
        "one block for the smoke question (1 V0, 8 extractions, X4 and X8 reasoning), every call replied": complete,
        "request fingerprints: sent = planned on every call, served snapshot": complete and all(r.get("integrity_valid") for r in records),
        "V0's request is the frozen E6 V request": complete and v0[0]["fingerprint"] == V0_REQUESTS[SMOKE_QID]["fingerprint"],
        "response parsing: every reply parsed, finish 'stop'": complete and all(r.get("contract_valid") for r in records),
        "extraction schema: the 8 pages of X8 parsed; provenance from the request": complete and sorted(r["page"] for r in extracts) == sorted(ARM_PAGES["X8"][SMOKE_QID])
        and all(r["extraction"]["status"] == "ok" for r in extracts),
        "Evidence Memory: schema valid, the arm's pages in rank order, every fact id carries its page": complete and all(
            validate_memory(r["memory"], ARM_PAGES[arm][SMOKE_QID]) and all(f["id"].startswith(f"p{f['page']}.f") for f in r["memory"]["facts"])
            for arm, r in reasons.items()),
        "cache reuse: 8 extraction calls serve 11 page readings; X4 reads X8's cached extractions": complete and len(extracts) == X8_K
        and set(reasons["X4"]["extraction_cache_keys"]) <= set(reasons["X8"]["extraction_cache_keys"]) == {r["cache_provenance"]["cache_key"] for r in extracts},
        "reasoning requests: built from the memory, leakage-checked, sent as planned": complete and all(r.get("integrity_valid") for r in reasons.values()),
        "arithmetic request / result path follows the frozen rule": complete and all(arithmetic_consistent(r) for r in reasons.values()),
        "citation provenance: calculation operand pages are among the cited pages": complete and all(
            set((r["arithmetic"] or {}).get("operand_pages", [])) <= set(r["cited_pages"]) for r in reasons.values()),
        "diagnostics: prompt tokens within tolerance, cached-token field recorded": complete and all(r.get("diagnostic_valid") for r in records),
        "cost accounting: billed list-price cost of the block <= its planned upper bound": complete
        and sum(reply_cost(r) for r in records) <= sum(item_cost_upper(i) for i in PLANS["smoke"]) + 1e-9,
    }
    observations = {} if not complete else {
        "abstained": {arm: r["pred"] == NOT_ANSWERABLE for arm, r in [("V0", v0[0]), *sorted(reasons.items())]},
        "calculation status": {arm: r["arithmetic"]["calc_status"] for arm, r in sorted(reasons.items())},
        "cited pages within the arm's pages": {arm: set(r["cited_pages"]) <= set(ARM_PAGES[arm][SMOKE_QID]) for arm, r in [("V0", v0[0]), *sorted(reasons.items())]},
        "facts per extracted page": {r["page"]: len(r["extraction"]["facts"]) for r in extracts},
        "pages flagged relevant (diagnostic)": sum(r["extraction"]["relevant"] is True for r in extracts),
        "reported-page mismatches": sum(bool(r["extraction"]["page_mismatch"]) for r in extracts),
        "billed list-price USD": round(sum(reply_cost(r) for r in records), 6),
        "planned upper bound USD": round(sum(item_cost_upper(i) for i in PLANS["smoke"]), 6)}
    return checks, all(checks.values()), observations


def run_stages(mode, engine, run_id, run_dir, *, budget_usd, prior_spent_usd=0.0, pacer=None, sleep=time.sleep, plan=None):
    plan = execution_plan(mode) if plan is None else plan
    run_dir.mkdir(parents=True, exist_ok=True)
    cache_path, log_path = run_dir / "response_cache.jsonl", run_dir / "execution_log.jsonl"
    log_now = lambda: read_jsonl(log_path) if log_path.is_file() else []
    segments = {"smoke": [i for i in plan if i["replicate_id"] == "technical_smoke"], "main": [i for i in plan if i["replicate_id"] == "main_r1"]}

    def run_segment(name, stop_on_gate_failure):
        todo, attempts = remaining_items(segments[name], log_now())
        reserve = sum(item_cost_upper(i) for i in remaining_items(segments["main"], log_now())[0]) if name == "smoke" else 0.0
        return execute_plan(todo, engine, run_id, load_cache(cache_path), cache_path, log_path, stop_on_gate_failure=stop_on_gate_failure,
                            budget_usd=budget_usd - prior_spent_usd - reserve, attempts=attempts, pause_s=EXECUTION["pause_between_calls_s"] if pacer else 0.0,
                            pacer=pacer, sleep=sleep) if todo else None

    stop = run_segment("smoke", True)
    smoke_checks, smoke_ok, smoke_observations = smoke_gate([r for r in latest_records(log_now()) if r["replicate_id"] == "technical_smoke"])
    if not (run_dir / "smoke_gate.json").exists():
        write_new_text(run_dir / "smoke_gate.json", json.dumps({"passed": smoke_ok, "checks": smoke_checks, "observations": smoke_observations,
                                                                "stop_reason": stop}, indent=2, default=str))
    if smoke_ok and mode == "full":
        stop = run_segment("main", False)
    elif not smoke_ok:
        stop = stop or "smoke gate failed"
    final = latest_records(log_now())
    replied = [r for r in final if r["status"] in RESPONDED]
    received = sum(reply_cost(e) for e in load_cache(cache_path).values())
    failed_charge = failed_attempt_charge(log_path)
    summary = {"run_id": run_id, "mode": mode, "complete": len(final) == len(plan) == len(replied) and stop is None, "stop_reason": stop,
               "smoke_gate_passed": smoke_ok, "smoke_checks": smoke_checks, "smoke_observations": smoke_observations, "planned_calls": len(plan),
               "api_calls_sent": sum(r["source"] == "live" or r["status"] in ("api_error", "rate_limited") for r in log_now()),
               "rate_limited_responses": sum(r["status"] == "rate_limited" for r in log_now()), "replied": len(replied),
               "status_counts": dict(Counter(r["status"] for r in final)), "integrity_valid": all(r.get("integrity_valid") for r in replied),
               "contract_failures": dict(Counter(r["kind"] for r in replied if not r.get("contract_valid", False))),
               "prompt_token_differences": dict(Counter(r.get("prompt_token_difference") for r in replied)),
               "diagnostic_flags": sum(not r.get("diagnostic_valid", False) for r in replied),
               "list_price_cost_usd": round(received, 6), "possibly_billed_failed_attempts_upper_usd": round(failed_charge, 6),
               "cumulative_list_price_cost_usd": round(prior_spent_usd + received + failed_charge, 6), "approved_cumulative_budget_usd": budget_usd,
               "completed_at": utc_now()}
    if summary["complete"]:
        for replicate_id in sorted({r["replicate_id"] for r in final}):
            mine = [r for r in final if r["replicate_id"] == replicate_id]
            write_new_text(run_dir / f"extractions_{replicate_id}.jsonl", "".join(json.dumps(r, default=str) + "\n" for r in mine if r["kind"] == "extract"))
            for arm in ARMS:
                write_new_text(run_dir / f"predictions_{replicate_id}_{arm}.jsonl",
                               "".join(json.dumps(r, default=str) + "\n" for r in mine if r["arm"] == arm))
        write_new_text(run_dir / "completion.json", json.dumps(summary, indent=2, default=str))
    else:
        write_new_text(run_dir / f"stop_{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}.json", json.dumps(summary, indent=2, default=str))
    return summary


def notebook_code_sha256(notebook_json):
    return sha256_bytes(json.dumps(["".join(c["source"]) if isinstance(c["source"], list) else c["source"]
                                    for c in notebook_json["cells"] if c["cell_type"] == "code"]).encode())


def code_provenance():
    head = git("rev-parse", "HEAD")
    disk_code = notebook_code_sha256(json.loads((PROJECT_ROOT / NOTEBOOK_RELPATH).read_text()))
    head_notebook = git_show_bytes(head, NOTEBOOK_RELPATH)
    return {"git_commit": head, "branch": git("rev-parse", "--abbrev-ref", "HEAD"), "spec_sha256": SPEC_SHA256,
            "spec_committed": git_show_bytes(head, SPEC_RELPATH) == SPEC_BYTES == (PROJECT_ROOT / SPEC_RELPATH).read_bytes(),
            "notebook06_code_sha256": disk_code,
            "notebook06_code_committed": head_notebook is not None and notebook_code_sha256(json.loads(head_notebook)) == disk_code,
            "protected_paths_unchanged": protected_changes() == [], "git_status": git("status", "--porcelain").splitlines()}


DESIGN = {
    "spec_id": SPEC["spec_id"], "spec_sha256": SPEC_SHA256, "spec_status": SPEC["status"], "generated_at": utc_now(), "model": SNAPSHOT,
    "prompts": {"v0": V0_READER["system_prompt_hash"], "extraction": EXTRACTION["system_prompt_hash"], "reasoning": REASONING["system_prompt_hash"]},
    "qids": QIDS, "documents": HELDOUT_DOCS, "smoke_qid": SMOKE_QID, "arm_pages": ARM_PAGES,
    "v0_fingerprints": {q: V0_REQUESTS[q]["fingerprint"] for q in QIDS},
    "extraction_fingerprints": {f"{q}|{p}": EXTRACT_REQUESTS[(q, p)]["fingerprint"] for q, p in EXTRACTION_UNITS},
    "extraction_unit_keys": {f"{q}|{p}": EXTRACTION_UNIT_KEY[(q, p)] for q, p in EXTRACTION_UNITS},
    "plans": PLANS, "decision_rule": RULE,
}
(E7_DIR / "design.json").write_text(json.dumps(DESIGN, indent=1))
pd.DataFrame([{k: (json.dumps(v) if isinstance(v, list) else v) for k, v in e.items() if k != "request"}
              for e in [*V0_REQUESTS.values(), *EXTRACT_REQUESTS.values()]]).to_csv(E7_DIR / "request_plan.csv", index=False)
print(f"Executor defined; offline design saved ({len(V0_REQUESTS)} V0 and {len(EXTRACT_REQUESTS)} extraction requests, plans of "
      f"{len(PLANS['smoke'])} and {len(PLANS['full'])} calls).")

Executor defined; offline design saved (118 V0 and 944 extraction requests, plans of 11 and 1309 calls).


## 9. Evaluation: written and tested before any E7 reply exists

From a completed run's log, one row per (question, arm), scored with the frozen scorer. The run is validated first:
- one reply per planned call;
- integrity everywhere;
- each extraction paid for once;
- every reasoning memory built from this run's own extraction replies for the arm's pages.

**Reported** (all secondary results descriptive):
- **Primary.** X4 − V0: the verdict, with the engineering criteria and the statistical evidence shown separately.
- **Secondary.** X8 − X4 (additional page access) and X8 − V0 (total change), with the same function.
- **Mechanism (diagnostic).** The fixed and broken questions' E6 categories and the share of fixes in targeted categories, for each comparison.
- **Change by evidence span and by stratum.** X4 − V0, X8 − X4, and fixed / broken, single-page vs multi-page and complete / partial / none.
- **Accuracy breakdowns.**
  - by evidence span (single-page / multi-page), and by E6's evidence stratum (complete / partial / none), for every arm;
  - X8 also by its own stratum over 8 pages.
- **Fixed and broken questions,** with their E6 audit labels, and every labelled failure category's correct count per arm (the targeted categories' change).
- **Abstention.** Correct abstention (16), false abstention (102), unsupported answers (an answered unanswerable question; an answer without a valid citation; an X answer from an empty memory).
- **Citations.** Validity (118) and grounded answers (101).
- **Extraction diagnostics.**
  - `relevant` and facts on annotated evidence pages vs other pages;
  - unreadable pages, page mismatches, dropped facts.
- **Arithmetic.** Calculations requested, evaluated and failed. The tool's own contribution: the computed answer vs the model's own answer, scored.
- **Costs.**
  - Tokens, cost and latency per question and arm. For the X arms, latency is given sequential (as run) and as the critical path if the page calls ran in parallel.
  - The experiment's actual spend, and the extraction cache hit rate.
- **Session noise.** The frozen E6 V answers vs this session's V0 answers, as the flips between two sessions of the same request.

In [19]:
# From a completed run to the pre-registered results.
E6_STRATUM = e6v["stratum"].to_dict()
EVIDENCE_SPAN = e6v["evidence_span"].to_dict()


def validate_run(final, plan, cache):
    # Checks before any number is reported; returns {check: passed}.
    by_index = {r["execution_index"]: r for r in final}
    main = [r for r in final if r["replicate_id"] == "main_r1"]
    reason = [r for r in main if r["kind"] == "reason"]
    extract_keys = {r["cache_provenance"]["cache_key"] for r in main if r["kind"] == "extract"}
    checks = {
        "one record per planned call, in plan order": [r["execution_index"] for r in final] == [i["execution_index"] for i in plan]
        and all(by_index[i["execution_index"]]["qid"] == i["qid"] and by_index[i["execution_index"]]["kind"] == i["kind"] for i in plan),
        "every planned call has a reply": all(r["status"] in RESPONDED for r in final),
        "integrity valid on every reply": all(r.get("integrity_valid") for r in final),
        "served model is the snapshot on every reply": all(r.get("served_model") == SNAPSHOT for r in final),
        "each (question, page) extracted exactly once": Counter((r["qid"], r["page"]) for r in main if r["kind"] == "extract") == Counter(EXTRACTION_UNITS),
        "no extraction reply paid twice (one live cache entry per fingerprint)": len({(e["replicate_id"], e["fingerprint"]) for e in cache.values()}) == len(cache),
        "each reasoning memory = the arm's pages, in rank order": all([p["page"] for p in r["memory"]["pages"]] == ARM_PAGES[r["arm"]][r["qid"]] for r in reason),
        "each reasoning memory built from this run's extraction replies": all(set(r["extraction_cache_keys"]) <= extract_keys for r in reason),
        "X4 reads the same cached extractions as X8 on its pages": all(
            set(next(r["extraction_cache_keys"] for r in reason if (r["qid"], r["arm"]) == (q, "X4")))
            <= set(next(r["extraction_cache_keys"] for r in reason if (r["qid"], r["arm"]) == (q, "X8"))) for q in QIDS),
    }
    return checks


def arm_table(final, replicate="main_r1"):
    mine = [r for r in final if r["replicate_id"] == replicate]
    v0 = {r["qid"]: r for r in mine if r["kind"] == "v0"}
    ext = {(r["qid"], r["page"]): r for r in mine if r["kind"] == "extract"}
    rea = {(r["qid"], r["arm"]): r for r in mine if r["kind"] == "reason"}
    rows = []
    for arm in ARMS:
        for q in QIDS:
            ref = held_idx.loc[q]
            rec = v0[q] if arm == "V0" else rea[(q, arm)]
            supplied = ARM_PAGES[arm][q]
            responded = rec["status"] in RESPONDED
            score = eval_score(ref["answer"], rec["pred"], ref["answer_format"]) if responded else 0.0
            has_evidence = q in EVIDENCE_QIDS
            check = citation_check(rec["pred"], rec["cited_pages"], supplied, ref["evidence_pages"] if has_evidence else None)
            calls = [rec] if arm == "V0" else [ext[(q, p)] for p in supplied] + [rec]
            arithmetic = rec.get("arithmetic") or {}
            model_answer = arithmetic.get("model_answer", rec["pred"])
            model_score = eval_score(ref["answer"], model_answer, ref["answer_format"]) if responded else 0.0
            memory = rec.get("memory") or {"pages": [], "facts": []}
            latencies = [c.get("api_latency_s") or 0.0 for c in calls]
            rows.append({
                "qid": q, "arm": arm, "doc_id": ref["doc_id"], "answerable": bool(ref["answerable"]), "status": rec["status"],
                "pred": str(rec["pred"])[:200], "score": score, "correct": score > 0, "strict": score >= 1, "abstained": check["abstained"],
                "citation_valid": check["valid"], "citation_ok": check["valid"] and not check["missing"], "missing_citation": check["missing"],
                "cites_evidence": check["cites_evidence"],
                "evidence_supplied": bool(set(ref["evidence_pages"]) & set(supplied)) if has_evidence else None,
                "grounded": (not check["abstained"] and check["valid"] and bool(rec["cited_pages"]) and bool(check["cites_evidence"])) if has_evidence else None,
                "evidence_span": EVIDENCE_SPAN.get(q), "e6_stratum": E6_STRATUM.get(q), "own_stratum": evidence_stratum(q, supplied) if ref["answerable"] else None,
                "e6_label": audit.loc[q, "category"] if q in audit.index else None, "pages": len(supplied), "calls": len(calls),
                "prompt_tokens": sum(c.get("prompt_tokens") or 0 for c in calls), "completion_tokens": sum(c.get("completion_tokens") or 0 for c in calls),
                "cost_usd": sum(reply_cost(c) for c in calls), "latency_sequential_s": sum(latencies),
                "latency_parallel_s": latencies[0] if arm == "V0" else max(latencies[:-1]) + latencies[-1],
                "facts_in_memory": len(memory["facts"]) if arm != "V0" else None,
                "unreadable_pages": sum(p["status"] == PAGE_STATUS["unreadable"] for p in memory["pages"]) if arm != "V0" else None,
                "calc_status": arithmetic.get("calc_status") if arm != "V0" else None, "model_answer_correct": model_score > 0})
    return pd.DataFrame(rows)


def evaluate_e7(final, replicate="main_r1"):
    table = arm_table(final, replicate)
    by = {arm: table[table["arm"] == arm].set_index("qid").loc[QIDS] for arm in ARMS}
    docs = held_idx.loc[ANSWERABLE, "doc_id"].to_numpy()

    def compare(a, b):
        return paired_comparison(by[a].loc[ANSWERABLE, "correct"].astype(float), by[b].loc[ANSWERABLE, "correct"].astype(float), docs)

    primary, pages, total = compare("V0", "X4"), compare("X4", "X8"), compare("V0", "X8")
    abstentions = {arm: int(by[arm].loc[UNANSWERABLE, "correct"].sum()) for arm in ARMS}
    validity = {arm: float(by[arm]["citation_ok"].mean()) for arm in ARMS}
    def guardrails_of(before, after):
        return {f"G1 correct abstentions {after} >= {before} - 2": abstentions[after] >= abstentions[before] - RULE["guardrails"]["G1_tolerance"],
                f"G2 citation validity {after} - {before} >= -0.05": validity[after] - validity[before] >= -RULE["guardrails"]["G2_tolerance"]}

    def changed(before, after):
        fixed = [q for q in ANSWERABLE if not by[before].loc[q, "correct"] and by[after].loc[q, "correct"]]
        broken = [q for q in ANSWERABLE if by[before].loc[q, "correct"] and not by[after].loc[q, "correct"]]
        return fixed, broken

    fixed, broken = changed("V0", "X4")
    decision = decide_e7(primary, guardrails_of("V0", "X4"))  # the primary verdict
    pages_decision = decide_e7(pages, guardrails_of("X4", "X8"))  # X8 - X4: additional page access, guardrails read against X4
    total_decision = decide_e7(total, guardrails_of("V0", "X8"))  # X8 - V0: total practical change (two factors; descriptive)
    mechanism = {"X4 vs V0": mechanism_diagnostic(fixed, broken), "X8 vs X4": mechanism_diagnostic(*changed("X4", "X8")),
                 "X8 vs V0": mechanism_diagnostic(*changed("V0", "X8"))}
    e6_correct = np.array([E6_CORRECT["V"][q] for q in ANSWERABLE], dtype=float)
    noise = paired_comparison(e6_correct, by["V0"].loc[ANSWERABLE, "correct"].astype(float), docs)

    ans = table[table["answerable"]]
    def accuracy_by(column, frame=ans):
        pivot = frame.pivot_table(index=column, columns="arm", values="correct", aggfunc=lambda s: f"{int(s.sum())}/{len(s)}")
        return pivot[[arm for arm in ARMS if arm in pivot.columns]]

    transitions = ans[ans["e6_label"].notna()].pivot_table(index="e6_label", columns="arm", values="correct", aggfunc="sum")[list(ARMS)].astype(int)
    transitions.insert(0, "E6 errors", ans[(ans["arm"] == "V0") & ans["e6_label"].notna()].groupby("e6_label").size())
    transitions["targeted"] = transitions.index.isin(TARGETED)
    def delta_by(column):
        rows = {}
        for group, frame in ans.groupby(column):
            correct = frame.pivot_table(index="qid", columns="arm", values="correct", aggfunc="first").astype(bool)
            rows[group] = {"questions": len(correct), **{f"{arm} correct": int(correct[arm].sum()) for arm in ARMS},
                           "X4 - V0": int(correct["X4"].sum() - correct["V0"].sum()), "X8 - X4": int(correct["X8"].sum() - correct["X4"].sum()),
                           "X4 fixed / broken vs V0": f"{int((~correct['V0'] & correct['X4']).sum())} / {int((correct['V0'] & ~correct['X4']).sum())}"}
        return pd.DataFrame(rows).T

    unanswerable_labels = table[table["qid"].isin(audit_unanswerable.index)].pivot_table(index="qid", columns="arm", values="abstained", aggfunc="first")[list(ARMS)]
    reliability = pd.DataFrame({arm: {
        "correct abstention (16)": abstentions[arm],
        "false abstention (102)": int(by[arm].loc[ANSWERABLE, "abstained"].sum()),
        "  of which evidence was supplied (101)": int((by[arm].loc[EVIDENCE_QIDS, "abstained"] & by[arm].loc[EVIDENCE_QIDS, "evidence_supplied"].astype(bool)).sum()),
        "answered an unanswerable question (16)": int((~by[arm].loc[UNANSWERABLE, "abstained"]).sum()),
        "answer without a valid citation (118)": int((~by[arm]["abstained"] & ~by[arm]["citation_ok"]).sum()),
        "answer from an empty memory (118)": None if arm == "V0" else int((~by[arm]["abstained"] & (by[arm]["facts_in_memory"] == 0)).sum()),
        "citation validity (118)": round(validity[arm], 4),
        "grounded answers (101)": int(by[arm].loc[EVIDENCE_QIDS, "grounded"].astype(bool).sum()),
        "parse failures (118)": int((by[arm]["status"] != "ok").sum())} for arm in ARMS})
    resources = pd.DataFrame({arm: {
        "calls per question": by[arm]["calls"].mean(), "prompt tokens per question (mean)": by[arm]["prompt_tokens"].mean(),
        "completion tokens per question (mean)": by[arm]["completion_tokens"].mean(), "USD per question if run alone (mean)": by[arm]["cost_usd"].mean(),
        "USD total if run alone": by[arm]["cost_usd"].sum(), "latency sequential s (median)": by[arm]["latency_sequential_s"].median(),
        "latency sequential s (p90)": by[arm]["latency_sequential_s"].quantile(0.9), "latency if page calls parallel s (median)": by[arm]["latency_parallel_s"].median(),
        "latency if page calls parallel s (p90)": by[arm]["latency_parallel_s"].quantile(0.9)} for arm in ARMS}).round(5)
    mine = [r for r in final if r["replicate_id"] == replicate]
    readings = sum(len(ARM_PAGES[a][q]) for a in ("X4", "X8") for q in QIDS)
    live_extractions = sum(r["kind"] == "extract" and r["source"] == "live" for r in mine)
    extraction_rows = []
    for r in (r for r in mine if r["kind"] == "extract"):
        ex = r.get("extraction") or {}
        extraction_rows.append({"qid": r["qid"], "page": r["page"], "status": ex.get("status"), "relevant": ex.get("relevant"), "facts": len(ex.get("facts") or []),
                                "page_mismatch": ex.get("page_mismatch"), "dropped_facts": ex.get("dropped_facts"),
                                "is_evidence": None if r["qid"] not in EVIDENCE_QIDS else r["page"] in held_idx.loc[r["qid"], "evidence_pages"]})
    extraction = pd.DataFrame(extraction_rows)
    ev = extraction[extraction["is_evidence"].notna()]
    extraction_summary = {
        "pages read": len(extraction), "unreadable": int((extraction["status"] != "ok").sum()), "page mismatches": int(extraction["page_mismatch"].fillna(False).sum()),
        "dropped facts": int(extraction["dropped_facts"].fillna(0).sum()), "facts per page (mean)": float(extraction["facts"].mean()),
        "annotated evidence pages flagged relevant": f"{int((ev[ev['is_evidence'].astype(bool)]['relevant'] == True).sum())}/{int(ev['is_evidence'].astype(bool).sum())}",
        "annotated evidence pages with >= 1 fact": f"{int((ev[ev['is_evidence'].astype(bool)]['facts'] > 0).sum())}/{int(ev['is_evidence'].astype(bool).sum())}",
        "other pages flagged relevant": f"{int((ev[~ev['is_evidence'].astype(bool)]['relevant'] == True).sum())}/{int((~ev['is_evidence'].astype(bool)).sum())}"}
    arithmetic = pd.DataFrame({arm: {
        "calculations requested": int(by[arm]["calc_status"].isin(["ok", "error", "ignored: the reply abstained"]).sum()),
        "evaluated": int((by[arm]["calc_status"] == "ok").sum()), "failed (model answer used)": int((by[arm]["calc_status"] == "error").sum()),
        "correct with the tool (102)": int(by[arm].loc[ANSWERABLE, "correct"].sum()),
        "correct with the model's own answer (102)": int(by[arm].loc[ANSWERABLE, "model_answer_correct"].sum())} for arm in ("X4", "X8")})
    summary = {
        "spec_id": SPEC["spec_id"], "spec_sha256": SPEC_SHA256, "population": "development (E6's 118 held-out questions, inspected for E7's design)",
        "accuracy": {arm: f"{int(by[arm].loc[ANSWERABLE, 'correct'].sum())}/102" for arm in ARMS},
        "primary_X4_minus_V0": primary, "decision": decision, "mechanism_diagnostic": mechanism, "fixed": fixed, "broken": broken,
        "additional_pages_X8_minus_X4": {**pages, "decision": pages_decision}, "total_X8_minus_V0": {**total, "decision_descriptive": total_decision},
        "session_noise_frozen_E6_V_vs_V0": {k: noise[k] for k in ("before", "after", "delta", "fixed", "broken")},
        "abstentions": abstentions, "citation_validity": validity, "extraction": extraction_summary,
        "cache": {"page readings by X4 and X8": readings, "extraction calls paid": live_extractions,
                  "cache hit rate": 1 - live_extractions / readings if readings else None},
        "experiment_list_price_usd": float(sum(reply_cost(r) for r in mine if r["source"] == "live")),
    }
    tables = {"change by evidence span (single / multi-page)": delta_by("evidence_span"), "change by E6 evidence stratum": delta_by("e6_stratum"),
              "accuracy by evidence span": accuracy_by("evidence_span"), "accuracy by E6 evidence stratum": accuracy_by("e6_stratum"),
              "X8 accuracy by its own stratum": accuracy_by("own_stratum", ans[ans["arm"] == "X8"]), "E6 failure categories, correct per arm": transitions,
              "unanswerable questions E6 V answered: abstained per arm": unanswerable_labels, "reliability": reliability, "resources": resources,
              "arithmetic": arithmetic}
    return jsonable(summary), table, tables


def jsonable(value):
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(v) for v in value]
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    return value


def report(summary, tables):
    d = summary["decision"]
    p = summary["primary_X4_minus_V0"]
    display(pd.Series({"accuracy (102)": summary["accuracy"], "X4 - V0": f"{p['delta']:+.3f} (fixed {p['fixed']}, broken {p['broken']}, CI95 "
                       f"[{p['ci95'][0]:+.3f}, {p['ci95'][1]:+.3f}], one-sided p {p['sign_flip_p_one_sided']:.4f})",
                       "verdict": f"{d['verdict']} ({d['reason']})", "engineering criteria": d["engineering"], "statistical evidence": d["statistics"],
                       "mechanism (diagnostic)": summary["mechanism_diagnostic"]["X4 vs V0"],
                       "X8 - X4 (additional page access)": f"{summary['additional_pages_X8_minus_X4']['delta']:+.3f} -> {summary['additional_pages_X8_minus_X4']['decision']['verdict']}",
                       "X8 - V0 (total, two factors)": f"{summary['total_X8_minus_V0']['delta']:+.3f} -> {summary['total_X8_minus_V0']['decision_descriptive']['verdict']}",
                       "session noise (frozen E6 V -> V0)": summary["session_noise_frozen_E6_V_vs_V0"], "cache": summary["cache"],
                       "experiment cost (USD)": round(summary["experiment_list_price_usd"], 4)}, name="E7 (development)").to_frame())
    for name, frame in tables.items():
        print(name)
        display(frame)


print("Evaluation functions defined; they are exercised end to end on a mocked complete run in the next cell.")

Evaluation functions defined; they are exercised end to end on a mocked complete run in the next cell.


In [20]:
# Interface tests of the executor and the evaluation: real SDK ChatCompletion objects and SDK exceptions carrying invented replies.
# Every run here is written to a temporary directory and deleted; none of these replies ever enters a result.
MOCK_REQUEST = httpx.Request("POST", "https://api.openai.com/v1/chat/completions")


def mock_status_error(cls, status, body):
    return cls(f"Error code: {status} - {body}", response=httpx.Response(status, request=MOCK_REQUEST, headers={"x-request-id": "req_mock"}), body=body)


quota_error = mock_status_error(openai.RateLimitError, 429, {"message": "You have no credits remaining.", "type": "insufficient_quota", "param": None,
                                                              "code": "credit_balance_exhausted"})
rate_error = mock_status_error(openai.RateLimitError, 429, {"message": "Rate limit reached for tokens", "type": "tokens", "param": None, "code": "rate_limit_exceeded"})
server_error = mock_status_error(openai.InternalServerError, 500, {"message": "The server had an error.", "type": "server_error", "param": None, "code": None})
QID_OF_QUESTION = {question_of[q]: q for q in QIDS}
FROZEN_V_REPLY = {E6_RECORDS["V"][q]["fingerprint"]: E6_RECORDS["V"][q]["raw_response"] for q in QIDS}


def mock_completion(content, *, finish_reason, prompt_tokens, model):
    usage = {"prompt_tokens": prompt_tokens, "completion_tokens": 12, "total_tokens": prompt_tokens + 12, "prompt_tokens_details": {"cached_tokens": 0}}
    return ChatCompletion.model_validate({"id": "chatcmpl-mock-e7", "object": "chat.completion", "created": 0, "model": model, "system_fingerprint": "fp_mock_e7",
                                          "choices": [{"index": 0, "finish_reason": finish_reason, "message": {"role": "assistant", "content": content}}], "usage": usage})


def request_question(request):
    user = request["messages"][1]["content"]
    text = user if isinstance(user, str) else user[0]["text"]
    return re.match(r"Question: (.*?)\n\n(Page images:|Page image:|Evidence memory:)", text, re.S).group(1)


def request_page(request):
    return int(re.fullmatch(r"\[Page (\d+)\]", request["messages"][1]["content"][1]["text"]).group(1))


def request_memory(request):
    return json.loads(request["messages"][1]["content"].split("Evidence memory:\n", 1)[1])


def v0_frozen(request):
    # V0's request equals E6's: replay E6's reply (a tampered request, test 8, gets a neutral reply).
    return FROZEN_V_REPLY.get(request_fingerprint(request), '{"answer": "Not answerable", "cited_pages": []}'), "stop"


def extract_marker(request):
    page = request_page(request)
    return json.dumps({"page": page, "relevant": True, "facts": [{"label": f"marker of page {page}", "value": page, "unit": None, "period": None,
                                                                    "supporting_text": None}], "notes": None}), "stop"


def reason_copy_or_abstain(request):
    # X4 (<= 4 pages): repeat E6 V's answer and citations; X8 (8 pages): abstain. Lets the evaluation be checked against known numbers.
    qid, memory = QID_OF_QUESTION[request_question(request)], request_memory(request)
    if len(memory["pages"]) <= 4:
        frozen = E6_RECORDS["V"][qid]
        return json.dumps({"answer": frozen["pred"], "cited_pages": frozen["cited_pages"], "calculation": None}), "stop"
    return json.dumps({"answer": NOT_ANSWERABLE, "cited_pages": [], "calculation": None}), "stop"


def reason_count(request):
    memory = request_memory(request)
    return json.dumps({"answer": 0, "cited_pages": [], "calculation": {"expression": {"op": "count", "args": [{"fact": f["id"]} for f in memory["facts"]]},
                                                                        "round": None}}), "stop"


def fake_openai(*, v0=v0_frozen, extract=extract_marker, reason=reason_copy_or_abstain, errors=None, token_shift=1, model=None):
    calls = []

    def create(**request):
        calls.append(request_fingerprint(request))
        if errors and len(calls) in errors:
            raise errors[len(calls)]
        system = request["messages"][0]["content"]
        handler = v0 if system == IMAGE_SYSTEM_PROMPT else extract if system == EXTRACTION_SYSTEM_PROMPT else reason
        raw, finish = handler(request)
        return mock_completion(raw, finish_reason=finish, prompt_tokens=expected_prompt_tokens(request) + token_shift, model=model or SNAPSHOT)

    return create, calls


TEST_BUDGET = 50.0
SUBSET = PLANS["full"][: 11 + 11 * 3]  # the smoke block and the first three question blocks
results = {}
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)

    # 1. Smoke stage: 11 calls, gate passes, prediction files for the three arms.
    create, calls = fake_openai()
    summary = run_stages("smoke", make_engine(create), "t1", tmp / "t1", budget_usd=TEST_BUDGET)
    assert summary["complete"] and summary["smoke_gate_passed"] and len(calls) == 11
    assert all((tmp / "t1" / f"predictions_technical_smoke_{arm}.jsonl").is_file() for arm in ARMS)
    assert all(summary["smoke_checks"].values()) and len(summary["smoke_checks"]) == 12 and summary["smoke_observations"]["calculation status"] == {"X4": "none", "X8": "none"}
    results["smoke stage: 11 calls, gate passes"] = True

    # 2. Extraction reuse: each (question, page) sent once; X4's memory reads X8's cached replies; prompt tokens within tolerance.
    create, calls = fake_openai()
    summary = run_stages("full", make_engine(create), "t2", tmp / "t2", budget_usd=TEST_BUDGET, plan=SUBSET)
    final = latest_records(read_jsonl(tmp / "t2" / "execution_log.jsonl"))
    assert summary["complete"] and len(calls) == len(SUBSET)
    reasons = {(r["replicate_id"], r["qid"], r["arm"]): r for r in final if r["kind"] == "reason"}
    for (rep, q, arm), r in reasons.items():
        assert [p["page"] for p in r["memory"]["pages"]] == ARM_PAGES[arm][q]
        if arm == "X4":
            assert set(r["extraction_cache_keys"]) <= set(reasons[(rep, q, "X8")]["extraction_cache_keys"])
    assert sum(r["kind"] == "extract" for r in final) == 8 * 4 and all(r.get("diagnostic_valid") for r in final)
    results["extraction paid once; X4 reuses X8's cached pages"] = True

    # 3. Interruption and resume: a server error stops the run; the resumed run sends only what has no reply.
    create, calls = fake_openai(errors={5: server_error})
    stopped = run_stages("full", make_engine(create), "t3", tmp / "t3", budget_usd=TEST_BUDGET, plan=SUBSET)
    assert not stopped["complete"] and stopped["stop_reason"].startswith("server") and len(calls) == 5
    create2, calls2 = fake_openai()
    resumed = run_stages("full", make_engine(create2), "t3", tmp / "t3", budget_usd=TEST_BUDGET, plan=SUBSET)
    assert resumed["complete"] and len(calls2) == len(SUBSET) - 4 and resumed["possibly_billed_failed_attempts_upper_usd"] > 0
    results["resume sends only missing calls; failed attempt charged"] = True

    # 4. Quota: stops at once, not retried. 5. Rate limit: waits, then succeeds within the same attempt.
    create, calls = fake_openai(errors={3: quota_error})
    stopped = run_stages("smoke", make_engine(create), "t4", tmp / "t4", budget_usd=TEST_BUDGET)
    assert not stopped["complete"] and stopped["stop_reason"].startswith("quota_or_billing") and len(calls) == 3
    waits = []
    create, calls = fake_openai(errors={1: rate_error})
    done = run_stages("smoke", make_engine(create), "t5", tmp / "t5", budget_usd=TEST_BUDGET, sleep=waits.append)
    assert done["complete"] and done["rate_limited_responses"] == 1 and RATE_LIMIT["backoff_s"][0] in waits
    results["quota stops; rate limit waits and retries"] = True

    # 6. Budget: a ceiling below the smoke upper bound stops before any call.
    create, calls = fake_openai()
    stopped = run_stages("smoke", make_engine(create), "t6", tmp / "t6", budget_usd=0.01)
    assert not stopped["complete"] and stopped["stop_reason"].startswith("budget") and calls == []
    results["budget guard stops before any call"] = True

    # 7. A malformed extraction: fails the smoke gate; in the main stage the page enters the memory as unreadable and the run continues.
    smoke_page = ARM_PAGES["X8"][SMOKE_QID][0]
    bad_smoke = lambda request: ('{"page": 1, "facts": [{"label": "x", "val', "length") if request_page(request) == smoke_page and \
        request_question(request) == question_of[SMOKE_QID] else extract_marker(request)
    create, calls = fake_openai(extract=bad_smoke)
    stopped = run_stages("smoke", make_engine(create), "t7", tmp / "t7", budget_usd=TEST_BUDGET)
    assert not stopped["smoke_gate_passed"] and stopped["stop_reason"].startswith("technical gate failed")
    assert not stopped["smoke_checks"]["one block for the smoke question (1 V0, 8 extractions, X4 and X8 reasoning), every call replied"]
    target_qid = next(i["qid"] for i in SUBSET[11:] if i["qid"] != SMOKE_QID)
    target_page = ARM_PAGES["X4"][target_qid][0]
    bad_main = lambda request: ("not json", "stop") if request_page(request) == target_page and request_question(request) == question_of[target_qid] \
        else extract_marker(request)
    create, calls = fake_openai(extract=bad_main)
    done = run_stages("full", make_engine(create), "t7b", tmp / "t7b", budget_usd=TEST_BUDGET, plan=SUBSET)
    final = latest_records(read_jsonl(tmp / "t7b" / "execution_log.jsonl"))
    assert done["complete"] and done["contract_failures"] == {"extract": 1}
    for arm in ("X4", "X8"):
        r = next(r for r in final if r["kind"] == "reason" and r["qid"] == target_qid and r["arm"] == arm and r["replicate_id"] == "main_r1")
        assert r["page_status"][str(target_page)] == "unreadable" and r["memory"]["pages"][0]["status"] == PAGE_STATUS["unreadable"]
    results["unreadable extraction: smoke gate fails; main run keeps the page as unreadable"] = True

    # 8. Integrity: arguments altered between the plan and the SDK stop the run, which then cannot be resumed.
    create, calls = fake_openai()
    plain = make_engine(create)
    tampered = lambda request: plain({**request, "temperature": 0.7})
    stopped = run_stages("smoke", tampered, "t8", tmp / "t8", budget_usd=TEST_BUDGET)
    assert stopped["stop_reason"].startswith("integrity check failed")
    try:
        remaining_items(PLANS["smoke"], read_jsonl(tmp / "t8" / "execution_log.jsonl"))
        raise AssertionError("an integrity failure must block resumption")
    except AssertionError as error:
        assert "cannot be resumed" in str(error)
    results["integrity failure stops and blocks resumption"] = True

    # 9. Arithmetic in the run: a 'count' over the memory's facts replaces the model's answer; operand pages become citations.
    create, calls = fake_openai(reason=reason_count)
    done = run_stages("smoke", make_engine(create), "t9", tmp / "t9", budget_usd=TEST_BUDGET)
    x8 = next(r for r in latest_records(read_jsonl(tmp / "t9" / "execution_log.jsonl")) if r["arm"] == "X8")
    assert done["complete"] and x8["pred"] == X8_K and x8["cited_pages"] == ARM_PAGES["X8"][SMOKE_QID] and x8["arithmetic"]["calc_status"] == "ok"
    assert done["smoke_gate_passed"] and done["smoke_observations"]["calculation status"] == {"X4": "ok", "X8": "ok"}
    results["calculation evaluated in the run, operand pages cited"] = True

    # 10. End to end: a complete mocked full run (1,309 calls), validated and evaluated against numbers known by construction.
    started = time.perf_counter()
    create, calls = fake_openai()
    done = run_stages("full", make_engine(create), "t10", tmp / "t10", budget_usd=TEST_BUDGET)
    final = latest_records(read_jsonl(tmp / "t10" / "execution_log.jsonl"))
    checks = validate_run(final, PLANS["full"], load_cache(tmp / "t10" / "response_cache.jsonl"))
    assert done["complete"] and len(calls) == len(PLANS["full"]) and all(checks.values()), checks
    mock_summary, mock_table, mock_tables = evaluate_e7(final)
    v0_rows = mock_table[mock_table["arm"] == "V0"].set_index("qid")
    assert all(v0_rows.loc[q, "score"] == e6_table.loc[q, "score_V"] for q in QIDS)  # V0 replaying E6's replies scores exactly as E6 V
    assert mock_summary["accuracy"] == {"V0": "38/102", "X4": "38/102", "X8": "0/102"}
    assert (mock_summary["primary_X4_minus_V0"]["fixed"], mock_summary["primary_X4_minus_V0"]["broken"]) == (0, 0)
    assert mock_summary["decision"]["verdict"] == "no_meaningful_gain" and mock_summary["abstentions"] == {"V0": 7, "X4": 7, "X8": 16}
    assert mock_summary["additional_pages_X8_minus_X4"]["decision"]["verdict"] == "regression"  # -0.37: all answerable lost
    assert mock_summary["mechanism_diagnostic"]["X4 vs V0"]["fixed"] == 0 and mock_tables["change by evidence span (single / multi-page)"]["X4 - V0"].eq(0).all()
    assert mock_summary["cache"] == {"page readings by X4 and X8": 1403, "extraction calls paid": 944, "cache hit rate": 1 - 944 / 1403}
    assert mock_summary["session_noise_frozen_E6_V_vs_V0"]["fixed"] == mock_summary["session_noise_frozen_E6_V_vs_V0"]["broken"] == 0
    assert mock_tables["reliability"].loc["citation validity (118)", "V0"] == mock_tables["reliability"].loc["citation validity (118)", "X4"]
    results[f"full mocked run (1,309 calls, {time.perf_counter() - started:.0f} s): validated; evaluation reproduces the known numbers"] = True
display(pd.Series(results, name="passed").to_frame())

,passed
"smoke stage: 11 calls, gate passes",True
extraction paid once; X4 reuses X8's cached pages,True
resume sends only missing calls; failed attempt charged,True
quota stops; rate limit waits and retries,True
budget guard stops before any call,True
unreadable extraction: smoke gate fails; main run keeps the page as unreadable,True
integrity failure stops and blocks resumption,True
"calculation evaluated in the run, operand pages cited",True
"full mocked run (1,309 calls, 29 s): validated; evaluation reproduces the known numbers",True


## 10. Execution (paid modes only), and evaluation of a completed run

**Offline (the default).** No OpenAI client is created and nothing is sent.

**Paid modes.** `VISUALDOC_E7_MODE=smoke` or `full` opens the readiness gate, which requires all of:
- the spec frozen (status and `frozen_at`) with an approved budget;
- the spec and this notebook's code committed (equal to git HEAD);
- no change under a protected path;
- an API key present (never printed);
- the planned upper bound within the remaining cumulative budget;
- the date before the snapshot's shutdown.

Each stage needs the owner's explicit approval.

**Evaluation.** `VISUALDOC_E7_RUN_DIR` evaluates a completed run and writes its results to `data/processed/mmlongbench_doc/e7/evaluation_<run_id>/`.

In [21]:
# E7 execution. Offline (default): no client is created and nothing is sent.
e7_run_dir = None
provenance = code_provenance()
if E7_MODE == "offline":
    print("E7 offline mode: nothing was sent and no OpenAI client was created.")
    print(f"Spec status {SPEC['status']!r}; approved budget {BUDGET['approved_ceiling_usd']} (proposed {BUDGET['proposed_ceiling_usd']}). "
          f"Planned calls: smoke {len(PLANS['smoke'])}, full {len(PLANS['full'])}.")
    print(f"Code provenance (a gate only in paid modes): HEAD {provenance['git_commit'][:12]} on {provenance['branch']}; spec committed: "
          f"{provenance['spec_committed']}; notebook-06 code committed: {provenance['notebook06_code_committed']}; protected paths unchanged: "
          f"{provenance['protected_paths_unchanged']}.")
else:
    plan = PLANS[E7_MODE]
    budget = BUDGET["approved_ceiling_usd"]
    budget_ok = isinstance(budget, (int, float)) and not isinstance(budget, bool)
    planned_upper = float(UPPER[E7_MODE])
    resume_value = os.getenv("VISUALDOC_E7_RESUME_DIR", "").strip()
    resume_dir = (Path(resume_value) if Path(resume_value).is_absolute() else PROJECT_ROOT / resume_value) if resume_value else None
    resumed_id = json.loads((resume_dir / "settings.json").read_text())["run_id"] if resume_dir else None
    prior_spent = prior_spend_usd(RUNS_DIR, SPEC_SHA256, exclude_run_id=resumed_id)
    readiness = {
        "E7 still open (no frozen result in experiments/frozen/e7)": not (FROZEN_DIR / "e7").exists(),
        "spec frozen before live execution (status and frozen_at)": SPEC_FROZEN,
        "budget approved in the spec": budget_ok,
        f"planned upper bound ${planned_upper:.4f} within the remaining cumulative budget (already spent ${prior_spent:.4f})":
            budget_ok and prior_spent + planned_upper <= budget,
        "OPENAI_API_KEY present (value never printed)": bool(os.getenv("OPENAI_API_KEY")),
        "spec and notebook-06 code committed (equal to git HEAD)": provenance["spec_committed"] and provenance["notebook06_code_committed"],
        "no uncommitted change under a protected path": provenance["protected_paths_unchanged"],
        "before the snapshot's shutdown date (2027-01-14)": datetime.now(timezone.utc).date() < SHUTDOWN_DATE,
    }
    display(pd.Series(readiness, name="ready").to_frame())
    if not all(readiness.values()):
        print("E7 live gate closed: no request was sent. Fix the failed readiness check and rerun.")
    else:
        from openai import OpenAI

        client = OpenAI(max_retries=0, timeout=EXECUTION["timeout_s"])  # the SDK's silent retries are off
        if resume_dir:
            e7_run_dir = resume_dir
            settings_record = json.loads((e7_run_dir / "settings.json").read_text())
            assert settings_record["mode"] == E7_MODE and settings_record["spec_sha256"] == SPEC_SHA256
            assert json.loads((e7_run_dir / "execution_plan.json").read_text()) == plan, "plan differs from the interrupted run"
            assert not (e7_run_dir / "completion.json").exists(), "this run is already complete"
            run_id = settings_record["run_id"]
        else:
            run_id = f"{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}_{uuid.uuid4().hex[:8]}_{E7_MODE}"
            e7_run_dir = RUNS_DIR / run_id
            e7_run_dir.mkdir(parents=True, exist_ok=False)
            settings_record = {**{k: v for k, v in DESIGN.items() if k not in ("extraction_fingerprints", "extraction_unit_keys", "v0_fingerprints")},
                               "run_id": run_id, "mode": E7_MODE, "system_prompts": {"v0": IMAGE_SYSTEM_PROMPT, "extraction": EXTRACTION_SYSTEM_PROMPT,
                                                                                    "reasoning": REASONING_SYSTEM_PROMPT},
                               "planned_calls": len(plan), "max_attempts_per_request": MAX_ATTEMPTS, "approved_budget_usd": budget,
                               "budget_scope": BUDGET["scope"], "prior_spent_usd_at_creation": prior_spent, "planned_cost_upper_usd": planned_upper,
                               "code_provenance": provenance}
            write_new_text(e7_run_dir / "settings.json", json.dumps(settings_record, indent=2))
            write_new_text(e7_run_dir / "spec_snapshot.json", SPEC_BYTES.decode())
            write_new_text(e7_run_dir / "execution_plan.json", json.dumps(plan, indent=2))
        assert not e7_run_dir.resolve().is_relative_to(FROZEN_DIR.resolve()), "experiments/frozen/ is never written to"
        write_new_text(e7_run_dir / f"session_{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}.json",
                       json.dumps({"started_at": utc_now(), "openai": openai.__version__, "code_provenance": provenance}, indent=2, default=str))
        summary = run_stages(E7_MODE, make_engine(client.chat.completions.create), run_id, e7_run_dir, budget_usd=budget,
                             prior_spent_usd=prior_spent, pacer=TokenPacer(EXECUTION["pacer_tokens_per_minute"]))
        display(pd.Series({k: v for k, v in summary.items() if k not in ("smoke_checks", "smoke_observations")}, name="E7 run").to_frame())
        display(pd.Series(summary["smoke_checks"], name="smoke gate (technical only)").to_frame())
        display(pd.Series(summary["smoke_observations"], name="smoke observations (model behaviour; never a gate)", dtype=object).to_frame())
        print("E7 run saved without overwriting any earlier output:", e7_run_dir.relative_to(PROJECT_ROOT))

E7 offline mode: nothing was sent and no OpenAI client was created.
Spec status 'frozen-before-live-execution'; approved budget 9.0 (proposed 9.0). Planned calls: smoke 11, full 1309.
Code provenance (a gate only in paid modes): HEAD f19ac27a9b5c on refactor/simplify-structure; spec committed: True; notebook-06 code committed: True; protected paths unchanged: True.


In [22]:
# E7 evaluation. By default: the frozen full run (experiments/frozen/e7). Its checksums are verified, it is validated, the
# pre-registered analysis is re-run, the frozen tables are re-derived byte for byte and the regression references are asserted.
# VISUALDOC_E7_RUN_DIR=experiments/runs/e7/<run-id> evaluates another saved run instead. No API call is made here.
FROZEN_E7_DIR = FROZEN_DIR / "e7"
FROZEN_E7_RUN = FROZEN_E7_DIR / "run_20261124T160111456175Z_b3d0b5c0_full"
run_value = os.getenv("VISUALDOC_E7_RUN_DIR", "").strip()
evaluating_frozen = not run_value
if evaluating_frozen:
    assert (FROZEN_E7_DIR / "SHA256SUMS").is_file(), "the frozen E7 result is tracked in git: restore experiments/frozen/e7"
    frozen_e7_checks = verify_sha256sums(FROZEN_E7_DIR)
    assert all(frozen_e7_checks.values()), f"frozen E7 files differ from their SHA256SUMS: {[k for k, ok in frozen_e7_checks.items() if not ok]}"
    run_dir = FROZEN_E7_RUN
    print(f"Evaluating the frozen E7 full run: {len(frozen_e7_checks)} files equal their SHA256SUMS.")
else:
    run_dir = Path(run_value) if Path(run_value).is_absolute() else PROJECT_ROOT / run_value
settings_record = json.loads((run_dir / "settings.json").read_text())
completion = json.loads((run_dir / "completion.json").read_text())
assert settings_record["mode"] == "full" and completion["complete"], "only a complete full run is evaluated"
assert settings_record["spec_sha256"] == SPEC_SHA256 and (run_dir / "spec_snapshot.json").read_bytes() == SPEC_BYTES
run_plan = json.loads((run_dir / "execution_plan.json").read_text())
assert run_plan == PLANS["full"]
final = latest_records(read_jsonl(run_dir / "execution_log.jsonl"))
checks = validate_run(final, run_plan, load_cache(run_dir / "response_cache.jsonl"))
display(pd.Series(checks, name="run validation").to_frame())
assert all(checks.values()), "the run failed validation: no result is reported"
e7_summary, e7_table, e7_tables = evaluate_e7(final)
out = E7_DIR / f"evaluation_{settings_record['run_id']}"
out.mkdir(parents=True, exist_ok=True)
(out / "evaluation_summary.json").write_text(json.dumps(e7_summary, indent=2, default=str))
e7_table.to_csv(out / "per_question.csv", index=False)
report(e7_summary, e7_tables)
print("Development results (E6's questions were used to design E7) saved to", out.relative_to(PROJECT_ROOT))

if evaluating_frozen:
    for name in ("evaluation_summary.json", "per_question.csv"):
        assert (out / name).read_bytes() == (FROZEN_E7_DIR / "tables" / name).read_bytes(), f"{name} differs from the frozen table"
    primary = e7_summary["primary_X4_minus_V0"]
    regression = {  # observed, frozen reference
        "correct of 102 (V0, X4, X8)": (tuple(e7_summary["accuracy"][arm] for arm in ARMS), ("38/102", "31/102", "29/102")),
        "X4 - V0 fixed / broken": ((primary["fixed"], primary["broken"]), (3, 10)),
        "X4 - V0 verdict": (e7_summary["decision"]["verdict"], "regression"),
        "X8 - X4 verdict": (e7_summary["additional_pages_X8_minus_X4"]["decision"]["verdict"], "no_meaningful_gain"),
        "correct abstentions of 16 (V0, X4, X8)": (tuple(e7_summary["abstentions"][arm] for arm in ARMS), (7, 9, 8)),
        "cumulative list-price spend, USD": (completion["cumulative_list_price_cost_usd"], 5.911057),
    }
    display(pd.DataFrame({k: [str(o), str(r)] for k, (o, r) in regression.items()}, index=["observed", "frozen reference"]).T)
    assert all(observed == reference for observed, reference in regression.values()), "an E7 regression reference changed: explain it, do not silence it"
    print("The frozen tables were re-derived byte for byte, and every E7 regression reference holds.")


No E7 run to evaluate (VISUALDOC_E7_RUN_DIR is not set). No E7 result exists: the paid stages have not been run.


## Validation

In [23]:
# What this execution verified (every line is an assertion above that passed), and the protected paths re-checked at the end.
for label, name in FROZEN_FOLDERS.items():
    assert all(verify_sha256sums(FROZEN_DIR / name).values()), name
assert protected_changes() == [], protected_changes()
display(pd.Series({
    "frozen E0-E7": f"{sum(frozen_check.values())} files equal their SHA256SUMS at start and end; no uncommitted change under {len(PROTECTED_PATHS)} protected paths",
    "scorer": f"{len(SCORING_CASES)} hand-traced cases; every frozen E6 score reproduced (A 25/102, V 38/102, 13 fixed / 0 broken)",
    "E6 audit": "64 + 9 labels consistent with scores, evidence strata, guard losses and replies; diagnostics single-page 30/55 (+ 1/1 unannotated), multi-page 7/46, complete 32/61, partial 3/25, none 2/15 verified",
    "V0": f"{len(V0_REQUESTS)}/118 requests equal the frozen E6 V request (fingerprint, pages, images, tokens)",
    "pages": f"X4 = V0's pages; X8 = frozen ColQwen2 top {X8_K}; {len(EXTRACTION_UNITS)} extraction units, X4's a prefix of X8's",
    "stages": "extraction parser, Evidence Memory, arithmetic (incl. 17 refusals), reasoning parser and final answer: hand checks pass",
    "leakage": "structural, field, prompt, builder-signature and code-scan controls pass",
    "plan and cost": f"smoke {len(PLANS['smoke'])}, full {len(PLANS['full'])} calls; upper ${UPPER['all stages']:.4f}; ~{hours:.1f} h",
    "analysis": f"{len(cases)} decision-rule boundary cases (strong / promising / no gain / regression); E6's frozen primary reproduced exactly",
    "planning simulation": "P(engineering criteria) and P(strong_support) per true gain and noise level, recorded in the spec (planning only)",
    "executor and evaluation": f"{len(results)} interface tests with mocked SDK replies, incl. a complete mocked 1,309-call run",
}, name="verified").to_frame())
display(pd.Series({
    "E7": "closed: smoke and full runs of 2026-11-24, verdict regression, frozen in experiments/frozen/e7 (independent audit 38/38)",
    "evaluation": ("the frozen run validated, its tables re-derived byte for byte, every regression reference held" if evaluating_frozen
                   else f"evaluated {run_dir}"),
    "paid modes": "closed: a frozen E7 result exists, and the cumulative budget cannot cover another full plan",
}, name="status").to_frame())

,verified
frozen E0-E6,187 files equal their SHA256SUMS at start and end; no uncommitted ...
scorer,"17 hand-traced cases; every frozen E6 score reproduced (A 25/102, ..."
E6 audit,"64 + 9 labels consistent with scores, evidence strata, guard losse..."
V0,"118/118 requests equal the frozen E6 V request (fingerprint, pages..."
pages,"X4 = V0's pages; X8 = frozen ColQwen2 top 8; 944 extraction units,..."
stages,"extraction parser, Evidence Memory, arithmetic (incl. 17 refusals)..."
leakage,"structural, field, prompt, builder-signature and code-scan control..."
plan and cost,"smoke 11, full 1309 calls; upper $7.1201; ~4.7 h"
analysis,12 decision-rule boundary cases (strong / promising / no gain / re...
planning simulation,P(engineering criteria) and P(strong_support) per true gain and no...


,ready to execute
"spec frozen (status, frozen_at) with an approved $9.00 ceiling",True
spec and notebook 06 committed (equal to git HEAD),True
owner approval of the smoke stage,False
owner approval of the full stage,False


## Results and interpretation

**Development results.** E6's 118 questions were used to design E7. The run is frozen in `experiments/frozen/e7/`, and section 10 above re-derives every number below.

| | V0: one-shot | X4: extract-then-answer, V0's pages | X8: extract-then-answer, top 8 |
|---|---:|---:|---:|
| correct, 102 answerable | **38** | 31 | 29 |
| correct abstention, 16 | 7 | 9 | 8 |
| citation validity, 118 | 0.932 | 0.992 | 0.983 |
| calls per question | 1 | 4.9 | 9 |

**Verdict (frozen rule): `regression`.**
- **Primary.** X4 − V0 = −0.069: 3 fixed, 10 broken. Cluster 95 % interval [−0.135, −0.008]; one-sided sign-flip p = 0.990.
- **Secondary.** X8 − X4 = −0.020: `no_meaningful_gain`. Reading more pages through extraction does not recover the loss.
- **Session noise.** Frozen E6 V and this session's V0 both score 38/102, with 2 fixed and 2 broken between them.
- **Scoring sensitivity.** Strict scoring, or correcting `mmlb-0998`, gives −0.049. The conclusion is unchanged.

**What the evidence says.**
- **Where the loss sits.** It falls on questions whose evidence V0 already had in full. Turning a page into text facts discards what the one-shot reader uses: layout, visual properties, object counts and order.
- **The arithmetic tool.** It helped one question; 14 of 19 calculation requests were malformed.
- **The structured output contract.** It raised citation validity from 0.93 to 0.99. That is a property of the output format, not of the architecture, and is worth keeping in the one-shot reader.

**Limitations.**
- Development data only.
- 14 documents.
- One replicate per arm.
- Image and reasoning calls ran on different backends.
- The snapshot retires on 2027-01-14.

**Next step (the pre-registered consequence).** Keep E6's one-shot reader on ColQwen2 pages. Do not run further extraction or agentic scaffolds on this reader. The reader-migration evaluation and baseline consolidation follow in a separate stage.
